# Team mambers
- Youssef Abdalla — youssefmahmoud.abdalla@mail.polimi.it
- Sana Harighi — sana.harighi@mail.polimi.it
- Amirparsa Salmankhah — amirparsa.salmankhah@mail.polimi.it


# [Gitub repo](https://github.com/Youssef-mahmoud/NLP_Project)
# [Video Link](https://drive.google.com/file/d/1H5KiLjbbTr81EVnvX5zb6YDmcbPcM1vs/view?usp=sharing)

# PoliMillionaire — Configurable Live Web-RAG Experiment Notebook

This notebook is an updated version of the retrieval-focused notebook. It keeps the useful retrieval, chunking, filtering, and ranking ideas, but changes the experiment design so the final evaluation can run on the **live PoliMillionaire game wrapper**, not only on a fixed question bank.

Main goals:

1. Connect to the live game dynamically.
2. Choose one or more web retrieval tools per experiment.
3. Test multiple LLMs behind one interface.
4. Test multiple query strategies, filtering/ranking methods, and top-k evidence settings.
5. Run each combination across selected/all competitions and multiple runs.
6. Aggregate per-competition and overall scores.

The math extension is intentionally left as a placeholder at the end because it will be added separately.

## 0. Configuration

Start with small settings. Increase `NUM_RUNS_PER_COMBO`, `MAX_WEB_RESULTS_PER_TOOL`, and the number of combinations only after the pipeline works.

In [ ]:
# Safety flags
RUN_LIVE_EXPERIMENTS = True     # Change to True only when you are ready to consume game/API calls.
RUN_SMOKE_TESTS = False          # Useful for quick manual tests after setup.
RUN_FAST = False                 # Keeps defaults small and Colab-friendly.

# Game/API settings
API_URL = "http://131.175.15.22:51111/"
# USERNAME = "11124614"
# USERNAME = "youssef189B"
USERNAME = "amirparsa"

PASSWORD_SECRET_NAME = "poli-millionaire"  # Colab secret name used in your earlier notebook.

# Your package location in Google Drive. The code tries these paths in order.
CANDIDATE_PACKAGE_PARENT_DIRS = [
    "/content/gdrive/MyDrive/Polimi/NLP_Project",
    "/content/gdrive/MyDrive/NLP_Project",
    "/content/gdrive/MyDrive/Poli/NLP_Project",
]

# Live experiment controls
NUM_RUNS_PER_COMBO = 1 if RUN_FAST else 3
SLEEP_BETWEEN_QUESTIONS = 2
SLEEP_BETWEEN_RUNS = 3

# None means all competitions returned by the API after login.
# Example for testing: COMPETITION_IDS_TO_RUN = [0, 1]
COMPETITION_IDS_TO_RUN = None

# Retrieval controls
MAX_WEB_RESULTS_PER_TOOL = 1 if RUN_FAST else 3
MAX_WEBPAGES_TO_FETCH = 0 if RUN_FAST else 1
MAX_CHUNKS_PER_QUESTION = 60 if RUN_FAST else 180
DEFAULT_CHUNK_WORDS = 120
DEFAULT_CHUNK_OVERLAP = 25

# Ranking/filtering controls
DEFAULT_CANDIDATE_POOL = 12 if RUN_FAST else 30


# ============================================================
# Retrieval retry / timeout / rate-limit controls
# Used by the retrieval cell
# ============================================================

# General request timeout for Wikipedia/DDG/Tavily/SerpAPI calls
RETRIEVAL_REQUEST_TIMEOUT = 6

# Default fallback retry values used by retry_call()
DEFAULT_RETRIES = 1
DEFAULT_RETRY_BASE_SLEEP = 0.4
DEFAULT_RETRY_MAX_SLEEP = 1.5

# Wikipedia search API retry controls
WIKI_SEARCH_RETRIES = 1
WIKI_SEARCH_BASE_SLEEP = 0.4
WIKI_SEARCH_MAX_SLEEP = 1.5

# Wikipedia page summary API retry controls
WIKI_SUMMARY_RETRIES = 1
WIKI_SUMMARY_BASE_SLEEP = 0.4
WIKI_SUMMARY_MAX_SLEEP = 1.5

# Optional: max characters kept from each Wikipedia summary/page
WIKI_CHARS_PER_PAGE = 3500

# DuckDuckGo retry controls
DDG_SEARCH_RETRIES = 1
DDG_SEARCH_BASE_SLEEP = 0.35
DDG_SEARCH_MAX_SLEEP = 1.2

# Tavily retry controls
TAVILY_SEARCH_RETRIES = 1
TAVILY_SEARCH_BASE_SLEEP = 0.4
TAVILY_SEARCH_MAX_SLEEP = 1.5

# SerpAPI retry controls
SERPAPI_SEARCH_RETRIES = 1
SERPAPI_SEARCH_BASE_SLEEP = 0.4
SERPAPI_SEARCH_MAX_SLEEP = 1.5

# Webpage fetch controls
WEBPAGE_FETCH_RETRIES = 0
WEBPAGE_FETCH_TIMEOUT = 5
WEBPAGE_FETCH_BASE_SLEEP = 0.3
WEBPAGE_FETCH_MAX_SLEEP = 1.0
WEBPAGE_MAX_CHARS = 6000

# Small pause between retrieval calls inside one question
SLEEP_BETWEEN_RETRIEVAL_CALLS = 0.05


# Model registry.
# backend = "local_seq2seq", "local_causal", or "groq".
MODEL_REGISTRY = {
    "flan_t5_base": {
        "backend": "local_seq2seq",
        "model_id": "google/flan-t5-base",
        "load_in_4bit": False,
        "notes": "Weak/local baseline to show retrieval effect."
    },
    "qwen2_5_1_5b": {
        "backend": "local_causal",
        "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "load_in_4bit": False,
        "notes": "Small local Qwen model. Fast Colab baseline."
    },
    "qwen2_5_3b": {
        "backend": "local_causal",
        "model_id": "Qwen/Qwen2.5-3B-Instruct",
        "load_in_4bit": True,
        "notes": "Stronger local Qwen model."
    },
    "mistral_7b": {
        "backend": "local_causal",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
        "load_in_4bit": True,
        "notes": "Strong local 7B baseline; may require HuggingFace access."
    },
    "groq_llama_3_3_70b": {
        "backend": "groq",
        "model_id": "llama-3.3-70b-versatile",
        "notes": "Fast hosted Groq model from your earlier section."
    },
}

DEFAULT_LLM_KEY = "flan_t5_base" if RUN_FAST else "groq_llama_3_3_70b"

# Speech mode configuration
DEFAULT_GAME_MODE = "text"       # "text" or "speech"

SPEECH_AUDIO_DIR = "speech_audio"
WHISPER_MODEL_NAME = "medium"

'''
We can use "tiny" for faster but weaker transcription, but weaker transcription will not produce anything but false info to both llm, and web search. So it is not used
'''

WHISPER_LANGUAGE = "en"

DISPLAY_SPEECH_AUDIO = False     # True only for manual debugging
SAVE_SPEECH_AUDIO = True

print("Configuration loaded.")
print("Default LLM:", DEFAULT_LLM_KEY, MODEL_REGISTRY[DEFAULT_LLM_KEY])
print("Default game mode:", DEFAULT_GAME_MODE)
print("MAX_WEB_RESULTS_PER_TOOL:", MAX_WEB_RESULTS_PER_TOOL)
print("MAX_WEBPAGES_TO_FETCH:", MAX_WEBPAGES_TO_FETCH)
print("MAX_CHUNKS_PER_QUESTION:", MAX_CHUNKS_PER_QUESTION)
print("SLEEP_BETWEEN_QUESTIONS:", SLEEP_BETWEEN_QUESTIONS)
print("SLEEP_BETWEEN_RETRIEVAL_CALLS:", SLEEP_BETWEEN_RETRIEVAL_CALLS)

Configuration loaded.
Default LLM: groq_llama_3_3_70b {'backend': 'groq', 'model_id': 'llama-3.3-70b-versatile', 'notes': 'Fast hosted Groq model from your earlier section.'}
Default game mode: text
MAX_WEB_RESULTS_PER_TOOL: 3
MAX_WEBPAGES_TO_FETCH: 1
MAX_CHUNKS_PER_QUESTION: 180
SLEEP_BETWEEN_QUESTIONS: 2
SLEEP_BETWEEN_RETRIEVAL_CALLS: 0.05


## 1. Packages and imports

Install optional packages once at the beginning. If you only run Groq and DuckDuckGo, local model packages may still be useful for comparison but not mandatory.

In [ ]:

!pip -q install transformers accelerate sentencepiece bitsandbytes sentence-transformers rank-bm25 wikipedia ddgs beautifulsoup4 lxml requests tqdm groq google-search-results langchain-tavily

# Speech mode dependencies

!pip -q install -U openai-whisper
!apt-get -qq install -y ffmpeg

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 72.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import os
import re
import gc
import json
import time
import math
import random
import warnings
import traceback
import requests
import numpy as np
import pandas as pd
import whisper

from IPython.display import Audio, display
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple
from collections import defaultdict, Counter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    torch = None
    print("Torch import failed:", e)

random.seed(42)
np.random.seed(42)
if torch is not None:
    torch.manual_seed(42)

CUDA available: True
GPU: Tesla T4


In [ ]:
# Lazy Whisper model loader

WHISPER_MODEL = None


def get_whisper_model(model_name=WHISPER_MODEL_NAME):
    """
    Load Whisper only when speech mode is used.
    This avoids loading it during normal text experiments.
    """
    global WHISPER_MODEL

    if WHISPER_MODEL is None:
        print(f"Loading Whisper model: {model_name}")
        WHISPER_MODEL = whisper.load_model(model_name)
        print("Whisper model loaded.")

    return WHISPER_MODEL

## 2. Connect to the live game wrapper

This is the dynamic source of questions. The final experiments use `game.current_question` and `game.answer(...)`; they do not depend on a fixed question bank.

In [ ]:

from google.colab import drive, userdata
import sys

client = None
competitions = []
competition_id_to_name = {}

def find_existing_package_parent(candidate_dirs):
    for parent in candidate_dirs:
        if os.path.exists(os.path.join(parent, "millionaire_client")):
            return parent
    raise FileNotFoundError(
        "Could not find millionaire_client in candidate paths. "
        "Update CANDIDATE_PACKAGE_PARENT_DIRS in the configuration cell."
    )


def setup_game_client():
    global client, competitions, competition_id_to_name

    drive.mount('/content/gdrive/')

    package_parent_dir = find_existing_package_parent(CANDIDATE_PACKAGE_PARENT_DIRS)
    if package_parent_dir not in sys.path:
        sys.path.append(package_parent_dir)

    print("Using package path:", package_parent_dir)

    from millionaire_client import MillionaireClient, AuthenticationError

    password = userdata.get(PASSWORD_SECRET_NAME)
    if password is None:
        raise ValueError(f"Colab secret '{PASSWORD_SECRET_NAME}' was not found.")

    client = MillionaireClient(API_URL)

    try:
        user = client.login(USERNAME, password)
        print(f"Welcome, {user.username}! Role: {user.role}")
    except AuthenticationError as e:
        raise RuntimeError(f"Login failed: {e}")

    competitions = client.competitions.list_all()
    competition_id_to_name = {comp.id: comp.name for comp in competitions}

    print("\nAvailable competitions:")
    for comp in competitions:
        print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")

    return client, competitions


# Run this cell before live experiments.
client, competitions = setup_game_client()

Mounted at /content/gdrive/
Using package path: /content/gdrive/MyDrive/NLP_Project
Welcome, amirparsa! Role: student

Available competitions:
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)
  4: Philosophy and Psychology (15 questions)
  5: News (15 questions)


## 3. Common question formatting and answer parsing

The game API may use option IDs that are not exactly `0,1,2,3`, so the solver predicts a zero-based option index and the runner maps it back to the real API option ID.

In [ ]:

OPTION_IDS = [0, 1, 2, 3]


def clean_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def options_to_text(options):
    return "\n".join([f"{i}. {opt}" for i, opt in enumerate(options)])


def api_question_to_item(question_obj, competition_id=None, competition_name=None):
    options = list(getattr(question_obj, "options", []))
    return {
        "id": str(getattr(question_obj, "id", f"LIVE_{time.time()}")),
        "competition_id": competition_id,
        "competition": competition_name or "Live",
        "question_type": "live_api",
        "question": getattr(question_obj, "text", ""),
        "options": [opt.text for opt in options],
        "option_api_ids": [opt.id for opt in options],
        "expected_option": None,
        "expected_answer": None,
        "keywords": [],
    }


def normalize_question_item(q):
    if isinstance(q, dict):
        return q
    return api_question_to_item(q)


def parse_option_from_text(text, options=None):
    """
    Parse model output into a zero-based option index: 0, 1, 2, or 3.

    Handles:
    - "1"
    - "1."
    - "Option 1"
    - "Final answer: 1"
    - "The answer is 1"
    - "A", "B", "C", "D"
    - option text
    """

    text = str(text).strip()

    if not text:
        return None

    text_clean = text.lower()

    # Remove common wrappers / labels
    text_clean = text_clean.replace("final answer", "final_answer")
    text_clean = text_clean.replace("answer:", " answer ")
    text_clean = text_clean.replace("option:", " option ")
    text_clean = text_clean.replace("choice:", " choice ")
    text_clean = text_clean.replace("[", " ").replace("]", " ")
    text_clean = text_clean.replace("(", " ").replace(")", " ")
    text_clean = text_clean.strip()

    # Case 1: explicit final answer, e.g. "Final answer: 2"
    final_match = re.search(
        r"final_answer\s*[:\-]?\s*([0-3])",
        text_clean,
        flags=re.I
    )
    if final_match:
        return int(final_match.group(1))

    # Case 2: explicit numeric answer after labels
    labeled_match = re.search(
        r"\b(answer|option|choice)\s*[:\-]?\s*([0-3])\b",
        text_clean,
        flags=re.I
    )
    if labeled_match:
        return int(labeled_match.group(2))

    # Case 3: any standalone digit 0..3
    # IMPORTANT: use raw string r"\b[0-3]\b"
    match = re.search(r"\b([0-3])\b", text_clean)
    if match:
        return int(match.group(1))

    # Case 4: letter answer A/B/C/D
    letter_map = {
        "a": 0,
        "b": 1,
        "c": 2,
        "d": 3,
    }

    letter_match = re.search(r"\b([abcd])\b", text_clean)
    if letter_match:
        return letter_map[letter_match.group(1)]

    # Case 5: model outputs option text
    if options:
        for i, opt in enumerate(options):
            opt_low = str(opt).strip().lower()

            if not opt_low:
                continue

            if opt_low in text_clean or text_clean in opt_low:
                return i

    return None


def safe_option(option, fallback=0):
    if option in [0, 1, 2, 3]:
        return int(option)
    return fallback


def map_zero_based_to_api_option_id(predicted_zero_based, question_obj):
    options = list(getattr(question_obj, "options", []))
    idx = safe_option(predicted_zero_based)
    if idx < len(options):
        return options[idx].id
    return options[0].id


def build_direct_prompt(q, prompt_style="strong"):
    q = normalize_question_item(q)
    options_text = options_to_text(q["options"])
    domain = q.get("competition", "General")

    if prompt_style == "direct":
        instruction = "Choose the correct option. Return only one digit: 0, 1, 2, or 3."
    elif prompt_style == "strong":
        instruction = (
            "You are a precise multiple-choice competition assistant. "
            "Compare all options carefully. Return only the option number: 0, 1, 2, or 3. Do not explain."
        )
    elif prompt_style == "elimination":
        instruction = (
            "Check all four options. Eliminate wrong options. "
            "At the end, write exactly: Final answer: <0/1/2/3>."
        )
    else:
        instruction = "Return only the correct option number."

    return f"""
Domain: {domain}

Question:
{q['question']}

Options:
{options_text}

Instruction:
{instruction}
""".strip()

In [ ]:
# Speech mode formatting, transcription cleaning, and mapping

LETTER_TO_ZERO_BASED = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
}

ZERO_BASED_TO_LETTER = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
}


def zero_based_to_letter(answer_idx):
    """
    Converts 0/1/2/3 to A/B/C/D.
    """
    return ZERO_BASED_TO_LETTER.get(answer_idx)


def letter_to_zero_based(answer_letter):
    """
    Converts A/B/C/D to 0/1/2/3.
    """
    if answer_letter is None:
        return None

    return LETTER_TO_ZERO_BASED.get(str(answer_letter).strip().upper())


def clean_transcribed_option(text):
    """
    Cleans normal spoken option prefixes and common Whisper mistakes for Option D.
    """
    if text is None:
        return ""

    text = clean_text(text)

    # Normal option prefixes
    text = re.sub(r"^Option\s+[A-D][\.\,\:\-\!]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^[A-D][\.\,\:\-\!]?\s*", "", text, flags=re.IGNORECASE)

    # Common Whisper mistakes for "Option D"
    text = re.sub(r"^Topsy\s+and\s+D[\.\,\:\-\!]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Topshin\s+D[\.\,\:\-\!]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Topsyndee[\.\,\:\-\!]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Topsyndi[\.\,\:\-\!]?\s*", "", text, flags=re.IGNORECASE)

    return clean_text(text)


def format_numbered_options(options):
    """
    Formats options using the same numeric style expected by the solver.

    Example:
      [0] Quicklime
      [1] Saltpeter
      [2] Petroleum
      [3] Sulfur
    """
    return "\n".join([f"  [{i}] {opt}" for i, opt in enumerate(options)])


def build_q_item_from_speech_transcription(
    question_text,
    option_texts,
    option_api_ids=None,
    competition_id=None,
    competition_name=None,
    question_id=None,
):
    """
    Builds the same q_item structure used by api_question_to_item(),
    but from transcribed speech instead of API text.

    This keeps the downstream solver pipeline unchanged.
    """
    clean_question = clean_text(question_text)
    clean_options = [clean_transcribed_option(x) for x in option_texts]

    if option_api_ids is None:
        option_api_ids = [0, 1, 2, 3]

    return {
        "id": str(question_id or f"SPEECH_{time.time()}"),
        "competition_id": competition_id,
        "competition": competition_name or "Live",
        "question_type": "speech_api",
        "question": clean_question,
        "options": clean_options,
        "option_api_ids": option_api_ids,
        "expected_option": None,
        "expected_answer": None,
        "keywords": [],

        # Extra debug field only; solver does not depend on it
        "numbered_options": format_numbered_options(clean_options),
    }


def map_speech_prediction_to_api_id(predicted_zero_based, option_map):
    """
    Speech mode answer mapping.

    The solver still outputs 0/1/2/3.
    Speech API submission is mapped through letters:

        0 -> A -> option_map["A"] -> API option id
        1 -> B -> option_map["B"] -> API option id
        2 -> C -> option_map["C"] -> API option id
        3 -> D -> option_map["D"] -> API option id
    """
    predicted_zero_based = safe_option(predicted_zero_based)
    predicted_letter = zero_based_to_letter(predicted_zero_based)

    if predicted_letter is None:
        return None, None

    api_answer_id = option_map.get(predicted_letter)

    return predicted_letter, api_answer_id

## 4. LLM registry and unified invocation

All models are called through `invoke_llm(prompt, llm_key=...)`. This lets the experiment configuration switch between FLAN-T5, Qwen, Mistral, and Groq without changing the solver logic.

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from groq import Groq

CURRENT_LOCAL_MODEL = None
CURRENT_LOCAL_TOKENIZER = None
CURRENT_LOCAL_LLM_KEY = None
GROQ_CLIENT = None


def clear_local_model():
    global CURRENT_LOCAL_MODEL, CURRENT_LOCAL_TOKENIZER, CURRENT_LOCAL_LLM_KEY
    if CURRENT_LOCAL_MODEL is not None:
        del CURRENT_LOCAL_MODEL
    if CURRENT_LOCAL_TOKENIZER is not None:
        del CURRENT_LOCAL_TOKENIZER
    CURRENT_LOCAL_MODEL = None
    CURRENT_LOCAL_TOKENIZER = None
    CURRENT_LOCAL_LLM_KEY = None
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_groq_client():
    global GROQ_CLIENT
    if GROQ_CLIENT is not None:
        return GROQ_CLIENT
    key = userdata.get("GROQ_API_KEY") or os.environ.get("GROQ_API_KEY")
    if not key:
        raise ValueError("GROQ_API_KEY was not found in Colab secrets or environment variables.")
    os.environ["GROQ_API_KEY"] = key
    GROQ_CLIENT = Groq(api_key=key)
    return GROQ_CLIENT


def load_local_llm(llm_key):
    global CURRENT_LOCAL_MODEL, CURRENT_LOCAL_TOKENIZER, CURRENT_LOCAL_LLM_KEY

    if CURRENT_LOCAL_LLM_KEY == llm_key and CURRENT_LOCAL_MODEL is not None:
        return CURRENT_LOCAL_TOKENIZER, CURRENT_LOCAL_MODEL

    clear_local_model()
    cfg = MODEL_REGISTRY[llm_key]
    model_id = cfg["model_id"]
    backend = cfg["backend"]

    print("Loading local model:", llm_key, model_id)

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    if backend == "local_seq2seq":
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id, trust_remote_code=True)
        model = model.to("cuda" if torch is not None and torch.cuda.is_available() else "cpu")
    elif backend == "local_causal":
        quant_config = None
        if cfg.get("load_in_4bit", False):
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16 if torch is not None and torch.cuda.is_available() else torch.float32,
            quantization_config=quant_config,
            trust_remote_code=True,
        )
    else:
        raise ValueError(f"Backend {backend} is not a local backend.")

    CURRENT_LOCAL_TOKENIZER = tokenizer
    CURRENT_LOCAL_MODEL = model
    CURRENT_LOCAL_LLM_KEY = llm_key
    print("Loaded:", llm_key)
    return tokenizer, model


def invoke_local_llm(prompt, llm_key, max_new_tokens=64, temperature=0.0):
    cfg = MODEL_REGISTRY[llm_key]
    tokenizer, model = load_local_llm(llm_key)
    backend = cfg["backend"]

    if backend == "local_seq2seq":
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    messages = [{"role": "user", "content": prompt}]
    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
    except Exception:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def invoke_groq_llm(prompt, llm_key, max_new_tokens=64, temperature=0.0):
    cfg = MODEL_REGISTRY[llm_key]
    groq_client = get_groq_client()
    response = groq_client.chat.completions.create(
        model=cfg["model_id"],
        messages=[
            {
                "role": "system",
                "content": "You are a precise multiple-choice competition assistant. Return only the option number unless explicitly asked otherwise.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()


def invoke_llm(prompt, llm_key=DEFAULT_LLM_KEY, max_new_tokens=64, temperature=0.0):
    backend = MODEL_REGISTRY[llm_key]["backend"]
    if backend == "groq":
        return invoke_groq_llm(prompt, llm_key, max_new_tokens=max_new_tokens, temperature=temperature)
    return invoke_local_llm(prompt, llm_key, max_new_tokens=max_new_tokens, temperature=temperature)


if RUN_SMOKE_TESTS:
    print(invoke_llm("Return only the number 2.", llm_key=DEFAULT_LLM_KEY, max_new_tokens=10))

## 5. Query strategies

These are separated from retrieval tools. The same retriever can be tested with different query construction strategies.

In [ ]:

STOPWORDS = {
    "what", "which", "who", "when", "where", "why", "how",
    "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did",
    "the", "a", "an",
    "of", "in", "on", "at", "to", "for", "from", "with", "by", "as",
    "and", "or", "but",
    "this", "that", "these", "those",
    "following", "primary", "main", "general", "fundamental",
    "purpose", "term", "type", "part",
    "statement", "true", "false", "best", "most"
}

DOMAIN_PREFIXES = {
    "Entertainment": "entertainment music film television celebrity media culture",
    "Ancient History and Politics": "ancient history politics civilizations empires rulers archaeology",
    "Science and Nature": "science nature biology chemistry physics astronomy environment",
    "Maths": "mathematics statistics algebra geometry probability logic",
    "Philosophy and Psychology": "philosophy psychology ethics logic cognition behavior mental processes",
    "News": "current events news politics world affairs recent events",
}

COMPETITION_ID_TO_NAME = {
    0: "Entertainment",
    1: "Ancient History and Politics",
    2: "Science and Nature",
    3: "Maths",
    4: "Philosophy and Psychology",
    5: "News",
}


def normalize_text_for_query(text):
    text = str(text).replace("·", " ")
    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_competition_name_from_question(q):
    """
    Returns the competition name if available.

    Supports:
    - offline question dicts with q["competition"]
    - live question dicts where competition_id may be attached later
    - fallback to empty string
    """

    q = normalize_question_item(q)

    if q.get("competition"):
        return q["competition"]

    if q.get("competition_id") is not None:
        return COMPETITION_ID_TO_NAME.get(q["competition_id"], "")

    if q.get("comp_id") is not None:
        return COMPETITION_ID_TO_NAME.get(q["comp_id"], "")

    return ""


def raw_question_query(q):
    q = normalize_question_item(q)
    return q["question"]


def question_plus_options_query(q):
    q = normalize_question_item(q)

    return normalize_text_for_query(
        q["question"] + " " + " ".join(q.get("options", []))
    )


def keyword_condensed_query(q, max_words=12):
    q = normalize_question_item(q)

    text = normalize_text_for_query(
        q["question"] + " " + " ".join(q.get("options", []))
    )

    words = text.split()

    filtered = [
        w for w in words
        if w.lower() not in STOPWORDS and len(w) > 2
    ]

    return " ".join(filtered[:max_words])


def domain_prefixed_query(q):
    q = normalize_question_item(q)

    competition_name = get_competition_name_from_question(q)

    prefix = DOMAIN_PREFIXES.get(
        competition_name,
        normalize_text_for_query(competition_name)
    )

    return normalize_text_for_query(
        f"{prefix} {q['question']}"
    )


def entity_focused_query(q):
    q = normalize_question_item(q)

    if q.get("keywords"):
        return " ".join(q["keywords"])

    return keyword_condensed_query(q, max_words=8)


def llm_rewrite_query(q, llm_key=DEFAULT_LLM_KEY):
    q = normalize_question_item(q)

    competition_name = get_competition_name_from_question(q)

    prompt = f"""
Rewrite this multiple-choice question into a short web search query.

Use only:
- important entities
- concepts
- dates
- technical terms
- domain context if useful

Return only the search query.
Do not answer the question.

Competition:
{competition_name}

Question:
{q['question']}

Options:
{options_to_text(q['options'])}
""".strip()

    try:
        return clean_text(
            invoke_llm(
                prompt,
                llm_key=llm_key,
                max_new_tokens=32,
                temperature=0.0
            )
        )

    except Exception as e:
        print("LLM query rewrite failed:", e)
        return entity_focused_query(q)


QUERY_STRATEGY_REGISTRY = {
    "raw_question": raw_question_query,
    "question_plus_options": question_plus_options_query,
    "keyword_condensed": keyword_condensed_query,
    "domain_prefixed": domain_prefixed_query,
    "entity_focused": entity_focused_query,
    "llm_rewrite": llm_rewrite_query,
}


def generate_queries(
    q,
    query_strategies=("entity_focused", "raw_question"),
    llm_key=DEFAULT_LLM_KEY
):
    queries = []
    seen = set()

    for name in query_strategies:
        fn = QUERY_STRATEGY_REGISTRY[name]

        if name == "llm_rewrite":
            query = fn(q, llm_key=llm_key)
        else:
            query = fn(q)

        query = clean_text(query)

        if query and query.lower() not in seen:
            queries.append({
                "strategy": name,
                "query": query
            })
            seen.add(query.lower())

    return queries


print("Query strategies ready:", list(QUERY_STRATEGY_REGISTRY))
print("Competition prefixes ready:", DOMAIN_PREFIXES)

Query strategies ready: ['raw_question', 'question_plus_options', 'keyword_condensed', 'domain_prefixed', 'entity_focused', 'llm_rewrite']
Competition prefixes ready: {'Entertainment': 'entertainment music film television celebrity media culture', 'Ancient History and Politics': 'ancient history politics civilizations empires rulers archaeology', 'Science and Nature': 'science nature biology chemistry physics astronomy environment', 'Maths': 'mathematics statistics algebra geometry probability logic', 'Philosophy and Psychology': 'philosophy psychology ethics logic cognition behavior mental processes', 'News': 'current events news politics world affairs recent events'}


## 6. Web retrieval tools

The experiment configuration can use one tool or multiple tools. Results are merged before chunking and filtration.

In [ ]:
import os
import time
import random
import requests
import wikipedia

from dataclasses import dataclass
from ddgs import DDGS
from bs4 import BeautifulSoup


NON_RETRYABLE_ERROR_TEXTS = [
    "No results found",
    "no results found",
    "404",
    "not found",
]

RETRYABLE_ERROR_TEXTS = [
    "timed out",
    "timeout",
    "ConnectTimeout",
    "ReadTimeout",
    "RemoteDisconnected",
    "connection",
    "temporarily",
    "rate limit",
    "429",
    "500",
    "502",
    "503",
    "504",
]


def get_config_value(name, default):
    """
    Reads a global config variable if it exists.
    Keeps this cell safe even if an older config cell is used.
    """
    return globals().get(name, default)


def is_retryable_error(error):
    error_text = repr(error)

    for marker in NON_RETRYABLE_ERROR_TEXTS:
        if marker in error_text:
            return False

    for marker in RETRYABLE_ERROR_TEXTS:
        if marker in error_text:
            return True

    return True


def retry_call(
    fn,
    retries=None,
    base_sleep=None,
    max_sleep=None,
    label="operation",
):
    retries = get_config_value("DEFAULT_RETRIES", 1) if retries is None else retries
    base_sleep = get_config_value("DEFAULT_RETRY_BASE_SLEEP", 0.5) if base_sleep is None else base_sleep
    max_sleep = get_config_value("DEFAULT_RETRY_MAX_SLEEP", 2.0) if max_sleep is None else max_sleep

    last_error = None

    for attempt in range(retries + 1):
        try:
            return fn(), None

        except Exception as e:
            last_error = e

            if not is_retryable_error(e):
                print(f"{label} failed with non-retryable error: {repr(e)}")
                return None, e

            if attempt < retries:
                sleep_time = min(max_sleep, base_sleep * (2 ** attempt))
                sleep_time += random.uniform(0, 0.15)

                print(
                    f"{label} failed on attempt {attempt + 1}. "
                    f"Retrying in {sleep_time:.2f}s. Error: {repr(e)}"
                )

                time.sleep(sleep_time)

            else:
                print(
                    f"{label} failed after {retries + 1} attempts. "
                    f"Error: {repr(e)}"
                )

    return None, last_error


@dataclass
class WebDocument:
    source: str
    title: str
    url: str
    text: str
    query: str
    query_strategy: str
    rank: int


def safe_sleep(seconds=None):
    if seconds is None:
        seconds = get_config_value("SLEEP_BETWEEN_RETRIEVAL_CALLS", 0.1)

    time.sleep(seconds)


def safe_request_text(url, timeout=None, max_chars=None):
    """
    Safely fetch and clean webpage text.
    Uses config-cell variables.
    """

    timeout = get_config_value("WEBPAGE_FETCH_TIMEOUT", 5) if timeout is None else timeout
    max_chars = get_config_value("WEBPAGE_MAX_CHARS", 6000) if max_chars is None else max_chars

    def _request():
        headers = {"User-Agent": "Mozilla/5.0 educational research bot"}
        return requests.get(url, headers=headers, timeout=timeout)

    response, error = retry_call(
        _request,
        retries=get_config_value("WEBPAGE_FETCH_RETRIES", 0),
        base_sleep=get_config_value("WEBPAGE_FETCH_BASE_SLEEP", 0.5),
        max_sleep=get_config_value("WEBPAGE_FETCH_MAX_SLEEP", 2.0),
        label=f"Request page | url={url}",
    )

    if response is None:
        return ""

    try:
        if response.status_code != 200 or not response.text:
            return ""

        soup = BeautifulSoup(response.text, "lxml")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        return clean_text(soup.get_text(" "))[:max_chars]

    except Exception as e:
        print(f"Failed to parse page | url={url} | error={repr(e)}")
        return ""


def retrieve_wikipedia(
    query,
    query_strategy="raw_question",
    max_results=None,
    chars_per_page=None,
):
    """
    Wikipedia retriever using official MediaWiki API.
    Uses config-cell variables for timeout, retries, and sleeps.
    """

    max_results = get_config_value("MAX_WEB_RESULTS_PER_TOOL", 2) if max_results is None else max_results
    chars_per_page = get_config_value("WIKI_CHARS_PER_PAGE", 3500) if chars_per_page is None else chars_per_page

    docs = []
    search_url = "https://en.wikipedia.org/w/api.php"

    def _search():
        params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": max_results,
            "utf8": 1,
        }

        headers = {
            "User-Agent": "MCQ-RAG-Experiment/1.0 educational research"
        }

        response = requests.get(
            search_url,
            params=params,
            headers=headers,
            timeout=get_config_value("RETRIEVAL_REQUEST_TIMEOUT", 6),
        )

        response.raise_for_status()
        return response.json()

    search_response, search_error = retry_call(
        _search,
        retries=get_config_value("WIKI_SEARCH_RETRIES", 1),
        base_sleep=get_config_value("WIKI_SEARCH_BASE_SLEEP", 0.8),
        max_sleep=get_config_value("WIKI_SEARCH_MAX_SLEEP", 3.0),
        label=f"Wikipedia API search | query={query}",
    )

    if not search_response:
        return docs

    try:
        results = search_response.get("query", {}).get("search", [])
    except Exception:
        results = []

    if not results:
        return docs

    for rank, item in enumerate(results[:max_results], start=1):
        try:
            title = item.get("title", "")

            if not title:
                continue

            page_url = (
                "https://en.wikipedia.org/api/rest_v1/page/summary/"
                + requests.utils.quote(title)
            )

            def _summary():
                headers = {
                    "User-Agent": "MCQ-RAG-Experiment/1.0 educational research"
                }

                response = requests.get(
                    page_url,
                    headers=headers,
                    timeout=get_config_value("RETRIEVAL_REQUEST_TIMEOUT", 6),
                )

                response.raise_for_status()
                return response.json()

            page_response, page_error = retry_call(
                _summary,
                retries=get_config_value("WIKI_SUMMARY_RETRIES", 2),
                base_sleep=get_config_value("WIKI_SUMMARY_BASE_SLEEP", 1.0),
                max_sleep=get_config_value("WIKI_SUMMARY_MAX_SLEEP", 4.0),
                label=f"Wikipedia API summary | title={title}",
            )

            if not page_response:
                continue

            title = clean_text(page_response.get("title", title))
            url = page_response.get("content_urls", {}).get("desktop", {}).get("page", "")
            extract = clean_text(page_response.get("extract", ""))

            if not extract:
                continue

            docs.append(
                WebDocument(
                    source="wikipedia",
                    title=title,
                    url=url,
                    text=extract[:chars_per_page],
                    query=query,
                    query_strategy=query_strategy,
                    rank=rank,
                )
            )

        except Exception as e:
            print(f"Failed to build Wikipedia API document | title={title} | error={repr(e)}")
            continue

        safe_sleep()

    return docs


def retrieve_duckduckgo(
    query,
    query_strategy="raw_question",
    max_results=None,
    fetch_pages=None,
):
    """
    DuckDuckGo retriever optimized for live timed runs.
    Uses config-cell variables for retries, sleeps, and page fetching.
    """

    max_results = get_config_value("MAX_WEB_RESULTS_PER_TOOL", 2) if max_results is None else max_results
    fetch_pages = get_config_value("MAX_WEBPAGES_TO_FETCH", 0) > 0 if fetch_pages is None else fetch_pages

    docs = []

    def _search():
        with DDGS() as ddgs:
            return list(ddgs.text(query, max_results=max_results))

    search_results, search_error = retry_call(
        _search,
        retries=get_config_value("DDG_SEARCH_RETRIES", 1),
        base_sleep=get_config_value("DDG_SEARCH_BASE_SLEEP", 1.0),
        max_sleep=get_config_value("DDG_SEARCH_MAX_SLEEP", 3.0),
        label=f"DuckDuckGo search | query={query}",
    )

    if not search_results:
        return docs

    max_webpages_to_fetch = get_config_value("MAX_WEBPAGES_TO_FETCH", 0)

    for rank, item in enumerate(search_results, start=1):
        try:
            title = clean_text(item.get("title", ""))
            url = item.get("href", "") or item.get("url", "")
            snippet = clean_text(item.get("body", ""))

            text = snippet

            if fetch_pages and url and rank <= max_webpages_to_fetch:
                page_text = safe_request_text(url)
                if page_text:
                    text = f"{snippet} {page_text}".strip()

            if not text:
                continue

            docs.append(
                WebDocument(
                    source="duckduckgo",
                    title=title,
                    url=url,
                    text=text,
                    query=query,
                    query_strategy=query_strategy,
                    rank=rank,
                )
            )

        except Exception as e:
            print(f"Failed to build DuckDuckGo document | query={query} | error={repr(e)}")
            continue

        safe_sleep()

    return docs


def retrieve_tavily(query, query_strategy="raw_question", max_results=None):
    """
    Tavily retriever with limited retry.
    Uses config-cell variables.
    """

    max_results = get_config_value("MAX_WEB_RESULTS_PER_TOOL", 2) if max_results is None else max_results

    key = (
        userdata.get("Tavili-API-Key")
        or userdata.get("TAVILY_API_KEY")
        or os.environ.get("TAVILY_API_KEY")
    )

    if not key:
        return []

    os.environ["TAVILY_API_KEY"] = key

    try:
        from langchain_tavily import TavilySearch
    except Exception as e:
        print("Could not import TavilySearch:", repr(e))
        return []

    def _search():
        tool = TavilySearch(max_results=max_results, topic="general")
        return tool.invoke({"query": query})

    response, error = retry_call(
        _search,
        retries=get_config_value("TAVILY_SEARCH_RETRIES", 1),
        base_sleep=get_config_value("TAVILY_SEARCH_BASE_SLEEP", 0.8),
        max_sleep=get_config_value("TAVILY_SEARCH_MAX_SLEEP", 3.0),
        label=f"Tavily search | query={query}",
    )

    if not response:
        return []

    try:
        results = response.get("results", [])
    except Exception:
        results = []

    docs = []

    for rank, item in enumerate(results[:max_results], start=1):
        try:
            text = clean_text(item.get("content", ""))

            if not text:
                continue

            docs.append(
                WebDocument(
                    source="tavily",
                    title=clean_text(item.get("title", "")),
                    url=item.get("url", ""),
                    text=text,
                    query=query,
                    query_strategy=query_strategy,
                    rank=rank,
                )
            )

        except Exception as e:
            print(f"Failed to build Tavily document | query={query} | error={repr(e)}")
            continue

        safe_sleep()

    return docs


def retrieve_serpapi(query, query_strategy="raw_question", max_results=None):
    """
    SerpAPI retriever with limited retry.
    Uses config-cell variables.
    """

    max_results = get_config_value("MAX_WEB_RESULTS_PER_TOOL", 2) if max_results is None else max_results

    key = (
        userdata.get("SerpAPI")
        or userdata.get("SERPAPI_API_KEY")
        or os.environ.get("SERPAPI_API_KEY")
    )

    if not key:
        return []

    try:
        from serpapi import GoogleSearch
    except Exception as e:
        print("Could not import GoogleSearch from serpapi:", repr(e))
        return []

    def _search():
        params = {
            "engine": "google",
            "q": query,
            "api_key": key,
            "num": max_results,
        }
        return GoogleSearch(params).get_dict()

    response, error = retry_call(
        _search,
        retries=get_config_value("SERPAPI_SEARCH_RETRIES", 1),
        base_sleep=get_config_value("SERPAPI_SEARCH_BASE_SLEEP", 0.8),
        max_sleep=get_config_value("SERPAPI_SEARCH_MAX_SLEEP", 3.0),
        label=f"SerpAPI search | query={query}",
    )

    if not response:
        return []

    try:
        results = response.get("organic_results", [])
    except Exception:
        results = []

    docs = []

    for rank, item in enumerate(results[:max_results], start=1):
        try:
            text = clean_text(item.get("snippet", ""))

            if not text:
                continue

            docs.append(
                WebDocument(
                    source="serpapi",
                    title=clean_text(item.get("title", "")),
                    url=item.get("link", ""),
                    text=text,
                    query=query,
                    query_strategy=query_strategy,
                    rank=rank,
                )
            )

        except Exception as e:
            print(f"Failed to build SerpAPI document | query={query} | error={repr(e)}")
            continue

        safe_sleep()

    return docs


RETRIEVAL_TOOL_REGISTRY = {
    "wikipedia": retrieve_wikipedia,
    "duckduckgo": retrieve_duckduckgo,
    "tavily": retrieve_tavily,
    "serpapi": retrieve_serpapi,
}


def collect_live_web_documents(
    q,
    query_strategies=("entity_focused", "raw_question"),
    retrieval_tools=("wikipedia", "duckduckgo"),
    max_results_per_tool=None,
    fetch_pages=None,
    llm_key=None,
):
    """
    Collects documents from selected retrievers.

    Uses config-cell defaults:
    - MAX_WEB_RESULTS_PER_TOOL
    - MAX_WEBPAGES_TO_FETCH
    - DEFAULT_LLM_KEY
    """

    q = normalize_question_item(q)

    max_results_per_tool = (
        get_config_value("MAX_WEB_RESULTS_PER_TOOL", 2)
        if max_results_per_tool is None
        else max_results_per_tool
    )

    fetch_pages = (
        get_config_value("MAX_WEBPAGES_TO_FETCH", 0) > 0
        if fetch_pages is None
        else fetch_pages
    )

    llm_key = (
        get_config_value("DEFAULT_LLM_KEY", "qwen2_5_3b")
        if llm_key is None
        else llm_key
    )

    docs = []
    seen = set()

    try:
        queries = generate_queries(
            q,
            query_strategies=query_strategies,
            llm_key=llm_key,
        )
    except Exception as e:
        print("Query generation failed:", repr(e))
        return docs

    for query_item in queries:
        query = query_item.get("query", "")
        strategy = query_item.get("strategy", "unknown")

        if not query:
            continue

        for tool_name in retrieval_tools:
            retriever = RETRIEVAL_TOOL_REGISTRY.get(tool_name)

            if retriever is None:
                print(f"Unknown retriever skipped: {tool_name}")
                continue

            try:
                if tool_name == "duckduckgo":
                    new_docs = retriever(
                        query,
                        query_strategy=strategy,
                        max_results=max_results_per_tool,
                        fetch_pages=fetch_pages,
                    )
                else:
                    new_docs = retriever(
                        query,
                        query_strategy=strategy,
                        max_results=max_results_per_tool,
                    )

            except Exception as e:
                print(f"Retriever {tool_name} failed unexpectedly:", repr(e))
                new_docs = []

            if not new_docs:
                continue

            for d in new_docs:
                try:
                    key = (
                        d.source,
                        d.url,
                        d.title,
                        d.text[:100],
                    )

                    if d.text and key not in seen:
                        docs.append(d)
                        seen.add(key)

                except Exception as e:
                    print(f"Failed while deduplicating document: {repr(e)}")
                    continue

            safe_sleep()

    return docs


print("Retrieval tools ready:", list(RETRIEVAL_TOOL_REGISTRY))
print("MAX_WEB_RESULTS_PER_TOOL:", get_config_value("MAX_WEB_RESULTS_PER_TOOL", 2))
print("MAX_WEBPAGES_TO_FETCH:", get_config_value("MAX_WEBPAGES_TO_FETCH", 0))
print("DEFAULT_LLM_KEY:", get_config_value("DEFAULT_LLM_KEY", "qwen2_5_3b"))

Retrieval tools ready: ['wikipedia', 'duckduckgo', 'tavily', 'serpapi']
MAX_WEB_RESULTS_PER_TOOL: 3
MAX_WEBPAGES_TO_FETCH: 1
DEFAULT_LLM_KEY: groq_llama_3_3_70b


## 7. Chunking

Each question gets a temporary live web corpus. Documents are chunked before filtering/ranking.

In [ ]:

@dataclass
class Chunk:
    chunk_id: str
    source: str
    title: str
    url: str
    text: str
    query: str
    query_strategy: str
    doc_rank: int


def chunk_text(text, chunk_words=DEFAULT_CHUNK_WORDS, overlap_words=DEFAULT_CHUNK_OVERLAP):
    words = clean_text(text).split()
    if not words:
        return []
    chunks = []
    step = max(1, chunk_words - overlap_words)
    for start in range(0, len(words), step):
        part = words[start:start + chunk_words]
        if len(part) < 15:
            continue
        chunks.append(" ".join(part))
    return chunks


def make_chunks_from_docs(docs, max_chunks=MAX_CHUNKS_PER_QUESTION):
    chunks = []
    for doc_index, doc in enumerate(docs):
        for chunk_index, text in enumerate(chunk_text(doc.text)):
            chunks.append(Chunk(
                chunk_id=f"{doc_index}_{chunk_index}",
                source=doc.source,
                title=doc.title,
                url=doc.url,
                text=text,
                query=doc.query,
                query_strategy=doc.query_strategy,
                doc_rank=doc.rank,
            ))
            if len(chunks) >= max_chunks:
                return chunks
    return chunks

print("Chunker ready.")

Chunker ready.


## 8. Filtration and ranking methods

This is the main part from your friend's notebook. The selected method is controlled by `filter_method` in each experiment combination.

In [ ]:

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDER_CACHE = {}
RERANKER_CACHE = {}

EMBEDDING_MODELS = {
    "bge_small": "BAAI/bge-small-en-v1.5",
    "bge_base": "BAAI/bge-base-en-v1.5",
    "e5_base": "intfloat/e5-base-v2",
}

RERANKER_MODELS = {
    "mini_cross_encoder": "cross-encoder/ms-marco-MiniLM-L6-v2",
    "bge_reranker_base": "BAAI/bge-reranker-base",
}


def tokenize_simple(text):
    return re.findall(r"[A-Za-z0-9]+", str(text).lower())


def get_embedder(name="bge_small"):
    if name not in EMBEDDER_CACHE:
        print("Loading embedder:", name, EMBEDDING_MODELS[name])
        device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
        EMBEDDER_CACHE[name] = SentenceTransformer(EMBEDDING_MODELS[name], device=device)
    return EMBEDDER_CACHE[name]


def get_reranker(name="mini_cross_encoder"):
    if name not in RERANKER_CACHE:
        print("Loading reranker:", name, RERANKER_MODELS[name])
        device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
        RERANKER_CACHE[name] = CrossEncoder(RERANKER_MODELS[name], device=device)
    return RERANKER_CACHE[name]


def bm25_rank(query, chunks):
    if not chunks:
        return []
    corpus = [tokenize_simple(c.text) for c in chunks]
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(tokenize_simple(query))
    return sorted([(i, float(s)) for i, s in enumerate(scores)], key=lambda x: x[1], reverse=True)


def dense_rank(query, chunks, embedder_name="bge_small"):
    if not chunks:
        return []
    embedder = get_embedder(embedder_name)
    texts = [c.text for c in chunks]
    if embedder_name.startswith("e5"):
        q_text = "query: " + query
        d_texts = ["passage: " + t for t in texts]
    else:
        q_text = query
        d_texts = texts
    q_emb = embedder.encode([q_text], normalize_embeddings=True, convert_to_numpy=True)
    d_emb = embedder.encode(d_texts, normalize_embeddings=True, convert_to_numpy=True, batch_size=32)
    scores = (d_emb @ q_emb[0]).tolist()
    return sorted([(i, float(s)) for i, s in enumerate(scores)], key=lambda x: x[1], reverse=True)


def minmax_norm(score_dict):
    vals = list(score_dict.values())
    if not vals:
        return {}
    lo, hi = min(vals), max(vals)
    if abs(hi - lo) < 1e-9:
        return {k: 1.0 for k in score_dict}
    return {k: (v - lo) / (hi - lo) for k, v in score_dict.items()}


def hybrid_rank(query, chunks, embedder_name="bge_small", alpha=0.5):
    bm = minmax_norm(dict(bm25_rank(query, chunks)))
    de = minmax_norm(dict(dense_rank(query, chunks, embedder_name=embedder_name)))
    ids = set(bm) | set(de)
    scores = {i: alpha * bm.get(i, 0.0) + (1 - alpha) * de.get(i, 0.0) for i in ids}
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def rrf_fusion(rankings, k=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank_pos, (idx, _) in enumerate(ranking, start=1):
            scores[idx] += 1.0 / (k + rank_pos)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def mmr_select(query, chunks, candidate_indices, embedder_name="bge_small", top_k=5, lambda_mult=0.65):
    if not candidate_indices:
        return []
    embedder = get_embedder(embedder_name)
    texts = [chunks[i].text for i in candidate_indices]
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    d_emb = embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True, batch_size=32)
    rel = d_emb @ q_emb
    selected = []
    selected_local = []
    remaining = list(range(len(candidate_indices)))

    while remaining and len(selected) < top_k:
        if not selected_local:
            best_local = int(remaining[np.argmax([rel[i] for i in remaining])])
        else:
            selected_embs = d_emb[selected_local]
            scores = []
            for i in remaining:
                diversity = np.max(selected_embs @ d_emb[i])
                mmr_score = lambda_mult * rel[i] - (1 - lambda_mult) * diversity
                scores.append(mmr_score)
            best_local = int(remaining[np.argmax(scores)])
        selected_local.append(best_local)
        selected.append(candidate_indices[best_local])
        remaining.remove(best_local)

    return selected


def rerank_cross_encoder(query, chunks, candidate_indices, reranker_name="mini_cross_encoder"):
    if not candidate_indices:
        return []
    reranker = get_reranker(reranker_name)
    pairs = [(query, chunks[i].text) for i in candidate_indices]
    scores = reranker.predict(pairs).tolist()
    return sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)


def select_top_k_chunks(
    query,
    chunks,
    filter_method="bm25",
    top_k=3,
    embedder_name="bge_small",
    use_mmr=False,
    use_reranker=False,
    reranker_name="mini_cross_encoder",
    candidate_pool=DEFAULT_CANDIDATE_POOL,
):
    if not chunks or top_k <= 0:
        return []

    if filter_method in ["none", "raw"]:
        selected_indices = list(range(min(top_k, len(chunks))))
    elif filter_method == "bm25":
        ranking = bm25_rank(query, chunks)
        selected_indices = [idx for idx, _ in ranking[:candidate_pool]]
    elif filter_method == "dense":
        ranking = dense_rank(query, chunks, embedder_name=embedder_name)
        selected_indices = [idx for idx, _ in ranking[:candidate_pool]]
    elif filter_method == "hybrid":
        ranking = hybrid_rank(query, chunks, embedder_name=embedder_name, alpha=0.5)
        selected_indices = [idx for idx, _ in ranking[:candidate_pool]]
    elif filter_method == "rrf":
        rankings = [
            bm25_rank(query, chunks),
            dense_rank(query, chunks, embedder_name="bge_small"),
        ]
        if not RUN_FAST:
            rankings.append(dense_rank(query, chunks, embedder_name="e5_base"))
        ranking = rrf_fusion(rankings)
        selected_indices = [idx for idx, _ in ranking[:candidate_pool]]
    else:
        raise ValueError(f"Unknown filter_method: {filter_method}")

    if use_mmr and filter_method not in ["none", "raw"]:
        selected_indices = mmr_select(
            query,
            chunks,
            selected_indices,
            embedder_name=embedder_name,
            top_k=min(top_k * 3, len(selected_indices)),
        )

    if use_reranker and filter_method not in ["none", "raw"]:
        ranked = rerank_cross_encoder(query, chunks, selected_indices, reranker_name=reranker_name)
        selected_indices = [idx for idx, _ in ranked[:top_k]]
    else:
        selected_indices = selected_indices[:top_k]

    return [chunks[i] for i in selected_indices]

print("Filtration/ranking methods ready.")

Filtration/ranking methods ready.


## 9. Evidence formatting and RAG prompt

In [ ]:

DOMAIN_GUIDANCE = {
    "Entertainment": (
        "Use music, film, television, celebrity, theatre, literature, games, "
        "and media-history evidence carefully."
    ),

    "Ancient History and Politics": (
        "Use historical, archaeological, classical, imperial, civic, legal, "
        "and political evidence carefully."
    ),

    "Science and Nature": (
        "Use science, biology, chemistry, physics, astronomy, earth science, "
        "medicine, animals, plants, and nature evidence carefully."
    ),

    "Maths": (
        "Use arithmetic, algebra, geometry, probability, statistics, logic, "
        "and mathematical reasoning carefully. Prefer exact calculation over retrieval."
    ),

    "Philosophy and Psychology": (
        "Use philosophy, ethics, logic, epistemology, metaphysics, psychology, "
        "cognition, behavior, and mental-process evidence carefully."
    ),

    "News": (
        "Use current-events, world affairs, politics, economics, public figures, "
        "organizations, and recent-news evidence carefully. Prefer recent and reliable sources."
    ),

    "Live": (
        "Use the question topic and retrieved evidence carefully. "
        "Avoid being misled by irrelevant or low-quality text."
    ),
}


def format_evidence(chunks):
    blocks = []
    for i, chunk in enumerate(chunks, start=1):
        blocks.append(f"""
Evidence {i}
Source: {chunk.source}
Query Strategy: {chunk.query_strategy}
Title: {chunk.title}
URL: {chunk.url}
Text: {chunk.text}
""".strip())
    return "\n\n".join(blocks)


def build_rag_prompt(q, chunks, prompt_style="strong"):
    q = normalize_question_item(q)
    options_text = options_to_text(q["options"])
    domain = q.get("competition", "General")
    evidence = format_evidence(chunks)
    guidance = DOMAIN_GUIDANCE.get(domain, "Use evidence when relevant, but avoid being misled by irrelevant text.")

    if prompt_style == "elimination":
        instruction = (
          "Silently check each option against the evidence and the question. "
          "Eliminate options that are contradicted or unsupported. "
          "Do not show your reasoning. "
          "Return ONLY the final option number: 0, 1, 2, or 3."
          "Your entire response must be exactly one digit."
    )
    else:
        instruction = (
            "Choose the single best option. Use the evidence if relevant. "
            "Return only one digit: 0, 1, 2, or 3. Do not explain."
        )

    return f"""
You are answering a multiple-choice competition question.

Domain:
{domain}

Domain guidance:
{guidance}

Question:
{q['question']}

Options:
{options_text}

Retrieved live web evidence:
{evidence if evidence else 'No evidence retrieved.'}

Instruction:
{instruction}
""".strip()

## 10. Generic solver pipeline

This is the central part: every experiment combination is represented as a config object. The same `solve_question_with_config` function handles LLM-only, web-RAG, multi-tool retrieval, different filters, different top-k values, and different LLMs.

In [ ]:

@dataclass
class ExperimentConfig:
    combo_name: str
    llm_key: str = DEFAULT_LLM_KEY
    prompt_style: str = "strong"

    # Game input mode.
    game_mode: str = DEFAULT_GAME_MODE   # "text" or "speech"

    # Retrieval setup. Empty tuple means LLM-only.
    retrieval_tools: Tuple[str, ...] = ()
    query_strategies: Tuple[str, ...] = ("raw_question",)
    fetch_pages: bool = False
    max_results_per_tool: int = MAX_WEB_RESULTS_PER_TOOL

    # Filtering/ranking setup.
    filter_method: str = "none"
    top_k: int = 0
    embedder_name: str = "bge_small"
    use_mmr: bool = False
    use_reranker: bool = False
    reranker_name: str = "mini_cross_encoder"

    # LLM generation setup.
    temperature: float = 0.0
    max_new_tokens: int = 64


def config_to_row(cfg: ExperimentConfig):
    row = asdict(cfg)
    row["retrieval_tools"] = "+".join(cfg.retrieval_tools) if cfg.retrieval_tools else "none"
    row["query_strategies"] = "+".join(cfg.query_strategies) if cfg.query_strategies else "none"
    return row


def solve_question_with_config(q, cfg: ExperimentConfig, return_debug=True):
    q = normalize_question_item(q)
    start = time.time()

    docs = []
    chunks = []
    selected_chunks = []
    retrieval_time = 0.0
    filtering_time = 0.0

    try:
        if cfg.retrieval_tools:
            retrieval_start = time.time()
            docs = collect_live_web_documents(
                q,
                query_strategies=cfg.query_strategies,
                retrieval_tools=cfg.retrieval_tools,
                max_results_per_tool=cfg.max_results_per_tool,
                fetch_pages=cfg.fetch_pages,
                llm_key=cfg.llm_key,
            )
            retrieval_time = time.time() - retrieval_start

            chunks = make_chunks_from_docs(docs)
            filter_query = entity_focused_query(q)

            filtering_start = time.time()
            selected_chunks = select_top_k_chunks(
                query=filter_query,
                chunks=chunks,
                filter_method=cfg.filter_method,
                top_k=cfg.top_k,
                embedder_name=cfg.embedder_name,
                use_mmr=cfg.use_mmr,
                use_reranker=cfg.use_reranker,
                reranker_name=cfg.reranker_name,
            )
            filtering_time = time.time() - filtering_start

            prompt = build_rag_prompt(q, selected_chunks, prompt_style=cfg.prompt_style)
        else:
            prompt = build_direct_prompt(q, prompt_style=cfg.prompt_style)

        raw_answer = invoke_llm(
            prompt,
            llm_key=cfg.llm_key,
            max_new_tokens=cfg.max_new_tokens,
            temperature=cfg.temperature,
        )
        predicted_zero_based = parse_option_from_text(raw_answer, q.get("options", []))
        predicted_zero_based = safe_option(predicted_zero_based)
        error = None

    except Exception as e:
        raw_answer = traceback.format_exc()[-2500:]
        predicted_zero_based = 0
        error = str(e)

    total_time = time.time() - start

    debug = {
        "raw_answer": raw_answer,
        "num_docs": len(docs),
        "num_chunks": len(chunks),
        "num_selected_chunks": len(selected_chunks),
        "selected_titles": [c.title for c in selected_chunks],
        "selected_sources": [c.source for c in selected_chunks],
        "selected_urls": [c.url for c in selected_chunks],
        "retrieval_time_sec": round(retrieval_time, 3),
        "filtering_time_sec": round(filtering_time, 3),
        "total_solver_time_sec": round(total_time, 3),
        "error": error,
    }

    if return_debug:
        return predicted_zero_based, debug
    return predicted_zero_based

print("Generic solver pipeline ready.")

Generic solver pipeline ready.


## 11. Live game runner

This runner is fixed. It receives an `ExperimentConfig`, uses the generic solver, maps the zero-based answer to the API option ID, submits the answer, and logs everything.

In [ ]:
def save_speech_audio_bytes(audio_bytes, audio_path):
    """
    Saves speech audio bytes to disk.
    """
    os.makedirs(os.path.dirname(audio_path), exist_ok=True)

    with open(audio_path, "wb") as f:
        f.write(audio_bytes)

    return audio_path


def transcribe_speech_audio_file(audio_path, whisper_model):
    """
    Transcribes one audio file using Whisper.
    """
    result = whisper_model.transcribe(
        audio_path,
        language=WHISPER_LANGUAGE,
        fp16=False,
    )

    return clean_text(result.get("text", ""))


def fetch_and_transcribe_speech_question(
    game,
    competition_id,
    competition_name,
    run_id,
    whisper_model,
):
    """
    Speech mode flow for one question:

    1. Fetch question audio.
    2. Fetch A/B/C/D option audios.
    3. Save audio files.
    4. Build option_map: A/B/C/D -> API option id.
    5. Transcribe question and options.
    6. Build normal q_item for the existing solver pipeline.
    """

    level = game.current_level
    session_id = getattr(game, "session_id", "unknown_session")

    question_audio_path = None
    option_audio_paths = []
    option_map = {}

    base_dir = os.path.join(
        SPEECH_AUDIO_DIR,
        f"session_{session_id}",
        f"run_{run_id}",
        f"level_{level}",
    )

    # ----------------------------
    # 1. Fetch question audio
    # ----------------------------
    question_audio = game.fetch_audio_question()

    question_audio_path = os.path.join(base_dir, "question.wav")

    if SAVE_SPEECH_AUDIO:
        save_speech_audio_bytes(question_audio, question_audio_path)

    if DISPLAY_SPEECH_AUDIO:
        print("Question audio:")
        display(Audio(question_audio))

    # ----------------------------
    # 2. Fetch option audios A/B/C/D
    # ----------------------------
    for i in range(4):
        letter = zero_based_to_letter(i)

        option_audio = game.fetch_audio_option_next()
        option_audio_path = os.path.join(base_dir, f"option_{letter}.wav")

        if SAVE_SPEECH_AUDIO:
            save_speech_audio_bytes(option_audio, option_audio_path)

        option_audio_paths.append(option_audio_path)

        if DISPLAY_SPEECH_AUDIO:
            print(f"Option {letter} audio:")
            display(Audio(option_audio))

        # Store letter -> real API option id
        if game.current_question and i < len(game.current_question.options):
            option_map[letter] = game.current_question.options[i].id

    # Timer starts after option D is fetched.
    game.refresh_state()

    question_obj = game.current_question
    question_id = getattr(question_obj, "id", f"SPEECH_{time.time()}")

    # ----------------------------
    # 3. Transcribe question and options
    # ----------------------------
    question_text = transcribe_speech_audio_file(
        question_audio_path,
        whisper_model,
    )

    option_texts = [
        transcribe_speech_audio_file(path, whisper_model)
        for path in option_audio_paths
    ]

    option_api_ids = [
        option_map.get("A", 0),
        option_map.get("B", 1),
        option_map.get("C", 2),
        option_map.get("D", 3),
    ]

    q_item = build_q_item_from_speech_transcription(
        question_text=question_text,
        option_texts=option_texts,
        option_api_ids=option_api_ids,
        competition_id=competition_id,
        competition_name=competition_name,
        question_id=question_id,
    )

    speech_debug = {
        "question_audio_path": question_audio_path,
        "option_audio_paths": option_audio_paths,
        "option_map": option_map,
        "transcribed_question": question_text,
        "transcribed_options_raw": option_texts,
        "transcribed_options_clean": q_item["options"],
    }

    return q_item, question_obj, option_map, speech_debug


def play_live_game_with_config(
    client,
    competition_id,
    cfg: ExperimentConfig,
    run_id=1,
    sleep_between_questions=SLEEP_BETWEEN_QUESTIONS,
    verbose=True,
    whisper_model=None,
):
    competition_name = competition_id_to_name.get(competition_id, str(competition_id))

    # Start game based on mode
    if cfg.game_mode == "speech":
        if whisper_model is None:
            raise ValueError(
                "Speech mode requires whisper_model. "
                "Load Whisper once in Section 14 and pass it to play_live_game_with_config()."
            )

        game = client.game.start(competition_id=competition_id, mode="speech")

    else:
        game = client.game.start(competition_id=competition_id)

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"Combo: {cfg.combo_name} | "
            f"Mode: {cfg.game_mode} | "
            f"Competition: {competition_id} - {competition_name} | "
            f"Run: {run_id}"
        )
        print("Session ID:", game.session_id)
        print("=" * 80)

    rows = []

    while game.in_progress:
        level_before_answer = game.current_level

        speech_debug = {
            "question_audio_path": None,
            "option_audio_paths": None,
            "option_map": None,
            "transcribed_question": None,
            "transcribed_options_raw": None,
            "transcribed_options_clean": None,
        }

        option_map = None
        predicted_letter = None

        # ====================================================
        # Build q_item
        # ====================================================

        if cfg.game_mode == "speech":
            try:
                q_item, question_obj, option_map, speech_debug = fetch_and_transcribe_speech_question(
                    game=game,
                    competition_id=competition_id,
                    competition_name=competition_name,
                    run_id=run_id,
                    whisper_model=whisper_model,
                )
            except Exception as e:
                print("Speech fetch/transcription failed:", repr(e))
                break

        else:
            question_obj = game.current_question

            if question_obj is None:
                print("No question available. Game may have ended.")
                break

            q_item = api_question_to_item(
                question_obj,
                competition_id=competition_id,
                competition_name=competition_name,
            )

        # ====================================================
        # Display question
        # ====================================================

        if verbose:
            print(f"\n--- Level {level_before_answer} ---")
            print(q_item["question"])
            print(options_to_text(q_item["options"]))

            try:
                print(f"Time remaining: {game.time_remaining:.1f}s")
            except Exception:
                pass

        # ====================================================
        # Solve using the same generic solver pipeline
        # ====================================================

        solve_start = time.time()
        predicted_zero_based, debug = solve_question_with_config(
            q_item,
            cfg,
            return_debug=True,
        )
        decision_time = time.time() - solve_start

        # ====================================================
        # Map predicted answer to API option id
        # ====================================================

        if cfg.game_mode == "speech":
            predicted_letter, api_option_id = map_speech_prediction_to_api_id(
                predicted_zero_based,
                option_map,
            )

            # Keep current fallback behavior: if mapping fails, submit A/0.
            if api_option_id is None:
                predicted_letter = "A"
                api_option_id = option_map.get("A", 0)

        else:
            api_option_id = map_zero_based_to_api_option_id(
                predicted_zero_based,
                question_obj,
            )

        if verbose:
            print("Predicted zero-based option:", predicted_zero_based)

            if cfg.game_mode == "speech":
                print("Predicted letter:", predicted_letter)
                print("Speech option map:", option_map)

            print("API option id:", api_option_id)
            print("Decision time:", round(decision_time, 3))
            print("Raw answer:", str(debug.get("raw_answer", ""))[:250])

        # ====================================================
        # Submit answer
        # ====================================================

        result = game.answer(api_option_id)

        row = {
            **config_to_row(cfg),
            "competition_id": competition_id,
            "competition_name": competition_name,
            "run_id": run_id,
            "session_id": getattr(game, "session_id", None),
            "level_before_answer": level_before_answer,
            "question": q_item["question"],
            "options": q_item["options"],
            "predicted_zero_based": predicted_zero_based,
            "predicted_letter": predicted_letter,
            "api_option_id": api_option_id,
            "correct": result.correct,
            "timed_out": result.timed_out,
            "earned_amount_after_question": result.earned_amount,
            "game_over_after_question": result.game_over,
            "decision_time_sec": round(decision_time, 3),
            "num_docs": debug.get("num_docs"),
            "num_chunks": debug.get("num_chunks"),
            "num_selected_chunks": debug.get("num_selected_chunks"),
            "selected_titles": debug.get("selected_titles"),
            "selected_sources": debug.get("selected_sources"),
            "selected_urls": debug.get("selected_urls"),
            "retrieval_time_sec": debug.get("retrieval_time_sec"),
            "filtering_time_sec": debug.get("filtering_time_sec"),
            "raw_answer": debug.get("raw_answer"),
            "error": debug.get("error"),

            # Speech debug fields
            "question_audio_path": speech_debug.get("question_audio_path"),
            "option_audio_paths": speech_debug.get("option_audio_paths"),
            "speech_option_map": speech_debug.get("option_map"),
            "transcribed_question": speech_debug.get("transcribed_question"),
            "transcribed_options_raw": speech_debug.get("transcribed_options_raw"),
            "transcribed_options_clean": speech_debug.get("transcribed_options_clean"),
        }

        rows.append(row)

        if verbose:
            print(
                "Correct:",
                result.correct,
                "Timed out:",
                result.timed_out,
                "Earned:",
                result.earned_amount,
            )

        if result.game_over:
            break

        safe_sleep(sleep_between_questions)

    question_df = pd.DataFrame(rows)

    run_summary = {
        **config_to_row(cfg),
        "competition_id": competition_id,
        "competition_name": competition_name,
        "run_id": run_id,
        "questions_answered": len(question_df),
        "correct_answers": int(question_df["correct"].sum()) if not question_df.empty else 0,
        "timeout_count": int(question_df["timed_out"].sum()) if not question_df.empty else 0,
        "final_earned_amount": float(question_df["earned_amount_after_question"].iloc[-1]) if not question_df.empty else 0.0,
        "max_level_reached": int(question_df["level_before_answer"].max()) if not question_df.empty else 0,
        "avg_decision_time_sec": float(question_df["decision_time_sec"].mean()) if not question_df.empty else None,
    }

    if verbose:
        print("\nRun summary:")
        print(json.dumps(run_summary, indent=2))

    return question_df, run_summary


print("Live game runner ready.")

Live game runner ready.


## 12. Experiment combinations

Start small. The default list is intentionally limited. After you confirm the notebook runs correctly, add more combinations.

### EXPERIMENT_CONFIGS

In [ ]:
EXPERIMENT_CONFIGS = [

    # ========================================================
    # A) LLM-only baselines
    # ========================================================

    ExperimentConfig(
        combo_name="flan_llm_only_strong",
        llm_key="flan_t5_base",
        prompt_style="strong",
        retrieval_tools=(),
        query_strategies=(),
        filter_method="none",
        top_k=0,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen1_5b_llm_only_strong",
        llm_key="qwen2_5_1_5b",
        prompt_style="strong",
        retrieval_tools=(),
        query_strategies=(),
        filter_method="none",
        top_k=0,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_llm_only_strong",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=(),
        query_strategies=(),
        filter_method="none",
        top_k=0,
        max_new_tokens=32,
    ),

    # Groq disabled
    # ExperimentConfig(
    #     combo_name="groq_llama70b_llm_only_strong",
    #     llm_key="groq_llama_3_3_70b",
    #     prompt_style="strong",
    #     retrieval_tools=(),
    #     query_strategies=(),
    #     filter_method="none",
    #     top_k=0,
    #     max_new_tokens=20,
    # ),


    # ========================================================
    # A.1) Sanity-check aliases used by cluster 00
    # ========================================================

    ExperimentConfig(
        combo_name="qwen3b_sanity_llm_only",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=(),
        query_strategies=(),
        filter_method="none",
        top_k=0,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_sanity_wiki_bm25_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_sanity_ddg_bm25_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),


    # ========================================================
    # B) Wikipedia-only retrieval
    # ========================================================

    ExperimentConfig(
        combo_name="flan_wiki_bm25_k3_raw_options",
        llm_key="flan_t5_base",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("raw_question", "question_plus_options"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen1_5b_wiki_bm25_k3_entity_raw",
        llm_key="qwen2_5_1_5b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_bm25_k5_entity_domain",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused", "domain_prefixed"),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    # Groq disabled
    # ExperimentConfig(
    #     combo_name="groq_wiki_bm25_k3_entity_raw",
    #     llm_key="groq_llama_3_3_70b",
    #     prompt_style="strong",
    #     retrieval_tools=("wikipedia",),
    #     query_strategies=("entity_focused", "raw_question"),
    #     filter_method="bm25",
    #     top_k=3,
    #     max_new_tokens=20,
    # ),


    # ========================================================
    # C) DuckDuckGo-only retrieval
    # ========================================================

    ExperimentConfig(
        combo_name="flan_ddg_bm25_k3_entity_raw",
        llm_key="flan_t5_base",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen1_5b_ddg_bm25_k3_entity_raw",
        llm_key="qwen2_5_1_5b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_bm25_k5_entity_options",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "question_plus_options"),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    # Groq disabled
    # ExperimentConfig(
    #     combo_name="groq_ddg_bm25_k3_entity_raw",
    #     llm_key="groq_llama_3_3_70b",
    #     prompt_style="strong",
    #     retrieval_tools=("duckduckgo",),
    #     query_strategies=("entity_focused", "raw_question"),
    #     filter_method="bm25",
    #     top_k=3,
    #     max_new_tokens=20,
    # ),


    # ========================================================
    # D) Query strategy ablation — local Qwen3B replacements
    # ========================================================

    ExperimentConfig(
        combo_name="qwen3b_ddg_raw_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("raw_question",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_options_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("question_plus_options",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_keyword_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("keyword_condensed",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_entity_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_domain_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("domain_prefixed",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_entity_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_keyword_only_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("keyword_condensed",),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    # Groq query ablation disabled
    # groq_ddg_raw_only_k3
    # groq_ddg_options_only_k3
    # groq_ddg_keyword_only_k3
    # groq_ddg_entity_only_k3
    # groq_ddg_domain_only_k3
    # groq_ddg_llm_rewrite_only_k3


    # ========================================================
    # E) Multi-query retrieval — local only
    # ========================================================

    ExperimentConfig(
        combo_name="qwen1_5b_ddg_multiquery_bm25_k5",
        llm_key="qwen2_5_1_5b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=(
            "entity_focused",
            "keyword_condensed",
            "raw_question",
        ),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_multiquery_bm25_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=(
            "entity_focused",
            "keyword_condensed",
            "raw_question",
        ),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_multiquery_hybrid_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=(
            "entity_focused",
            "question_plus_options",
            "domain_prefixed",
        ),
        filter_method="hybrid",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_multiquery_rrf_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=(
            "entity_focused",
            "keyword_condensed",
            "raw_question",
            "question_plus_options",
        ),
        filter_method="rrf",
        top_k=5,
        max_new_tokens=32,
    ),

    # Groq multi-query disabled
    # groq_ddg_multiquery_bm25_k5
    # groq_ddg_multiquery_hybrid_k5


    # ========================================================
    # F) Multi-tool retrieval — Wikipedia + DuckDuckGo local only
    # ========================================================

    ExperimentConfig(
        combo_name="qwen1_5b_wiki_ddg_bm25_k5",
        llm_key="qwen2_5_1_5b",
        prompt_style="strong",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "raw_question",
            "question_plus_options",
        ),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_bm25_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "raw_question",
            "question_plus_options",
        ),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_rrf_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "domain_prefixed",
            "raw_question",
        ),
        filter_method="rrf",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_rrf_mmr_k3",
        llm_key="qwen2_5_3b",
        prompt_style="elimination",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "domain_prefixed",
            "raw_question",
        ),
        filter_method="rrf",
        top_k=3,
        use_mmr=True,
        max_new_tokens=40,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_hybrid_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "question_plus_options",
            "domain_prefixed",
        ),
        filter_method="hybrid",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_hybrid_mmr_k3",
        llm_key="qwen2_5_3b",
        prompt_style="elimination",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "question_plus_options",
            "domain_prefixed",
        ),
        filter_method="hybrid",
        top_k=3,
        use_mmr=True,
        max_new_tokens=40,
    ),

    # Groq multi-tool disabled
    # groq_wiki_ddg_bm25_k5
    # groq_wiki_ddg_rrf_k5
    # groq_wiki_ddg_rrf_mmr_k3


    # ========================================================
    # G) Prompt-style comparison — local Qwen3B replacements
    # ========================================================

    ExperimentConfig(
        combo_name="qwen3b_ddg_strong_prompt_k3",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_elimination_prompt_k3",
        llm_key="qwen2_5_3b",
        prompt_style="elimination",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=40,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_strong_rrf_k5",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "question_plus_options",
            "domain_prefixed",
        ),
        filter_method="rrf",
        top_k=5,
        use_mmr=True,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_ddg_elimination_rrf_k5",
        llm_key="qwen2_5_3b",
        prompt_style="elimination",
        retrieval_tools=("wikipedia", "duckduckgo"),
        query_strategies=(
            "entity_focused",
            "question_plus_options",
            "domain_prefixed",
        ),
        filter_method="rrf",
        top_k=5,
        use_mmr=True,
        max_new_tokens=40,
    ),

    # Groq prompt-style disabled
    # groq_ddg_strong_prompt_k3
    # groq_ddg_elimination_prompt_k3
    # groq_wiki_ddg_elimination_rrf_k5


    # ========================================================
    # H) top_k comparison — local Qwen3B replacements
    # ========================================================

    ExperimentConfig(
        combo_name="qwen3b_ddg_bm25_k1_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=1,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_bm25_k3_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_ddg_bm25_k5_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_bm25_k1_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=1,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_bm25_k3_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),

    ExperimentConfig(
        combo_name="qwen3b_wiki_bm25_k5_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        retrieval_tools=("wikipedia",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=5,
        max_new_tokens=32,
    ),

    # Groq top_k disabled
    # groq_ddg_bm25_k1_entity_raw
    # groq_ddg_bm25_k3_entity_raw
    # groq_ddg_bm25_k5_entity_raw


    # ========================================================
    # I) Optional paid/credit retrievers
    # Keep disabled unless needed.
    # ========================================================

    # Groq + Tavily / SerpAPI disabled
    # groq_tavily_bm25_k5_entity_raw
    # groq_tavily_hybrid_k5_multiquery
    # groq_serpapi_bm25_k5_entity_raw
    # groq_wiki_tavily_rrf_mmr_k5
]


MASTER_EXPERIMENT_CONFIGS = EXPERIMENT_CONFIGS.copy()

configs_preview_df = pd.DataFrame([config_to_row(cfg) for cfg in EXPERIMENT_CONFIGS])
display(configs_preview_df)

,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,flan_llm_only_strong,flan_t5_base,strong,text,none,none,False,3,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,text,none,none,False,3,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
2,qwen3b_llm_only_strong,qwen2_5_3b,strong,text,none,none,False,3,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
3,qwen3b_sanity_llm_only,qwen2_5_3b,strong,text,none,none,False,3,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
4,qwen3b_sanity_wiki_bm25_k3,qwen2_5_3b,strong,text,wikipedia,entity_focused+raw_question,False,3,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
5,qwen3b_sanity_ddg_bm25_k3,qwen2_5_3b,strong,text,duckduckgo,entity_focused+raw_question,False,3,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
6,flan_wiki_bm25_k3_raw_options,flan_t5_base,strong,text,wikipedia,raw_question+question_plus_options,False,3,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
7,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,text,wikipedia,entity_focused+raw_question,False,3,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
8,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,strong,text,wikipedia,entity_focused+domain_prefixed,False,3,bm25,5,bge_small,False,False,mini_cross_encoder,0.0,32
9,flan_ddg_bm25_k3_entity_raw,flan_t5_base,strong,text,duckduckgo,entity_focused+raw_question,False,3,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32


### Clusters

In [ ]:
# ============================================================
# 13B-alt. Combo clusters without Groq
# Same cluster names/numbers, Groq combos commented out
# ============================================================

ALL_EXPERIMENT_CONFIGS = EXPERIMENT_CONFIGS.copy()

EXPERIMENT_CLUSTERS = {
    # --------------------------------------------------------
    # 00: Sanity check — quick end-to-end test with local models.
    # LLM-only + minimal Wikipedia + minimal DuckDuckGo.
    # Run this first to verify the full pipeline works.
    # --------------------------------------------------------
    "00_sanity_check": [
        "qwen3b_sanity_llm_only",
        "qwen3b_sanity_wiki_bm25_k3",
        "qwen3b_sanity_ddg_bm25_k3",
        # Groq variants (re-enable if Groq key is available):
        # "groq_llama70b_llm_only_strong",
        # "groq_wiki_bm25_k3_entity_raw",
        # "groq_ddg_entity_only_k3",
    ],

    # --------------------------------------------------------
    # 01: LLM-only baselines — no retrieval at all.
    # Shows how much each model knows without web evidence.
    # --------------------------------------------------------
    "01_llm_only_baselines": [
        "flan_llm_only_strong",
        "qwen1_5b_llm_only_strong",
        "qwen3b_llm_only_strong",
        # "groq_llama70b_llm_only_strong",
    ],

    # --------------------------------------------------------
    # 02: Wikipedia-only retrieval.
    # Strong for science, history, geography, politics.
    # --------------------------------------------------------
    "02_wikipedia_only": [
        "flan_wiki_bm25_k3_raw_options",
        "qwen1_5b_wiki_bm25_k3_entity_raw",
        "qwen3b_wiki_bm25_k5_entity_domain",
        # "groq_wiki_bm25_k3_entity_raw",
    ],

    # --------------------------------------------------------
    # 03: DuckDuckGo model comparison.
    # Better for entertainment, media, current events.
    # --------------------------------------------------------
    "03_duckduckgo_model_comparison": [
        "flan_ddg_bm25_k3_entity_raw",
        "qwen1_5b_ddg_bm25_k3_entity_raw",
        "qwen3b_ddg_bm25_k5_entity_options",
        # "groq_ddg_bm25_k3_entity_raw",
    ],

    # --------------------------------------------------------
    # 04: Query strategy ablation — local models only.
    # Fixed: qwen3b + duckduckgo + bm25 + k=3.
    # Variable: query construction strategy.
    # Also includes wikipedia variants to compare tool effect.
    # --------------------------------------------------------
    "04_query_strategy_ablation": [
        "qwen3b_ddg_raw_only_k3",
        "qwen3b_ddg_options_only_k3",
        "qwen3b_ddg_keyword_only_k3",
        "qwen3b_ddg_entity_only_k3",
        "qwen3b_ddg_domain_only_k3",
        "qwen3b_wiki_entity_only_k3",
        "qwen3b_wiki_keyword_only_k3",
        # Groq variants (re-enable if key is available):
        # "groq_ddg_raw_only_k3",
        # "groq_ddg_options_only_k3",
        # "groq_ddg_keyword_only_k3",
        # "groq_ddg_entity_only_k3",
        # "groq_ddg_domain_only_k3",
        # "groq_ddg_llm_rewrite_only_k3",
    ],

    # --------------------------------------------------------
    # 05: Multi-query retrieval.
    # Combines several query formulations to maximise recall.
    # --------------------------------------------------------
    "05_multiquery_retrieval": [
        "qwen1_5b_ddg_multiquery_bm25_k5",
        "qwen3b_ddg_multiquery_bm25_k5",
        "qwen3b_ddg_multiquery_hybrid_k5",
        "qwen3b_ddg_multiquery_rrf_k5",
    ],

    # --------------------------------------------------------
    # 06: Multi-tool (Wikipedia + DuckDuckGo).
    # Usually one of the strongest RAG setups.
    # --------------------------------------------------------
    "06_multitool_wiki_ddg": [
        "qwen1_5b_wiki_ddg_bm25_k5",
        "qwen3b_wiki_ddg_bm25_k5",
        "qwen3b_wiki_ddg_rrf_k5",
        "qwen3b_wiki_ddg_rrf_mmr_k3",
        "qwen3b_wiki_ddg_hybrid_k5",
        "qwen3b_wiki_ddg_hybrid_mmr_k3",
    ],

    # --------------------------------------------------------
    # 07: Prompt-style comparison — local models only.
    # Does elimination-style reasoning help over direct prompting?
    # Fixed: qwen3b + duckduckgo/wiki+ddg + bm25/rrf + k=3/5.
    # --------------------------------------------------------
    "07_prompt_style_comparison": [
        "qwen3b_ddg_strong_prompt_k3",
        "qwen3b_ddg_elimination_prompt_k3",
        "qwen3b_wiki_ddg_strong_rrf_k5",
        "qwen3b_wiki_ddg_elimination_rrf_k5",
        # Groq variants:
        # "groq_ddg_strong_prompt_k3",
        # "groq_ddg_elimination_prompt_k3",
        # "groq_wiki_ddg_elimination_rrf_k5",
    ],

    # --------------------------------------------------------
    # 08: top_k comparison — local models only.
    # How many evidence chunks is best?
    # Fixed: qwen3b + entity_focused+raw_question + bm25.
    # Variable: top_k ∈ {1, 3, 5} for both DuckDuckGo and Wikipedia.
    # --------------------------------------------------------
    "08_top_k_comparison": [
        "qwen3b_ddg_bm25_k1_entity_raw",
        "qwen3b_ddg_bm25_k3_entity_raw",
        "qwen3b_ddg_bm25_k5_entity_raw",
        "qwen3b_wiki_bm25_k1_entity_raw",
        "qwen3b_wiki_bm25_k5_entity_raw",
        # Groq variants:
        # "groq_ddg_bm25_k1_entity_raw",
        # "groq_ddg_bm25_k3_entity_raw",
        # "groq_ddg_bm25_k5_entity_raw",
    ],


}


def select_cluster(cluster_name):
    selected_names = EXPERIMENT_CLUSTERS[cluster_name]

    selected_configs = [
        cfg for cfg in MASTER_EXPERIMENT_CONFIGS
        if cfg.combo_name in selected_names
    ]

    found_names = [cfg.combo_name for cfg in selected_configs]
    missing_names = [name for name in selected_names if name not in found_names]

    print("Selected cluster:", cluster_name)
    print("Configs found:", len(selected_configs))

    if missing_names:
        print("Missing configs:")
        for name in missing_names:
            print("-", name)

    if selected_configs:
        display(pd.DataFrame([config_to_row(cfg) for cfg in selected_configs]))
    else:
        print("This cluster is empty after filtering.")

    return selected_configs

## 13. Run all combinations across competitions and multiple runs

This is the main final experiment loop. It is disabled by default.

### Reusable functions

In [ ]:
MASTER_EXPERIMENT_CONFIGS = EXPERIMENT_CONFIGS.copy()

In [ ]:
# ============================================================
# Save helper
# ============================================================

def save_cluster_results(
    cluster_name,
    run_summary_df,
    question_level_results_df,
    per_competition_summary_df,
    overall_summary_df,
):
    import os

    OUTPUT_DIR = "experiment_outputs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    safe_cluster_name = cluster_name.replace("/", "_").replace(" ", "_")

    if run_summary_df is not None and not run_summary_df.empty:
        path = f"{OUTPUT_DIR}/{safe_cluster_name}_run_summary.csv"
        run_summary_df.to_csv(path, index=False)
        print("Saved:", path)

    if question_level_results_df is not None and not question_level_results_df.empty:
        path = f"{OUTPUT_DIR}/{safe_cluster_name}_question_results.csv"
        question_level_results_df.to_csv(path, index=False)
        print("Saved:", path)

    if per_competition_summary_df is not None and not per_competition_summary_df.empty:
        path = f"{OUTPUT_DIR}/{safe_cluster_name}_per_competition_summary.csv"
        per_competition_summary_df.to_csv(path, index=False)
        print("Saved:", path)

    if overall_summary_df is not None and not overall_summary_df.empty:
        path = f"{OUTPUT_DIR}/{safe_cluster_name}_overall_summary.csv"
        overall_summary_df.to_csv(path, index=False)
        print("Saved:", path)

# ============================================================
# 15A. Aggregation helper functions
# ============================================================

def aggregate_per_competition(run_summary_df):
    if run_summary_df is None or run_summary_df.empty:
        return pd.DataFrame()

    group_cols = [
        "cluster_name",
        "combo_name",
        "llm_key",
        "retrieval_tools",
        "query_strategies",
        "filter_method",
        "top_k",
        "use_mmr",
        "use_reranker",
        "competition_id",
        "competition_name",
    ]

    group_cols = [c for c in group_cols if c in run_summary_df.columns]

    agg = (
        run_summary_df
        .groupby(group_cols, dropna=False)
        .agg(
            runs=("run_id", "count"),
            avg_max_level_reached=("max_level_reached", "mean"),
            std_max_level_reached=("max_level_reached", "std"),
            avg_correct_answers=("correct_answers", "mean"),
            avg_final_earned_amount=("final_earned_amount", "mean"),
            std_final_earned_amount=("final_earned_amount", "std"),
            avg_decision_time_sec=("avg_decision_time_sec", "mean"),
            total_timeouts=("timeout_count", "sum"),
            failed_runs=("run_error", lambda s: s.notna().sum() if "run_error" in run_summary_df.columns else 0),
        )
        .reset_index()
    )

    round_cols = [
        "avg_max_level_reached",
        "std_max_level_reached",
        "avg_correct_answers",
        "avg_final_earned_amount",
        "std_final_earned_amount",
        "avg_decision_time_sec",
    ]

    for col in round_cols:
        if col in agg.columns:
            agg[col] = agg[col].round(3)

    return agg.sort_values(
        ["avg_final_earned_amount", "avg_max_level_reached"],
        ascending=[False, False],
    )


def aggregate_overall(run_summary_df):
    if run_summary_df is None or run_summary_df.empty:
        return pd.DataFrame()

    group_cols = [
        "cluster_name",
        "combo_name",
        "llm_key",
        "retrieval_tools",
        "query_strategies",
        "filter_method",
        "top_k",
        "use_mmr",
        "use_reranker",
    ]

    group_cols = [c for c in group_cols if c in run_summary_df.columns]

    agg = (
        run_summary_df
        .groupby(group_cols, dropna=False)
        .agg(
            total_runs=("run_id", "count"),
            competitions_tested=("competition_id", "nunique"),
            avg_max_level_reached=("max_level_reached", "mean"),
            std_max_level_reached=("max_level_reached", "std"),
            avg_correct_answers=("correct_answers", "mean"),
            avg_final_earned_amount=("final_earned_amount", "mean"),
            std_final_earned_amount=("final_earned_amount", "std"),
            avg_decision_time_sec=("avg_decision_time_sec", "mean"),
            total_timeouts=("timeout_count", "sum"),
            failed_runs=("run_error", lambda s: s.notna().sum() if "run_error" in run_summary_df.columns else 0),
        )
        .reset_index()
    )

    round_cols = [
        "avg_max_level_reached",
        "std_max_level_reached",
        "avg_correct_answers",
        "avg_final_earned_amount",
        "std_final_earned_amount",
        "avg_decision_time_sec",
    ]

    for col in round_cols:
        if col in agg.columns:
            agg[col] = agg[col].round(3)

    return agg.sort_values(
        ["avg_final_earned_amount", "avg_max_level_reached"],
        ascending=[False, False],
    )

In [ ]:
# ============================================================
# Cluster runner helpers: run first, summarize separately
# ============================================================

import time
import traceback
import pandas as pd


# ------------------------------------------------------------
# 1. Resolve competitions
# ------------------------------------------------------------

def resolve_competition_ids():
    if COMPETITION_IDS_TO_RUN is not None:
        return COMPETITION_IDS_TO_RUN

    return [comp.id for comp in competitions]


# ------------------------------------------------------------
# 2. Speed helpers
# ------------------------------------------------------------

def is_local_config(cfg):
    backend = MODEL_REGISTRY[cfg.llm_key]["backend"]
    return backend in ["local_seq2seq", "local_causal"]


def sort_configs_for_speed(experiment_configs):
    """
    Groups configs by backend + llm_key so local models are reused.
    """
    return sorted(
        experiment_configs,
        key=lambda cfg: (
            MODEL_REGISTRY[cfg.llm_key]["backend"],
            cfg.llm_key,
            cfg.combo_name,
        ),
    )


def print_gpu_status():
    if torch is not None and torch.cuda.is_available():
        print("GPU available:", torch.cuda.get_device_name(0))
        print("CUDA memory allocated:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
        print("CUDA memory reserved:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")
    else:
        print("GPU not available. Local models will run on CPU.")


# ------------------------------------------------------------
# 3. Main experiment grid runner
# ------------------------------------------------------------

def run_live_experiment_grid(
    client,
    experiment_configs,
    competition_ids=None,
    num_runs=NUM_RUNS_PER_COMBO,
    verbose=False,
    sleep_between_runs=0.1,
    sort_for_speed=True,
):
    if competition_ids is None:
        competition_ids = resolve_competition_ids()

    if sort_for_speed:
        experiment_configs = sort_configs_for_speed(experiment_configs)

    all_question_dfs = []
    run_summaries = []

    total_runs = len(experiment_configs) * len(competition_ids) * num_runs
    counter = 0
    last_llm_key = None

    print("Total runs:", total_runs)
    print("Competitions:", competition_ids)
    print_gpu_status()

    start_grid_time = time.time()

    for cfg in experiment_configs:
        current_backend = MODEL_REGISTRY[cfg.llm_key]["backend"]

        # Clear old local model when switching local LLMs
        if last_llm_key is not None and cfg.llm_key != last_llm_key:
            previous_backend = MODEL_REGISTRY[last_llm_key]["backend"]

            if previous_backend in ["local_seq2seq", "local_causal"]:
                clear_local_model()

        last_llm_key = cfg.llm_key

        # Preload local model once per config
        if current_backend in ["local_seq2seq", "local_causal"]:
            print(f"\nPreloading local model for GPU reuse: {cfg.llm_key}")

            try:
                load_local_llm(cfg.llm_key)
                print_gpu_status()
            except Exception as e:
                print("Local model preload failed:", repr(e))

        for competition_id in competition_ids:
            for run_id in range(1, num_runs + 1):
                counter += 1
                run_start_time = time.time()

                print(
                    f"\n[{counter}/{total_runs}] "
                    f"{cfg.combo_name} | competition={competition_id} | run={run_id}"
                )

                try:
                    q_df, summary = play_live_game_with_config(
                        client=client,
                        competition_id=competition_id,
                        cfg=cfg,
                        run_id=run_id,
                        verbose=verbose,
                    )

                    if summary is None:
                        summary = {
                            **config_to_row(cfg),
                            "competition_id": competition_id,
                            "competition_name": competition_id_to_name.get(
                                competition_id,
                                str(competition_id),
                            ),
                            "run_id": run_id,
                            "questions_answered": 0,
                            "correct_answers": 0,
                            "timeout_count": 0,
                            "final_earned_amount": 0.0,
                            "max_level_reached": 0,
                            "avg_decision_time_sec": None,
                            "run_error": "summary returned None",
                        }

                except Exception as e:
                    print("Run failed:", repr(e))

                    q_df = pd.DataFrame()

                    summary = {
                        **config_to_row(cfg),
                        "competition_id": competition_id,
                        "competition_name": competition_id_to_name.get(
                            competition_id,
                            str(competition_id),
                        ),
                        "run_id": run_id,
                        "questions_answered": 0,
                        "correct_answers": 0,
                        "timeout_count": 0,
                        "final_earned_amount": 0.0,
                        "max_level_reached": 0,
                        "avg_decision_time_sec": None,
                        "run_error": traceback.format_exc()[-2500:],
                    }

                if q_df is not None and not q_df.empty:
                    all_question_dfs.append(q_df)

                run_summaries.append(summary)

                elapsed = time.time() - run_start_time

                print(
                    f"Finished in {elapsed:.2f}s | "
                    f"level={summary.get('max_level_reached')} | "
                    f"correct={summary.get('correct_answers')} | "
                    f"earned={summary.get('final_earned_amount')} | "
                    f"error={summary.get('run_error') is not None}"
                )

                if sleep_between_runs and sleep_between_runs > 0:
                    safe_sleep(sleep_between_runs)

    total_elapsed = time.time() - start_grid_time

    question_level_df = (
        pd.concat(all_question_dfs, ignore_index=True)
        if all_question_dfs
        else pd.DataFrame()
    )

    run_summary_df = pd.DataFrame(run_summaries)

    print("\nFinished experiment grid.")
    print("Total elapsed time:", round(total_elapsed, 2), "seconds")
    print("Question-level rows:", len(question_level_df))
    print("Run-summary rows:", len(run_summary_df))

    return question_level_df, run_summary_df


# ------------------------------------------------------------
# 4. Run cluster only
# ------------------------------------------------------------

def run_cluster(
    cluster_name,
    num_runs=None,
    competition_ids=None,
    verbose=False,
    sleep_between_runs=0.1,
    sort_for_speed=True,
):
    global CLUSTER_TO_RUN
    global EXPERIMENT_CONFIGS
    global question_level_results_df
    global run_summary_df

    CLUSTER_TO_RUN = cluster_name
    EXPERIMENT_CONFIGS = select_cluster(CLUSTER_TO_RUN)

    print("\n" + "=" * 100)
    print("Running cluster:", CLUSTER_TO_RUN)
    print("=" * 100)

    if competition_ids is None:
        competition_ids = resolve_competition_ids()

    if num_runs is None:
        num_runs = NUM_RUNS_PER_COMBO

    question_level_results_df, run_summary_df = run_live_experiment_grid(
        client=client,
        experiment_configs=EXPERIMENT_CONFIGS,
        competition_ids=competition_ids,
        num_runs=num_runs,
        verbose=verbose,
        sleep_between_runs=sleep_between_runs,
        sort_for_speed=sort_for_speed,
    )

    # Add cluster name immediately
    if run_summary_df is not None and not run_summary_df.empty:
        run_summary_df = run_summary_df.copy()
        run_summary_df["cluster_name"] = CLUSTER_TO_RUN

    if question_level_results_df is not None and not question_level_results_df.empty:
        question_level_results_df = question_level_results_df.copy()
        question_level_results_df["cluster_name"] = CLUSTER_TO_RUN

    return question_level_results_df, run_summary_df


# ------------------------------------------------------------
# 5. Summarize latest cluster run
# ------------------------------------------------------------

def summarize_cluster_run(save_results=True):
    global per_competition_summary_df
    global overall_summary_df
    global run_summary_df
    global question_level_results_df

    if "CLUSTER_TO_RUN" not in globals():
        print("No cluster selected yet.")
        return pd.DataFrame(), pd.DataFrame()

    if run_summary_df is None or run_summary_df.empty:
        print("No results returned for cluster:", CLUSTER_TO_RUN)
        per_competition_summary_df = pd.DataFrame()
        overall_summary_df = pd.DataFrame()
        return per_competition_summary_df, overall_summary_df

    # Ensure cluster label exists
    run_summary_df = run_summary_df.copy()
    if "cluster_name" not in run_summary_df.columns:
        run_summary_df["cluster_name"] = CLUSTER_TO_RUN

    if question_level_results_df is not None and not question_level_results_df.empty:
        question_level_results_df = question_level_results_df.copy()
        if "cluster_name" not in question_level_results_df.columns:
            question_level_results_df["cluster_name"] = CLUSTER_TO_RUN

    # Ensure run_error exists so aggregation does not fail
    if "run_error" not in run_summary_df.columns:
        run_summary_df["run_error"] = None

    per_competition_summary_df = aggregate_per_competition(run_summary_df)
    overall_summary_df = aggregate_overall(run_summary_df)

    print("\nCluster:", CLUSTER_TO_RUN)

    print("\nRun-level summary:")
    display(run_summary_df)

    print("\nPer-competition summary:")
    display(per_competition_summary_df)

    print("\nOverall summary:")
    display(overall_summary_df)

    if save_results:
        save_cluster_results(
            cluster_name=CLUSTER_TO_RUN,
            run_summary_df=run_summary_df,
            question_level_results_df=question_level_results_df,
            per_competition_summary_df=per_competition_summary_df,
            overall_summary_df=overall_summary_df,
        )

    return per_competition_summary_df, overall_summary_df

### 01_llm_only_baselines

In [ ]:
run_cluster("01_llm_only_baselines")

Selected cluster: 01_llm_only_baselines
Configs found: 3


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,flan_llm_only_strong,flan_t5_base,strong,none,none,False,2,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
2,qwen3b_llm_only_strong,qwen2_5_3b,strong,none,none,False,2,none,0,bge_small,False,False,mini_cross_encoder,0.0,32



Running cluster: 01_llm_only_baselines
Total runs: 54
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.42 GB

Preloading local model for GPU reuse: qwen2_5_1_5b
Loading local model: qwen2_5_1_5b Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: qwen2_5_1_5b
GPU available: Tesla T4
CUDA memory allocated: 3.418 GB
CUDA memory reserved: 3.469 GB

[1/54] qwen1_5b_llm_only_strong | competition=0 | run=1
Finished in 0.96s | level=2 | correct=1 | earned=100.0 | error=False

[2/54] qwen1_5b_llm_only_strong | competition=0 | run=2
Finished in 0.42s | level=1 | correct=0 | earned=0.0 | error=False

[3/54] qwen1_5b_llm_only_strong | competition=0 | run=3
Finished in 1.17s | level=3 | correct=2 | earned=200.0 | error=False

[4/54] qwen1_5b_llm_only_strong | competition=1 | run=1
Finished in 0.44s | level=1 | correct=0 | earned=0.0 | error=False

[5/54] qwen1_5b_llm_only_strong | competition=1 | run=2
Finished in 0.45s | level=1 | correct=0 | earned=0.0 | error=False

[6/54] qwen1_5b_llm_only_strong | competition=1 | run=3
Finished in 0.43s | level=1 | correct=0 | earned=0.0 | error=False

[7/54] qwen1_5b_llm_only_strong | competition=2 | run=1
Finished in 5.10s | level=13 | correct=12 | earned=128000.0 | error=False

[8/54] qwen1

...

Loaded: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.462 GB
CUDA memory reserved: 2.615 GB

[19/54] qwen3b_llm_only_strong | competition=0 | run=1
Finished in 0.74s | level=1 | correct=0 | earned=0.0 | error=False

[20/54] qwen3b_llm_only_strong | competition=0 | run=2
Finished in 1.16s | level=2 | correct=1 | earned=100.0 | error=False

[21/54] qwen3b_llm_only_strong | competition=0 | run=3
Finished in 4.42s | level=8 | correct=7 | earned=4000.0 | error=False

[22/54] qwen3b_llm_only_strong | competition=1 | run=1
Finished in 5.11s | level=10 | correct=9 | earned=16000.0 | error=False

[23/54] qwen3b_llm_only_strong | competition=1 | run=2
Finished in 1.08s | level=2 | correct=1 | earned=100.0 | error=False

[24/54] qwen3b_llm_only_strong | competition=1 | run=3
Finished in 0.56s | level=1 | correct=0 | earned=0.0 | error=False

[25/54] qwen3b_llm_only_strong | competition=2 | run=1
Finished in 1.10s | level=2 | correct=1 | earned=100.0 | error=False

[26/54] qwen3b_llm

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded: flan_t5_base
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.41 GB

[37/54] flan_llm_only_strong | competition=0 | run=1
Finished in 0.55s | level=1 | correct=0 | earned=0.0 | error=False

[38/54] flan_llm_only_strong | competition=0 | run=2
Finished in 0.39s | level=1 | correct=0 | earned=0.0 | error=False

[39/54] flan_llm_only_strong | competition=0 | run=3
Finished in 0.39s | level=1 | correct=0 | earned=0.0 | error=False

[40/54] flan_llm_only_strong | competition=1 | run=1
Finished in 0.40s | level=1 | correct=0 | earned=0.0 | error=False

[41/54] flan_llm_only_strong | competition=1 | run=2
Finished in 0.39s | level=1 | correct=0 | earned=0.0 | error=False

[42/54] flan_llm_only_strong | competition=1 | run=3
Finished in 1.06s | level=3 | correct=2 | earned=200.0 | error=False

[43/54] flan_llm_only_strong | competition=2 | run=1
Finished in 0.41s | level=1 | correct=0 | earned=0.0 | error=False

[44/54] flan_llm_only_strong | competition=

(                   combo_name       llm_key prompt_style retrieval_tools  \
 0    qwen1_5b_llm_only_strong  qwen2_5_1_5b       strong            none   
 1    qwen1_5b_llm_only_strong  qwen2_5_1_5b       strong            none   
 2    qwen1_5b_llm_only_strong  qwen2_5_1_5b       strong            none   
 3    qwen1_5b_llm_only_strong  qwen2_5_1_5b       strong            none   
 4    qwen1_5b_llm_only_strong  qwen2_5_1_5b       strong            none   
 ..                        ...           ...          ...             ...   
 149      flan_llm_only_strong  flan_t5_base       strong            none   
 150      flan_llm_only_strong  flan_t5_base       strong            none   
 151      flan_llm_only_strong  flan_t5_base       strong            none   
 152      flan_llm_only_strong  flan_t5_base       strong            none   
 153      flan_llm_only_strong  flan_t5_base       strong            none   
 
     query_strategies  fetch_pages  max_results_per_tool filter_method  to

In [ ]:
summarize_cluster_run()


Cluster: 01_llm_only_baselines

Run-level summary:


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,cluster_name,run_error
0,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Entertainment,1,2,1,0,100.0,2,0.103000,01_llm_only_baselines,None
1,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Entertainment,2,1,0,0,0.0,1,0.089000,01_llm_only_baselines,None
2,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Entertainment,3,3,2,0,200.0,3,0.099000,01_llm_only_baselines,None
3,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Ancient History and Politics,1,1,0,0,0.0,1,0.104000,01_llm_only_baselines,None
4,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Ancient History and Politics,2,1,0,0,0.0,1,0.107000,01_llm_only_baselines,None
5,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Ancient History and Politics,3,1,0,0,0.0,1,0.099000,01_llm_only_baselines,None
6,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Science and Nature,1,13,12,0,128000.0,13,0.115538,01_llm_only_baselines,None
7,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Science and Nature,2,3,2,0,200.0,3,0.100333,01_llm_only_baselines,None
8,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Science and Nature,3,11,10,0,32000.0,11,0.114909,01_llm_only_baselines,None
9,qwen1_5b_llm_only_strong,qwen2_5_1_5b,strong,none,none,False,2,none,0,bge_small,...,Maths,1,2,1,0,100.0,2,0.090000,01_llm_only_baselines,None



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
16,01_llm_only_baselines,qwen3b_llm_only_strong,qwen2_5_3b,none,none,none,0,False,False,4,Philosophy and Psychology,3,8.000,6.245,7.333,341733.333,590860.401,0.282,0,0
8,01_llm_only_baselines,qwen1_5b_llm_only_strong,qwen2_5_1_5b,none,none,none,0,False,False,2,Science and Nature,3,9.000,5.292,8.000,53400.000,66533.300,0.110,0,0
13,01_llm_only_baselines,qwen3b_llm_only_strong,qwen2_5_3b,none,none,none,0,False,False,1,Ancient History and Politics,3,4.333,4.933,3.333,5366.667,9208.873,0.233,0,0
10,01_llm_only_baselines,qwen1_5b_llm_only_strong,qwen2_5_1_5b,none,none,none,0,False,False,4,Philosophy and Psychology,3,4.000,3.606,3.000,1400.000,2253.886,0.111,0,0
12,01_llm_only_baselines,qwen3b_llm_only_strong,qwen2_5_3b,none,none,none,0,False,False,0,Entertainment,3,3.667,3.786,2.667,1366.667,2281.082,0.268,0,0
14,01_llm_only_baselines,qwen3b_llm_only_strong,qwen2_5_3b,none,none,none,0,False,False,2,Science and Nature,3,3.667,2.887,2.667,733.333,1096.966,0.270,0,0
9,01_llm_only_baselines,qwen1_5b_llm_only_strong,qwen2_5_1_5b,none,none,none,0,False,False,3,Maths,3,3.333,2.309,2.333,400.000,519.615,0.099,0,0
6,01_llm_only_baselines,qwen1_5b_llm_only_strong,qwen2_5_1_5b,none,none,none,0,False,False,0,Entertainment,3,2.000,1.000,1.000,100.000,100.000,0.097,0,0
1,01_llm_only_baselines,flan_llm_only_strong,flan_t5_base,none,none,none,0,False,False,1,Ancient History and Politics,3,1.667,1.155,0.667,66.667,115.470,0.061,0,0
4,01_llm_only_baselines,flan_llm_only_strong,flan_t5_base,none,none,none,0,False,False,4,Philosophy and Psychology,3,1.667,1.155,0.667,66.667,115.470,0.077,0,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
2,01_llm_only_baselines,qwen3b_llm_only_strong,qwen2_5_3b,none,none,none,0,False,False,18,6,3.778,3.919,2.833,58216.667,241057.460,0.254,0,0
1,01_llm_only_baselines,qwen1_5b_llm_only_strong,qwen2_5_1_5b,none,none,none,0,False,False,18,6,3.500,3.634,2.500,9227.778,30575.323,0.103,0,0
0,01_llm_only_baselines,flan_llm_only_strong,flan_t5_base,none,none,none,0,False,False,18,6,1.278,0.669,0.278,27.778,66.911,0.070,0,0


Saved: experiment_outputs/01_llm_only_baselines_run_summary.csv
Saved: experiment_outputs/01_llm_only_baselines_question_results.csv
Saved: experiment_outputs/01_llm_only_baselines_per_competition_summary.csv
Saved: experiment_outputs/01_llm_only_baselines_overall_summary.csv


(             cluster_name                combo_name       llm_key  \
 16  01_llm_only_baselines    qwen3b_llm_only_strong    qwen2_5_3b   
 8   01_llm_only_baselines  qwen1_5b_llm_only_strong  qwen2_5_1_5b   
 13  01_llm_only_baselines    qwen3b_llm_only_strong    qwen2_5_3b   
 10  01_llm_only_baselines  qwen1_5b_llm_only_strong  qwen2_5_1_5b   
 12  01_llm_only_baselines    qwen3b_llm_only_strong    qwen2_5_3b   
 14  01_llm_only_baselines    qwen3b_llm_only_strong    qwen2_5_3b   
 9   01_llm_only_baselines  qwen1_5b_llm_only_strong  qwen2_5_1_5b   
 6   01_llm_only_baselines  qwen1_5b_llm_only_strong  qwen2_5_1_5b   
 1   01_llm_only_baselines      flan_llm_only_strong  flan_t5_base   
 4   01_llm_only_baselines      flan_llm_only_strong  flan_t5_base   
 11  01_llm_only_baselines  qwen1_5b_llm_only_strong  qwen2_5_1_5b   
 17  01_llm_only_baselines    qwen3b_llm_only_strong    qwen2_5_3b   
 5   01_llm_only_baselines      flan_llm_only_strong  flan_t5_base   
 15  01_llm_only_bas

### 02_wikipedia_only

In [ ]:
run_cluster("02_wikipedia_only")

Selected cluster: 02_wikipedia_only
Configs found: 3


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,flan_wiki_bm25_k3_raw_options,flan_t5_base,strong,wikipedia,raw_question+question_plus_options,False,2,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
2,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,strong,wikipedia,entity_focused+domain_prefixed,False,2,bm25,5,bge_small,False,False,mini_cross_encoder,0.0,32



Running cluster: 02_wikipedia_only
Total runs: 54
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.424 GB

Preloading local model for GPU reuse: qwen2_5_1_5b
Loading local model: qwen2_5_1_5b Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: qwen2_5_1_5b
GPU available: Tesla T4
CUDA memory allocated: 3.418 GB
CUDA memory reserved: 3.469 GB

[1/54] qwen1_5b_wiki_bm25_k3_entity_raw | competition=0 | run=1
Finished in 5.33s | level=2 | correct=1 | earned=100.0 | error=False

[2/54] qwen1_5b_wiki_bm25_k3_entity_raw | competition=0 | run=2
Wikipedia API summary | title=Katy Perry failed on attempt 1. Retrying in 0.47s. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/Katy%20Perry')
Wikipedia API summary | title=Katy Perry failed after 2 attempts. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/Katy%20Perry')
Wikipedia API summary | title=The Doors failed on attempt 1. Retrying in 0.46s. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/The%20Doors')
Wikipedia API summary | title=The Doors failed after 2 attempts. Error: HTTPError('

...

Loaded: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.462 GB
CUDA memory reserved: 2.615 GB

[19/54] qwen3b_wiki_bm25_k5_entity_domain | competition=0 | run=1
Finished in 2.98s | level=1 | correct=0 | earned=0.0 | error=False

[20/54] qwen3b_wiki_bm25_k5_entity_domain | competition=0 | run=2
Finished in 6.18s | level=3 | correct=2 | earned=200.0 | error=False

[21/54] qwen3b_wiki_bm25_k5_entity_domain | competition=0 | run=3
Wikipedia API summary | title=Pulp Fiction failed on attempt 1. Retrying in 0.44s. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/Pulp%20Fiction')
Wikipedia API summary | title=Pulp Fiction failed after 2 attempts. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/Pulp%20Fiction')
Wikipedia API summary | title=Dreyfus affair failed on attempt 1. Retrying in 0.42s. Error: HTTPError('429 Client Error: Too Many Requests for url

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded: flan_t5_base
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.41 GB

[37/54] flan_wiki_bm25_k3_raw_options | competition=0 | run=1
Finished in 2.77s | level=1 | correct=0 | earned=0.0 | error=False

[38/54] flan_wiki_bm25_k3_raw_options | competition=0 | run=2
Wikipedia API summary | title=Rhythm and blues failed on attempt 1. Retrying in 0.52s. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/Rhythm%20and%20blues')
Wikipedia API summary | title=Rhythm and blues failed after 2 attempts. Error: HTTPError('429 Client Error: Too Many Requests for url: https://en.wikipedia.org/api/rest_v1/page/summary/Rhythm%20and%20blues')
Wikipedia API search | query=What genre did Ray Charles pioneer in the 1950s by combining elements of blues jazz rhythm and blues and gospel Hip Hop Soul Country Rock and Roll failed on attempt 1. Retrying in 0.52s. Error: HTTPError('429 Client Error: Too Many Requests

(                           combo_name       llm_key prompt_style  \
 0    qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 1    qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 2    qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 3    qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 4    qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 ..                                ...           ...          ...   
 177     flan_wiki_bm25_k3_raw_options  flan_t5_base       strong   
 178     flan_wiki_bm25_k3_raw_options  flan_t5_base       strong   
 179     flan_wiki_bm25_k3_raw_options  flan_t5_base       strong   
 180     flan_wiki_bm25_k3_raw_options  flan_t5_base       strong   
 181     flan_wiki_bm25_k3_raw_options  flan_t5_base       strong   
 
     retrieval_tools                    query_strategies  fetch_pages  \
 0         wikipedia         entity_focused+raw_question        False   
 1         wikipedia    

In [ ]:
summarize_cluster_run()


Cluster: 02_wikipedia_only

Run-level summary:


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,cluster_name,run_error
0,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Entertainment,1,2,1,0,100.0,2,2.272500,02_wikipedia_only,None
1,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Entertainment,2,9,8,0,8000.0,9,1.733222,02_wikipedia_only,None
2,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Entertainment,3,3,2,0,200.0,3,1.511333,02_wikipedia_only,None
3,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Ancient History and Politics,1,2,1,0,100.0,2,1.507500,02_wikipedia_only,None
4,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Ancient History and Politics,2,2,1,0,100.0,2,2.192000,02_wikipedia_only,None
5,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Ancient History and Politics,3,3,2,0,200.0,3,2.424000,02_wikipedia_only,None
6,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Science and Nature,1,6,5,0,1000.0,6,1.895000,02_wikipedia_only,None
7,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Science and Nature,2,2,1,0,100.0,2,1.541000,02_wikipedia_only,None
8,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Science and Nature,3,3,2,0,200.0,3,1.499000,02_wikipedia_only,None
9,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,strong,wikipedia,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Maths,1,5,4,0,500.0,5,1.450200,02_wikipedia_only,None



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
16,02_wikipedia_only,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,wikipedia,entity_focused+domain_prefixed,bm25,5,False,False,4,Philosophy and Psychology,3,6.667,7.371,6.000,341433.333,591120.092,2.040,0,0
10,02_wikipedia_only,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,wikipedia,entity_focused+raw_question,bm25,3,False,False,4,Philosophy and Psychology,3,7.333,6.658,6.333,170833.333,295459.004,1.652,0,0
13,02_wikipedia_only,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,wikipedia,entity_focused+domain_prefixed,bm25,5,False,False,1,Ancient History and Politics,3,5.667,4.726,4.667,10800.000,18360.011,1.817,0,0
11,02_wikipedia_only,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,wikipedia,entity_focused+raw_question,bm25,3,False,False,5,News,3,4.333,5.774,3.333,10666.667,18475.209,1.440,0,0
6,02_wikipedia_only,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,wikipedia,entity_focused+raw_question,bm25,3,False,False,0,Entertainment,3,4.667,3.786,3.667,2766.667,4532.475,1.839,0,0
14,02_wikipedia_only,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,wikipedia,entity_focused+domain_prefixed,bm25,5,False,False,2,Science and Nature,3,4.667,2.517,3.667,866.667,1001.665,1.668,0,0
8,02_wikipedia_only,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,wikipedia,entity_focused+raw_question,bm25,3,False,False,2,Science and Nature,3,3.667,2.082,2.667,433.333,493.288,1.645,0,0
4,02_wikipedia_only,flan_wiki_bm25_k3_raw_options,flan_t5_base,wikipedia,raw_question+question_plus_options,bm25,3,False,False,4,Philosophy and Psychology,3,3.333,2.517,2.333,400.000,529.150,2.939,0,0
12,02_wikipedia_only,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,wikipedia,entity_focused+domain_prefixed,bm25,5,False,False,0,Entertainment,3,3.000,2.000,2.000,233.333,251.661,2.054,0,0
0,02_wikipedia_only,flan_wiki_bm25_k3_raw_options,flan_t5_base,wikipedia,raw_question+question_plus_options,bm25,3,False,False,0,Entertainment,3,2.333,2.309,1.333,166.667,288.675,1.707,0,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
2,02_wikipedia_only,qwen3b_wiki_bm25_k5_entity_domain,qwen2_5_3b,wikipedia,entity_focused+domain_prefixed,bm25,5,False,False,18,6,3.833,3.808,2.889,58905.556,240971.955,1.989,0,0
1,02_wikipedia_only,qwen1_5b_wiki_bm25_k3_entity_raw,qwen2_5_1_5b,wikipedia,entity_focused+raw_question,bm25,3,False,False,18,6,4.111,3.879,3.111,30833.333,120323.349,1.689,0,0
0,02_wikipedia_only,flan_wiki_bm25_k3_raw_options,flan_t5_base,wikipedia,raw_question+question_plus_options,bm25,3,False,False,18,6,2.167,1.543,1.167,150.000,252.633,1.725,0,0


Saved: experiment_outputs/02_wikipedia_only_run_summary.csv
Saved: experiment_outputs/02_wikipedia_only_question_results.csv
Saved: experiment_outputs/02_wikipedia_only_per_competition_summary.csv
Saved: experiment_outputs/02_wikipedia_only_overall_summary.csv


(         cluster_name                         combo_name       llm_key  \
 16  02_wikipedia_only  qwen3b_wiki_bm25_k5_entity_domain    qwen2_5_3b   
 10  02_wikipedia_only   qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b   
 13  02_wikipedia_only  qwen3b_wiki_bm25_k5_entity_domain    qwen2_5_3b   
 11  02_wikipedia_only   qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b   
 6   02_wikipedia_only   qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b   
 14  02_wikipedia_only  qwen3b_wiki_bm25_k5_entity_domain    qwen2_5_3b   
 8   02_wikipedia_only   qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b   
 4   02_wikipedia_only      flan_wiki_bm25_k3_raw_options  flan_t5_base   
 12  02_wikipedia_only  qwen3b_wiki_bm25_k5_entity_domain    qwen2_5_3b   
 0   02_wikipedia_only      flan_wiki_bm25_k3_raw_options  flan_t5_base   
 9   02_wikipedia_only   qwen1_5b_wiki_bm25_k3_entity_raw  qwen2_5_1_5b   
 5   02_wikipedia_only      flan_wiki_bm25_k3_raw_options  flan_t5_base   
 7   02_wikipedia_only   

### 03_duckduckgo_model_comparison

In [ ]:
run_cluster("03_duckduckgo_model_comparison")

Selected cluster: 03_duckduckgo_model_comparison
Configs found: 3


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,flan_ddg_bm25_k3_entity_raw,flan_t5_base,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
2,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,strong,duckduckgo,entity_focused+question_plus_options,False,2,bm25,5,bge_small,False,False,mini_cross_encoder,0.0,32



Running cluster: 03_duckduckgo_model_comparison
Total runs: 54
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.434 GB

Preloading local model for GPU reuse: qwen2_5_1_5b
Loading local model: qwen2_5_1_5b Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: qwen2_5_1_5b
GPU available: Tesla T4
CUDA memory allocated: 3.418 GB
CUDA memory reserved: 3.469 GB

[1/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=0 | run=1
Finished in 23.73s | level=3 | correct=2 | earned=200.0 | error=False

[2/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=0 | run=2
Finished in 11.69s | level=2 | correct=1 | earned=100.0 | error=False

[3/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=0 | run=3
Finished in 36.82s | level=6 | correct=5 | earned=1000.0 | error=False

[4/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=1 | run=1
Finished in 32.26s | level=4 | correct=3 | earned=300.0 | error=False

[5/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=1 | run=2
Finished in 50.20s | level=7 | correct=6 | earned=2000.0 | error=False

[6/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=1 | run=3
Finished in 116.85s | level=14 | correct=13 | earned=256000.0 | error=False

[7/54] qwen1_5b_ddg_bm25_k3_entity_raw | competition=2 | run=1
Finished in 52.0

...

Loaded: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.463 GB
CUDA memory reserved: 2.604 GB

[19/54] qwen3b_ddg_bm25_k5_entity_options | competition=0 | run=1
Finished in 20.19s | level=3 | correct=2 | earned=200.0 | error=False

[20/54] qwen3b_ddg_bm25_k5_entity_options | competition=0 | run=2
Finished in 58.30s | level=7 | correct=6 | earned=2000.0 | error=False

[21/54] qwen3b_ddg_bm25_k5_entity_options | competition=0 | run=3
Finished in 22.44s | level=3 | correct=2 | earned=200.0 | error=False

[22/54] qwen3b_ddg_bm25_k5_entity_options | competition=1 | run=1
Finished in 19.25s | level=2 | correct=1 | earned=100.0 | error=False

[23/54] qwen3b_ddg_bm25_k5_entity_options | competition=1 | run=2
Finished in 23.03s | level=3 | correct=2 | earned=200.0 | error=False

[24/54] qwen3b_ddg_bm25_k5_entity_options | competition=1 | run=3
Finished in 59.11s | level=9 | correct=8 | earned=8000.0 | error=False

[25/54] qwen3b_ddg_bm25_k5_entity_options | competition=2 | run=1
Fin

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded: flan_t5_base
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.41 GB

[37/54] flan_ddg_bm25_k3_entity_raw | competition=0 | run=1
Finished in 27.57s | level=3 | correct=2 | earned=200.0 | error=False

[38/54] flan_ddg_bm25_k3_entity_raw | competition=0 | run=2
Finished in 10.99s | level=1 | correct=0 | earned=0.0 | error=False

[39/54] flan_ddg_bm25_k3_entity_raw | competition=0 | run=3
Finished in 2.73s | level=1 | correct=0 | earned=0.0 | error=False

[40/54] flan_ddg_bm25_k3_entity_raw | competition=1 | run=1
Finished in 32.03s | level=3 | correct=2 | earned=200.0 | error=False

[41/54] flan_ddg_bm25_k3_entity_raw | competition=1 | run=2
Finished in 5.27s | level=1 | correct=0 | earned=0.0 | error=False

[42/54] flan_ddg_bm25_k3_entity_raw | competition=1 | run=3
Finished in 10.01s | level=1 | correct=0 | earned=0.0 | error=False

[43/54] flan_ddg_bm25_k3_entity_raw | competition=2 | run=1
Finished in 11.60s | level=1 | correct=0 | earned=0.0 | 

(                          combo_name       llm_key prompt_style  \
 0    qwen1_5b_ddg_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 1    qwen1_5b_ddg_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 2    qwen1_5b_ddg_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 3    qwen1_5b_ddg_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 4    qwen1_5b_ddg_bm25_k3_entity_raw  qwen2_5_1_5b       strong   
 ..                               ...           ...          ...   
 207      flan_ddg_bm25_k3_entity_raw  flan_t5_base       strong   
 208      flan_ddg_bm25_k3_entity_raw  flan_t5_base       strong   
 209      flan_ddg_bm25_k3_entity_raw  flan_t5_base       strong   
 210      flan_ddg_bm25_k3_entity_raw  flan_t5_base       strong   
 211      flan_ddg_bm25_k3_entity_raw  flan_t5_base       strong   
 
     retrieval_tools             query_strategies  fetch_pages  \
 0        duckduckgo  entity_focused+raw_question        False   
 1        duckduckgo  entity_focused+raw_question 

In [ ]:
summarize_cluster_run()


Cluster: 03_duckduckgo_model_comparison

Run-level summary:


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,run_error,cluster_name
0,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Entertainment,1,3,2,0,200.0,3,7.459333,NaN,03_duckduckgo_model_comparison
1,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Entertainment,2,2,1,0,100.0,2,5.387500,NaN,03_duckduckgo_model_comparison
2,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Entertainment,3,6,5,0,1000.0,6,5.803667,NaN,03_duckduckgo_model_comparison
3,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Ancient History and Politics,1,4,3,0,300.0,4,7.702250,NaN,03_duckduckgo_model_comparison
4,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Ancient History and Politics,2,7,6,0,2000.0,7,6.826429,NaN,03_duckduckgo_model_comparison
5,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Ancient History and Politics,3,14,13,0,256000.0,14,7.973000,NaN,03_duckduckgo_model_comparison
6,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Science and Nature,1,8,7,0,4000.0,8,6.142625,NaN,03_duckduckgo_model_comparison
7,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Science and Nature,2,8,7,0,4000.0,8,5.254625,NaN,03_duckduckgo_model_comparison
8,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Science and Nature,3,0,0,0,0.0,0,NaN,"e RemoteDisconnected(""Remote end closed connec...",03_duckduckgo_model_comparison
9,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,strong,duckduckgo,entity_focused+raw_question,False,2,bm25,3,bge_small,...,Maths,1,1,0,0,0.0,1,2.985000,NaN,03_duckduckgo_model_comparison



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
10,03_duckduckgo_model_comparison,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,4,Philosophy and Psychology,3,11.333,4.041,10.667,363333.333,572993.310,8.431,0,0
16,03_duckduckgo_model_comparison,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,duckduckgo,entity_focused+question_plus_options,bm25,5,False,False,4,Philosophy and Psychology,3,9.333,7.371,8.667,362666.667,573624.732,7.765,0,0
7,03_duckduckgo_model_comparison,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,1,Ancient History and Politics,3,8.333,5.132,7.333,86100.000,147140.171,7.501,0,0
13,03_duckduckgo_model_comparison,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,duckduckgo,entity_focused+question_plus_options,bm25,5,False,False,1,Ancient History and Politics,3,4.667,3.786,3.667,2766.667,4532.475,7.588,0,0
8,03_duckduckgo_model_comparison,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,2,Science and Nature,3,5.333,4.619,4.667,2666.667,2309.401,5.699,0,1
4,03_duckduckgo_model_comparison,flan_ddg_bm25_k3_entity_raw,flan_t5_base,duckduckgo,entity_focused+raw_question,bm25,3,False,False,4,Philosophy and Psychology,3,3.333,4.041,2.333,1333.333,2309.401,10.239,0,0
12,03_duckduckgo_model_comparison,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,duckduckgo,entity_focused+question_plus_options,bm25,5,False,False,0,Entertainment,3,4.333,2.309,3.333,800.000,1039.230,7.126,0,0
17,03_duckduckgo_model_comparison,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,duckduckgo,entity_focused+question_plus_options,bm25,5,False,False,5,News,3,3.000,3.464,2.000,666.667,1154.701,7.249,0,0
14,03_duckduckgo_model_comparison,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,duckduckgo,entity_focused+question_plus_options,bm25,5,False,False,2,Science and Nature,3,4.000,1.732,3.000,466.667,461.880,7.469,0,0
6,03_duckduckgo_model_comparison,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,0,Entertainment,3,3.667,2.082,2.667,433.333,493.288,6.217,0,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
1,03_duckduckgo_model_comparison,qwen1_5b_ddg_bm25_k3_entity_raw,qwen2_5_1_5b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,18,6,5.556,4.540,4.667,75494.444,244427.394,6.369,0,1
2,03_duckduckgo_model_comparison,qwen3b_ddg_bm25_k5_entity_options,qwen2_5_3b,duckduckgo,entity_focused+question_plus_options,bm25,5,False,False,18,6,4.389,4.146,3.444,61227.778,240742.244,7.567,0,0
0,03_duckduckgo_model_comparison,flan_ddg_bm25_k3_entity_raw,flan_t5_base,duckduckgo,entity_focused+raw_question,bm25,3,False,False,18,6,1.833,1.689,0.833,266.667,934.313,7.540,0,0


Saved: experiment_outputs/03_duckduckgo_model_comparison_run_summary.csv
Saved: experiment_outputs/03_duckduckgo_model_comparison_question_results.csv
Saved: experiment_outputs/03_duckduckgo_model_comparison_per_competition_summary.csv
Saved: experiment_outputs/03_duckduckgo_model_comparison_overall_summary.csv


(                      cluster_name                         combo_name  \
 10  03_duckduckgo_model_comparison    qwen1_5b_ddg_bm25_k3_entity_raw   
 16  03_duckduckgo_model_comparison  qwen3b_ddg_bm25_k5_entity_options   
 7   03_duckduckgo_model_comparison    qwen1_5b_ddg_bm25_k3_entity_raw   
 13  03_duckduckgo_model_comparison  qwen3b_ddg_bm25_k5_entity_options   
 8   03_duckduckgo_model_comparison    qwen1_5b_ddg_bm25_k3_entity_raw   
 4   03_duckduckgo_model_comparison        flan_ddg_bm25_k3_entity_raw   
 12  03_duckduckgo_model_comparison  qwen3b_ddg_bm25_k5_entity_options   
 17  03_duckduckgo_model_comparison  qwen3b_ddg_bm25_k5_entity_options   
 14  03_duckduckgo_model_comparison  qwen3b_ddg_bm25_k5_entity_options   
 6   03_duckduckgo_model_comparison    qwen1_5b_ddg_bm25_k3_entity_raw   
 11  03_duckduckgo_model_comparison    qwen1_5b_ddg_bm25_k3_entity_raw   
 0   03_duckduckgo_model_comparison        flan_ddg_bm25_k3_entity_raw   
 1   03_duckduckgo_model_comparison   

### 04_query_strategy_ablation

In [ ]:
run_cluster("04_query_strategy_ablation")

Selected cluster: 04_query_strategy_ablation
Configs found: 7


,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,qwen3b_ddg_raw_only_k3,qwen2_5_3b,strong,text,duckduckgo,raw_question,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen3b_ddg_options_only_k3,qwen2_5_3b,strong,text,duckduckgo,question_plus_options,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
2,qwen3b_ddg_keyword_only_k3,qwen2_5_3b,strong,text,duckduckgo,keyword_condensed,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
3,qwen3b_ddg_entity_only_k3,qwen2_5_3b,strong,text,duckduckgo,entity_focused,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
4,qwen3b_ddg_domain_only_k3,qwen2_5_3b,strong,text,duckduckgo,domain_prefixed,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
5,qwen3b_wiki_entity_only_k3,qwen2_5_3b,strong,text,wikipedia,entity_focused,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
6,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,strong,text,wikipedia,keyword_condensed,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32



Running cluster: 04_query_strategy_ablation
Total runs: 126
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 1.928 GB
CUDA memory reserved: 2.111 GB

Preloading local model for GPU reuse: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 1.928 GB
CUDA memory reserved: 2.111 GB

[1/126] qwen3b_ddg_domain_only_k3 | competition=0 | run=1
Finished in 5.70s | level=2 | correct=1 | earned=100.0 | error=False

[2/126] qwen3b_ddg_domain_only_k3 | competition=0 | run=2
Finished in 4.71s | level=2 | correct=1 | earned=100.0 | error=False

[3/126] qwen3b_ddg_domain_only_k3 | competition=0 | run=3
Finished in 3.87s | level=1 | correct=0 | earned=0.0 | error=False

[4/126] qwen3b_ddg_domain_only_k3 | competition=1 | run=1
Finished in 28.34s | level=9 | correct=8 | earned=8000.0 | error=False

[5/126] qwen3b_ddg_domain_only_k3 | competition=1 | run=2
Finished in 2.64s | level=1 | correct=0 | earned=0.0 | error=False

[6/126] qwen3b_ddg_domain_only_k3 | competi

(                      combo_name     llm_key prompt_style game_mode  \
 0      qwen3b_ddg_domain_only_k3  qwen2_5_3b       strong      text   
 1      qwen3b_ddg_domain_only_k3  qwen2_5_3b       strong      text   
 2      qwen3b_ddg_domain_only_k3  qwen2_5_3b       strong      text   
 3      qwen3b_ddg_domain_only_k3  qwen2_5_3b       strong      text   
 4      qwen3b_ddg_domain_only_k3  qwen2_5_3b       strong      text   
 ..                           ...         ...          ...       ...   
 520  qwen3b_wiki_keyword_only_k3  qwen2_5_3b       strong      text   
 521  qwen3b_wiki_keyword_only_k3  qwen2_5_3b       strong      text   
 522  qwen3b_wiki_keyword_only_k3  qwen2_5_3b       strong      text   
 523  qwen3b_wiki_keyword_only_k3  qwen2_5_3b       strong      text   
 524  qwen3b_wiki_keyword_only_k3  qwen2_5_3b       strong      text   
 
     retrieval_tools   query_strategies  fetch_pages  max_results_per_tool  \
 0        duckduckgo    domain_prefixed        False    

In [ ]:
summarize_cluster_run()


Cluster: 04_query_strategy_ablation

Run-level summary:


,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,cluster_name,run_error
0,qwen3b_ddg_domain_only_k3,qwen2_5_3b,strong,text,duckduckgo,domain_prefixed,False,5,bm25,3,...,Entertainment,1,2,1,0,100.0,2,2.473500,04_query_strategy_ablation,None
1,qwen3b_ddg_domain_only_k3,qwen2_5_3b,strong,text,duckduckgo,domain_prefixed,False,5,bm25,3,...,Entertainment,2,2,1,0,100.0,2,2.055500,04_query_strategy_ablation,None
2,qwen3b_ddg_domain_only_k3,qwen2_5_3b,strong,text,duckduckgo,domain_prefixed,False,5,bm25,3,...,Entertainment,3,1,0,0,0.0,1,3.539000,04_query_strategy_ablation,None
3,qwen3b_ddg_domain_only_k3,qwen2_5_3b,strong,text,duckduckgo,domain_prefixed,False,5,bm25,3,...,Ancient History and Politics,1,9,8,0,8000.0,9,2.872222,04_query_strategy_ablation,None
4,qwen3b_ddg_domain_only_k3,qwen2_5_3b,strong,text,duckduckgo,domain_prefixed,False,5,bm25,3,...,Ancient History and Politics,2,1,0,0,0.0,1,2.316000,04_query_strategy_ablation,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,strong,text,wikipedia,keyword_condensed,False,5,bm25,3,...,Philosophy and Psychology,2,15,15,0,1024000.0,15,0.954200,04_query_strategy_ablation,None
122,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,strong,text,wikipedia,keyword_condensed,False,5,bm25,3,...,Philosophy and Psychology,3,2,1,0,100.0,2,0.876500,04_query_strategy_ablation,None
123,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,strong,text,wikipedia,keyword_condensed,False,5,bm25,3,...,News,1,3,2,0,200.0,3,0.963333,04_query_strategy_ablation,None
124,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,strong,text,wikipedia,keyword_condensed,False,5,bm25,3,...,News,2,2,1,0,100.0,2,0.976000,04_query_strategy_ablation,None



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
16,04_query_strategy_ablation,qwen3b_ddg_keyword_only_k3,qwen2_5_3b,duckduckgo,keyword_condensed,bm25,3,False,False,4,Philosophy and Psychology,3,12.667,4.041,12.333,684000.000,588897.275,9.325,0,0
40,04_query_strategy_ablation,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,wikipedia,keyword_condensed,bm25,3,False,False,4,Philosophy and Psychology,3,10.667,7.506,10.333,682700.000,591148.941,1.273,0,0
34,04_query_strategy_ablation,qwen3b_wiki_entity_only_k3,qwen2_5_3b,wikipedia,entity_focused,bm25,3,False,False,4,Philosophy and Psychology,3,8.333,5.774,7.667,341666.667,590918.001,1.420,0,0
4,04_query_strategy_ablation,qwen3b_ddg_domain_only_k3,qwen2_5_3b,duckduckgo,domain_prefixed,bm25,3,False,False,4,Philosophy and Psychology,3,10.667,3.215,9.667,64666.667,63002.645,9.675,0,0
8,04_query_strategy_ablation,qwen3b_ddg_entity_only_k3,qwen2_5_3b,duckduckgo,entity_focused,bm25,3,False,False,2,Science and Nature,3,8.667,3.215,7.667,16166.667,15750.661,14.888,0,0
2,04_query_strategy_ablation,qwen3b_ddg_domain_only_k3,qwen2_5_3b,duckduckgo,domain_prefixed,bm25,3,False,False,2,Science and Nature,3,8.000,3.000,7.000,12166.667,17265.090,4.937,0,0
6,04_query_strategy_ablation,qwen3b_ddg_entity_only_k3,qwen2_5_3b,duckduckgo,entity_focused,bm25,3,False,False,0,Entertainment,3,5.667,4.619,4.667,10800.000,18359.739,13.668,0,0
10,04_query_strategy_ablation,qwen3b_ddg_entity_only_k3,qwen2_5_3b,duckduckgo,entity_focused,bm25,3,False,False,4,Philosophy and Psychology,3,7.000,3.606,6.000,6733.333,8247.020,14.408,0,0
20,04_query_strategy_ablation,qwen3b_ddg_options_only_k3,qwen2_5_3b,duckduckgo,question_plus_options,bm25,3,False,False,2,Science and Nature,3,7.667,0.577,6.667,3333.333,1154.701,6.655,0,0
1,04_query_strategy_ablation,qwen3b_ddg_domain_only_k3,qwen2_5_3b,duckduckgo,domain_prefixed,bm25,3,False,False,1,Ancient History and Politics,3,5.333,4.041,4.333,3000.000,4358.899,2.836,0,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
2,04_query_strategy_ablation,qwen3b_ddg_keyword_only_k3,qwen2_5_3b,duckduckgo,keyword_condensed,bm25,3,False,False,18,6,4.833,4.515,3.944,114988.889,330711.570,8.039,0,0
6,04_query_strategy_ablation,qwen3b_wiki_keyword_only_k3,qwen2_5_3b,wikipedia,keyword_condensed,bm25,3,False,False,18,6,3.944,4.465,3.056,114127.778,331015.958,1.359,0,0
5,04_query_strategy_ablation,qwen3b_wiki_entity_only_k3,qwen2_5_3b,wikipedia,entity_focused,bm25,3,False,False,18,6,4.278,3.357,3.333,57444.444,241222.439,1.987,0,0
0,04_query_strategy_ablation,qwen3b_ddg_domain_only_k3,qwen2_5_3b,duckduckgo,domain_prefixed,bm25,3,False,False,18,6,4.778,4.223,3.778,13333.333,32889.566,6.905,0,0
1,04_query_strategy_ablation,qwen3b_ddg_entity_only_k3,qwen2_5_3b,duckduckgo,entity_focused,bm25,3,False,False,18,6,5.167,3.601,4.167,6133.333,10747.476,14.276,0,0
3,04_query_strategy_ablation,qwen3b_ddg_options_only_k3,qwen2_5_3b,duckduckgo,question_plus_options,bm25,3,False,False,18,6,3.500,2.895,2.500,983.333,1528.263,5.746,0,0
4,04_query_strategy_ablation,qwen3b_ddg_raw_only_k3,qwen2_5_3b,duckduckgo,raw_question,bm25,3,False,False,18,6,2.667,1.879,1.667,222.222,288.109,5.281,0,0


Saved: experiment_outputs/04_query_strategy_ablation_run_summary.csv
Saved: experiment_outputs/04_query_strategy_ablation_question_results.csv
Saved: experiment_outputs/04_query_strategy_ablation_per_competition_summary.csv
Saved: experiment_outputs/04_query_strategy_ablation_overall_summary.csv


(                  cluster_name                   combo_name     llm_key  \
 16  04_query_strategy_ablation   qwen3b_ddg_keyword_only_k3  qwen2_5_3b   
 40  04_query_strategy_ablation  qwen3b_wiki_keyword_only_k3  qwen2_5_3b   
 34  04_query_strategy_ablation   qwen3b_wiki_entity_only_k3  qwen2_5_3b   
 4   04_query_strategy_ablation    qwen3b_ddg_domain_only_k3  qwen2_5_3b   
 8   04_query_strategy_ablation    qwen3b_ddg_entity_only_k3  qwen2_5_3b   
 2   04_query_strategy_ablation    qwen3b_ddg_domain_only_k3  qwen2_5_3b   
 6   04_query_strategy_ablation    qwen3b_ddg_entity_only_k3  qwen2_5_3b   
 10  04_query_strategy_ablation    qwen3b_ddg_entity_only_k3  qwen2_5_3b   
 20  04_query_strategy_ablation   qwen3b_ddg_options_only_k3  qwen2_5_3b   
 1   04_query_strategy_ablation    qwen3b_ddg_domain_only_k3  qwen2_5_3b   
 7   04_query_strategy_ablation    qwen3b_ddg_entity_only_k3  qwen2_5_3b   
 12  04_query_strategy_ablation   qwen3b_ddg_keyword_only_k3  qwen2_5_3b   
 14  04_quer

### 05_multiquery_retrieval

In [ ]:
run_cluster("05_multiquery_retrieval")

Selected cluster: 05_multiquery_retrieval
Configs found: 1


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,False,False,mini_cross_encoder,0.0,32



Running cluster: 05_multiquery_retrieval
Total runs: 18
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 1.465 GB
CUDA memory reserved: 2.41 GB

Preloading local model for GPU reuse: qwen2_5_3b
Loading local model: qwen2_5_3b Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.462 GB
CUDA memory reserved: 2.615 GB

[1/18] qwen3b_ddg_multiquery_bm25_k5 | competition=0 | run=1
Finished in 18.48s | level=1 | correct=0 | earned=0.0 | error=False

[2/18] qwen3b_ddg_multiquery_bm25_k5 | competition=0 | run=2
Finished in 7.57s | level=1 | correct=0 | earned=0.0 | error=False

[3/18] qwen3b_ddg_multiquery_bm25_k5 | competition=0 | run=3
Finished in 34.75s | level=3 | correct=2 | earned=200.0 | error=False

[4/18] qwen3b_ddg_multiquery_bm25_k5 | competition=1 | run=1
Finished in 48.08s | level=4 | correct=3 | earned=300.0 | error=False

[5/18] qwen3b_ddg_multiquery_bm25_k5 | competition=1 | run=2
Finished in 76.70s | level=6 | correct=5 | earned=1000.0 | error=False

[6/18] qwen3b_ddg_multiquery_bm25_k5 | competition=1 | run=3
Finished in 20.94s | level=2 | correct=1 | earned=100.0 | error=False

[7/18] qwen3b_ddg_multiquery_bm25_k5 | competition=2 | run=1
Finished in 71.20s | level=6 | correct=5 | e

(                       combo_name     llm_key prompt_style retrieval_tools  \
 0   qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 1   qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 2   qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 3   qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 4   qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 ..                            ...         ...          ...             ...   
 59  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 60  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 61  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 62  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 63  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b       strong      duckduckgo   
 
                                  query_strategies

In [ ]:
summarize_cluster_run()


Cluster: 05_multiquery_retrieval

Run-level summary:


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,cluster_name,run_error
0,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Entertainment,1,1,0,0,0.0,1,17.818000,05_multiquery_retrieval,None
1,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Entertainment,2,1,0,0,0.0,1,7.073000,05_multiquery_retrieval,None
2,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Entertainment,3,3,2,0,200.0,3,11.188667,05_multiquery_retrieval,None
3,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Ancient History and Politics,1,4,3,0,300.0,4,11.580250,05_multiquery_retrieval,None
4,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Ancient History and Politics,2,6,5,0,1000.0,6,12.373000,05_multiquery_retrieval,None
5,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Ancient History and Politics,3,2,1,0,100.0,2,10.009500,05_multiquery_retrieval,None
6,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Science and Nature,1,6,5,0,1000.0,6,11.429333,05_multiquery_retrieval,None
7,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Science and Nature,2,7,6,0,2000.0,7,9.048143,05_multiquery_retrieval,None
8,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Science and Nature,3,4,3,0,300.0,4,10.136500,05_multiquery_retrieval,None
9,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,strong,duckduckgo,entity_focused+keyword_condensed+raw_question,False,2,bm25,5,bge_small,...,Maths,1,1,0,0,0.0,1,7.845000,05_multiquery_retrieval,None



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
2,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,2,Science and Nature,3,5.667,1.528,4.667,1100.000,854.400,10.205,0,0
4,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,4,Philosophy and Psychology,3,5.000,2.646,4.000,1033.333,950.438,10.310,0,0
1,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,1,Ancient History and Politics,3,4.000,2.000,3.000,466.667,472.582,11.321,0,0
5,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,5,News,3,3.667,1.528,2.667,300.000,200.000,10.007,0,0
0,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,0,Entertainment,3,1.667,1.155,0.667,66.667,115.470,12.027,0,0
3,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,3,Maths,3,1.333,0.577,0.333,33.333,57.735,11.820,0,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
0,05_multiquery_retrieval,qwen3b_ddg_multiquery_bm25_k5,qwen2_5_3b,duckduckgo,entity_focused+keyword_condensed+raw_question,bm25,5,False,False,18,6,3.556,2.175,2.556,500.0,646.256,10.948,0,0


Saved: experiment_outputs/05_multiquery_retrieval_run_summary.csv
Saved: experiment_outputs/05_multiquery_retrieval_question_results.csv
Saved: experiment_outputs/05_multiquery_retrieval_per_competition_summary.csv
Saved: experiment_outputs/05_multiquery_retrieval_overall_summary.csv


(              cluster_name                     combo_name     llm_key  \
 2  05_multiquery_retrieval  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b   
 4  05_multiquery_retrieval  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b   
 1  05_multiquery_retrieval  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b   
 5  05_multiquery_retrieval  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b   
 0  05_multiquery_retrieval  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b   
 3  05_multiquery_retrieval  qwen3b_ddg_multiquery_bm25_k5  qwen2_5_3b   
 
   retrieval_tools                               query_strategies  \
 2      duckduckgo  entity_focused+keyword_condensed+raw_question   
 4      duckduckgo  entity_focused+keyword_condensed+raw_question   
 1      duckduckgo  entity_focused+keyword_condensed+raw_question   
 5      duckduckgo  entity_focused+keyword_condensed+raw_question   
 0      duckduckgo  entity_focused+keyword_condensed+raw_question   
 3      duckduckgo  entity_focused+keyword_condensed+raw_question 

### 06_multitool_wiki_ddg

In [ ]:
run_cluster("06_multitool_wiki_ddg")

Selected cluster: 06_multitool_wiki_ddg
Configs found: 1


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,True,False,mini_cross_encoder,0.0,40



Running cluster: 06_multitool_wiki_ddg
Total runs: 18
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 2.462 GB
CUDA memory reserved: 2.959 GB

Preloading local model for GPU reuse: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.462 GB
CUDA memory reserved: 2.959 GB

[1/18] qwen3b_wiki_ddg_rrf_mmr_k3 | competition=0 | run=1
Finished in 101.18s | level=4 | correct=3 | earned=300.0 | error=False

[2/18] qwen3b_wiki_ddg_rrf_mmr_k3 | competition=0 | run=2
Finished in 54.12s | level=2 | correct=1 | earned=100.0 | error=False

[3/18] qwen3b_wiki_ddg_rrf_mmr_k3 | competition=0 | run=3
Finished in 45.70s | level=2 | correct=1 | earned=100.0 | error=False

[4/18] qwen3b_wiki_ddg_rrf_mmr_k3 | competition=1 | run=1
Finished in 105.94s | level=5 | correct=4 | earned=500.0 | error=False

[5/18] qwen3b_wiki_ddg_rrf_mmr_k3 | competition=1 | run=2
Finished in 45.17s | level=2 | correct=1 | earned=100.0 | error=False

[6/18] qwen3b_wiki_ddg_rrf_mmr_k3 | comp

(                    combo_name     llm_key prompt_style       retrieval_tools  \
 0   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 1   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 2   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 3   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 4   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 5   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 6   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 7   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 8   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 9   qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 10  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b  elimination  wikipedia+duckduckgo   
 11  qwen3b_wiki

In [ ]:
summarize_cluster_run()


Cluster: 06_multitool_wiki_ddg

Run-level summary:


,combo_name,llm_key,prompt_style,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,cluster_name,run_error
0,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Entertainment,1,4,3,0,300.0,4,24.814750,06_multitool_wiki_ddg,None
1,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Entertainment,2,2,1,0,100.0,2,26.602000,06_multitool_wiki_ddg,None
2,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Entertainment,3,2,1,1,100.0,2,22.391000,06_multitool_wiki_ddg,None
3,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Ancient History and Politics,1,5,4,0,500.0,5,20.750600,06_multitool_wiki_ddg,None
4,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Ancient History and Politics,2,2,1,0,100.0,2,22.121500,06_multitool_wiki_ddg,None
5,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Ancient History and Politics,3,2,1,1,100.0,2,32.936500,06_multitool_wiki_ddg,None
6,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Science and Nature,1,1,0,0,0.0,1,26.694000,06_multitool_wiki_ddg,None
7,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Science and Nature,2,4,3,1,300.0,4,21.008750,06_multitool_wiki_ddg,None
8,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Science and Nature,3,1,0,1,0.0,1,35.428000,06_multitool_wiki_ddg,None
9,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,elimination,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,False,2,rrf,3,bge_small,...,Maths,1,1,0,0,0.0,1,27.426000,06_multitool_wiki_ddg,None



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
4,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,4,Philosophy and Psychology,3,8.000,6.557,7.333,342033.333,590601.222,18.905,1,0
1,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,1,Ancient History and Politics,3,3.000,1.732,2.000,233.333,230.940,25.270,1,0
0,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,0,Entertainment,3,2.667,1.155,1.667,166.667,115.470,24.603,1,0
2,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,2,Science and Nature,3,2.000,1.732,1.000,100.000,173.205,27.710,2,0
5,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,5,News,3,1.667,0.577,0.667,66.667,57.735,15.640,0,0
3,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,3,Maths,3,1.000,0.000,0.000,0.000,0.000,29.892,2,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
0,06_multitool_wiki_ddg,qwen3b_wiki_ddg_rrf_mmr_k3,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+domain_prefixed+raw_question,rrf,3,True,False,18,6,3.056,3.404,2.111,57100.0,241306.873,23.67,7,0


Saved: experiment_outputs/06_multitool_wiki_ddg_run_summary.csv
Saved: experiment_outputs/06_multitool_wiki_ddg_question_results.csv
Saved: experiment_outputs/06_multitool_wiki_ddg_per_competition_summary.csv
Saved: experiment_outputs/06_multitool_wiki_ddg_overall_summary.csv


(            cluster_name                  combo_name     llm_key  \
 4  06_multitool_wiki_ddg  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b   
 1  06_multitool_wiki_ddg  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b   
 0  06_multitool_wiki_ddg  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b   
 2  06_multitool_wiki_ddg  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b   
 5  06_multitool_wiki_ddg  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b   
 3  06_multitool_wiki_ddg  qwen3b_wiki_ddg_rrf_mmr_k3  qwen2_5_3b   
 
         retrieval_tools                             query_strategies  \
 4  wikipedia+duckduckgo  entity_focused+domain_prefixed+raw_question   
 1  wikipedia+duckduckgo  entity_focused+domain_prefixed+raw_question   
 0  wikipedia+duckduckgo  entity_focused+domain_prefixed+raw_question   
 2  wikipedia+duckduckgo  entity_focused+domain_prefixed+raw_question   
 5  wikipedia+duckduckgo  entity_focused+domain_prefixed+raw_question   
 3  wikipedia+duckduckgo  entity_focused+domain_prefixed+raw_question   
 
  

### 07_prompt_style_comparison

In [ ]:
run_cluster("07_prompt_style_comparison")

Selected cluster: 07_prompt_style_comparison
Configs found: 4


,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,qwen3b_ddg_strong_prompt_k3,qwen2_5_3b,strong,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,elimination,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,40
2,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,strong,text,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,False,5,rrf,5,bge_small,True,False,mini_cross_encoder,0.0,32
3,qwen3b_wiki_ddg_elimination_rrf_k5,qwen2_5_3b,elimination,text,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,False,5,rrf,5,bge_small,True,False,mini_cross_encoder,0.0,40



Running cluster: 07_prompt_style_comparison
Total runs: 72
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 2.465 GB
CUDA memory reserved: 3.223 GB

Preloading local model for GPU reuse: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.465 GB
CUDA memory reserved: 3.223 GB

[1/72] qwen3b_ddg_elimination_prompt_k3 | competition=0 | run=1
DuckDuckGo search | query=Which term describes the distinctive visual style often associated with classic film noirs? failed on attempt 1. Retrying in 0.50s. Error: TimeoutException(TimeoutException("Request timed out: ConnectTimeout('timed out')"))
DuckDuckGo search | query=Which term describes the distinctive visual style often associated with classic film noirs? failed with non-retryable error: DDGSException('No results found.')
DuckDuckGo search | query=describes fifth Beatle bandmate Beatles left before fame failed on attempt 1. Retrying in 0.46s. Error: TimeoutException(TimeoutException("Request timed out

(                           combo_name     llm_key prompt_style game_mode  \
 0    qwen3b_ddg_elimination_prompt_k3  qwen2_5_3b  elimination      text   
 1    qwen3b_ddg_elimination_prompt_k3  qwen2_5_3b  elimination      text   
 2    qwen3b_ddg_elimination_prompt_k3  qwen2_5_3b  elimination      text   
 3    qwen3b_ddg_elimination_prompt_k3  qwen2_5_3b  elimination      text   
 4    qwen3b_ddg_elimination_prompt_k3  qwen2_5_3b  elimination      text   
 ..                                ...         ...          ...       ...   
 191     qwen3b_wiki_ddg_strong_rrf_k5  qwen2_5_3b       strong      text   
 192     qwen3b_wiki_ddg_strong_rrf_k5  qwen2_5_3b       strong      text   
 193     qwen3b_wiki_ddg_strong_rrf_k5  qwen2_5_3b       strong      text   
 194     qwen3b_wiki_ddg_strong_rrf_k5  qwen2_5_3b       strong      text   
 195     qwen3b_wiki_ddg_strong_rrf_k5  qwen2_5_3b       strong      text   
 
           retrieval_tools                                   query_strateg

In [ ]:
summarize_cluster_run()


Cluster: 07_prompt_style_comparison

Run-level summary:


,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,...,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec,run_error,cluster_name
0,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,elimination,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,...,Entertainment,1,6,5,1,1000.0,6,17.214833,NaN,07_prompt_style_comparison
1,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,elimination,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,...,Entertainment,2,4,3,0,300.0,4,11.834000,NaN,07_prompt_style_comparison
2,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,elimination,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,...,Entertainment,3,1,0,0,0.0,1,11.992000,NaN,07_prompt_style_comparison
3,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,elimination,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,...,Ancient History and Politics,1,3,2,0,200.0,3,11.860000,NaN,07_prompt_style_comparison
4,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,elimination,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,...,Ancient History and Politics,2,9,8,0,8000.0,9,9.550889,NaN,07_prompt_style_comparison
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,strong,text,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,False,5,rrf,5,...,Philosophy and Psychology,2,2,1,1,100.0,2,27.260000,NaN,07_prompt_style_comparison
68,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,strong,text,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,False,5,rrf,5,...,Philosophy and Psychology,3,4,3,1,300.0,4,29.355750,NaN,07_prompt_style_comparison
69,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,strong,text,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,False,5,rrf,5,...,News,1,1,0,0,0.0,1,14.979000,NaN,07_prompt_style_comparison
70,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,strong,text,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,False,5,rrf,5,...,News,2,3,2,0,200.0,3,22.612333,NaN,07_prompt_style_comparison



Per-competition summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,competition_id,competition_name,runs,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
10,07_prompt_style_comparison,qwen3b_ddg_strong_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,4,Philosophy and Psychology,3,8.333,6.506,7.667,342700.000,590026.330,16.120,1,0
4,07_prompt_style_comparison,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,4,Philosophy and Psychology,3,5.667,6.658,5.000,42766.667,73814.384,11.080,0,1
1,07_prompt_style_comparison,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,1,Ancient History and Politics,3,5.000,3.464,4.000,2800.000,4503.332,10.449,0,0
2,07_prompt_style_comparison,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,2,Science and Nature,3,6.333,1.528,5.333,1833.333,1892.969,13.289,1,0
7,07_prompt_style_comparison,qwen3b_ddg_strong_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,1,Ancient History and Politics,3,4.667,2.517,3.667,866.667,1001.665,11.208,1,0
0,07_prompt_style_comparison,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,0,Entertainment,3,3.667,2.517,2.667,433.333,513.160,13.680,1,0
8,07_prompt_style_comparison,qwen3b_ddg_strong_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,2,Science and Nature,3,3.333,2.517,2.333,400.000,529.150,18.313,2,0
16,07_prompt_style_comparison,qwen3b_wiki_ddg_elimination_rrf_k5,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,rrf,5,True,False,4,Philosophy and Psychology,3,3.000,1.732,2.000,233.333,230.940,30.167,3,0
14,07_prompt_style_comparison,qwen3b_wiki_ddg_elimination_rrf_k5,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,rrf,5,True,False,2,Science and Nature,3,2.333,2.309,1.333,166.667,288.675,39.006,3,0
18,07_prompt_style_comparison,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,rrf,5,True,False,0,Entertainment,3,2.333,1.528,1.333,133.333,152.753,34.201,3,0



Overall summary:


,cluster_name,combo_name,llm_key,retrieval_tools,query_strategies,filter_method,top_k,use_mmr,use_reranker,total_runs,competitions_tested,avg_max_level_reached,std_max_level_reached,avg_correct_answers,avg_final_earned_amount,std_final_earned_amount,avg_decision_time_sec,total_timeouts,failed_runs
1,07_prompt_style_comparison,qwen3b_ddg_strong_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,18,6,3.444,3.650,2.500,57350.000,241246.148,14.512,5,0
0,07_prompt_style_comparison,qwen3b_ddg_elimination_prompt_k3,qwen2_5_3b,duckduckgo,entity_focused+raw_question,bm25,3,False,False,18,6,4.000,3.361,3.056,7994.444,30015.829,12.768,2,1
2,07_prompt_style_comparison,qwen3b_wiki_ddg_elimination_rrf_k5,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,rrf,5,True,False,18,6,1.833,1.339,0.833,94.444,162.597,32.055,15,0
3,07_prompt_style_comparison,qwen3b_wiki_ddg_strong_rrf_k5,qwen2_5_3b,wikipedia+duckduckgo,entity_focused+question_plus_options+domain_pr...,rrf,5,True,False,18,6,1.611,1.037,0.611,61.111,103.690,33.107,14,0


Saved: experiment_outputs/07_prompt_style_comparison_run_summary.csv
Saved: experiment_outputs/07_prompt_style_comparison_question_results.csv
Saved: experiment_outputs/07_prompt_style_comparison_per_competition_summary.csv
Saved: experiment_outputs/07_prompt_style_comparison_overall_summary.csv


(                  cluster_name                          combo_name  \
 10  07_prompt_style_comparison         qwen3b_ddg_strong_prompt_k3   
 4   07_prompt_style_comparison    qwen3b_ddg_elimination_prompt_k3   
 1   07_prompt_style_comparison    qwen3b_ddg_elimination_prompt_k3   
 2   07_prompt_style_comparison    qwen3b_ddg_elimination_prompt_k3   
 7   07_prompt_style_comparison         qwen3b_ddg_strong_prompt_k3   
 0   07_prompt_style_comparison    qwen3b_ddg_elimination_prompt_k3   
 8   07_prompt_style_comparison         qwen3b_ddg_strong_prompt_k3   
 16  07_prompt_style_comparison  qwen3b_wiki_ddg_elimination_rrf_k5   
 14  07_prompt_style_comparison  qwen3b_wiki_ddg_elimination_rrf_k5   
 18  07_prompt_style_comparison       qwen3b_wiki_ddg_strong_rrf_k5   
 22  07_prompt_style_comparison       qwen3b_wiki_ddg_strong_rrf_k5   
 6   07_prompt_style_comparison         qwen3b_ddg_strong_prompt_k3   
 13  07_prompt_style_comparison  qwen3b_wiki_ddg_elimination_rrf_k5   
 3   0

### 08_top_k_comparison

In [ ]:
run_cluster("08_top_k_comparison")

Selected cluster: 08_top_k_comparison
Configs found: 5


,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,qwen3b_ddg_bm25_k1_entity_raw,qwen2_5_3b,strong,text,duckduckgo,entity_focused+raw_question,False,5,bm25,1,bge_small,False,False,mini_cross_encoder,0.0,32
1,qwen3b_ddg_bm25_k3_entity_raw,qwen2_5_3b,strong,text,duckduckgo,entity_focused+raw_question,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32
2,qwen3b_ddg_bm25_k5_entity_raw,qwen2_5_3b,strong,text,duckduckgo,entity_focused+raw_question,False,5,bm25,5,bge_small,False,False,mini_cross_encoder,0.0,32
3,qwen3b_wiki_bm25_k1_entity_raw,qwen2_5_3b,strong,text,wikipedia,entity_focused+raw_question,False,5,bm25,1,bge_small,False,False,mini_cross_encoder,0.0,32
4,qwen3b_wiki_bm25_k5_entity_raw,qwen2_5_3b,strong,text,wikipedia,entity_focused+raw_question,False,5,bm25,5,bge_small,False,False,mini_cross_encoder,0.0,32



Running cluster: 08_top_k_comparison
Total runs: 90
Competitions: [0, 1, 2, 3, 4, 5]
GPU available: Tesla T4
CUDA memory allocated: 2.465 GB
CUDA memory reserved: 3.223 GB

Preloading local model for GPU reuse: qwen2_5_3b
GPU available: Tesla T4
CUDA memory allocated: 2.465 GB
CUDA memory reserved: 3.223 GB

[1/90] qwen3b_ddg_bm25_k1_entity_raw | competition=0 | run=1
Finished in 11.22s | level=1 | correct=0 | earned=0.0 | error=False

[2/90] qwen3b_ddg_bm25_k1_entity_raw | competition=0 | run=2
Finished in 13.84s | level=1 | correct=0 | earned=0.0 | error=False

[3/90] qwen3b_ddg_bm25_k1_entity_raw | competition=0 | run=3
DuckDuckGo search | query=What was the fundamental principle of John Lennon's songwriting with Paul McCartney in the Beatles? failed on attempt 1. Retrying in 0.37s. Error: TimeoutException(TimeoutException("Request timed out: ConnectTimeout('timed out')"))
DuckDuckGo search | query=What was the fundamental principle of John Lennon's songwriting with Paul McCartney 

(                         combo_name     llm_key prompt_style game_mode  \
 0     qwen3b_ddg_bm25_k1_entity_raw  qwen2_5_3b       strong      text   
 1     qwen3b_ddg_bm25_k1_entity_raw  qwen2_5_3b       strong      text   
 2     qwen3b_ddg_bm25_k1_entity_raw  qwen2_5_3b       strong      text   
 3     qwen3b_ddg_bm25_k1_entity_raw  qwen2_5_3b       strong      text   
 4     qwen3b_ddg_bm25_k1_entity_raw  qwen2_5_3b       strong      text   
 ..                              ...         ...          ...       ...   
 292  qwen3b_wiki_bm25_k5_entity_raw  qwen2_5_3b       strong      text   
 293  qwen3b_wiki_bm25_k5_entity_raw  qwen2_5_3b       strong      text   
 294  qwen3b_wiki_bm25_k5_entity_raw  qwen2_5_3b       strong      text   
 295  qwen3b_wiki_bm25_k5_entity_raw  qwen2_5_3b       strong      text   
 296  qwen3b_wiki_bm25_k5_entity_raw  qwen2_5_3b       strong      text   
 
     retrieval_tools             query_strategies  fetch_pages  \
 0        duckduckgo  entity_foc

In [ ]:
summarize_cluster_run()

## 14. Speech Experiments

### Speech experiment combinations


In [ ]:

SPEECH_EXPERIMENT_CONFIGS = [

    # ========================================================
    # A) Speech LLM-only baseline
    # Purpose:
    # - Validate the full speech pipeline:
    #   audio -> Whisper -> q_item -> LLM -> answer mapping -> submit.
    # - No retrieval, so it is the fastest speech test.
    # ========================================================

    ExperimentConfig(
        combo_name="speech_qwen3b_llm_only_strong",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        game_mode="speech",
        retrieval_tools=(),
        query_strategies=(),
        filter_method="none",
        top_k=0,
        max_new_tokens=32,
    ),

    # ========================================================
    # B) Speech + DuckDuckGo RAG
    # Purpose:
    # - Tests whether retrieval still helps after speech transcription.
    # - Uses the same generic solver pipeline as text mode.
    # ========================================================

    ExperimentConfig(
        combo_name="speech_qwen3b_ddg_bm25_k3_entity_raw",
        llm_key="qwen2_5_3b",
        prompt_style="strong",
        game_mode="speech",
        retrieval_tools=("duckduckgo",),
        query_strategies=("entity_focused", "raw_question"),
        filter_method="bm25",
        top_k=3,
        max_new_tokens=32,
    ),
]


speech_configs_preview_df = pd.DataFrame(
    [config_to_row(cfg) for cfg in SPEECH_EXPERIMENT_CONFIGS]
)

display(speech_configs_preview_df)

,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,speech_qwen3b_llm_only_strong,qwen2_5_3b,strong,speech,none,none,False,5,none,0,bge_small,False,False,mini_cross_encoder,0.0,32
1,speech_qwen3b_ddg_bm25_k3_entity_raw,qwen2_5_3b,strong,speech,duckduckgo,entity_focused+raw_question,False,5,bm25,3,bge_small,False,False,mini_cross_encoder,0.0,32


### Speech combo clusters


In [ ]:

SPEECH_EXPERIMENT_CLUSTERS = {
    "speech_00_sanity_check": [
        "speech_qwen3b_llm_only_strong",
    ],

    "speech_01_small_rag_test": [
        "speech_qwen3b_llm_only_strong",
        "speech_qwen3b_ddg_bm25_k3_entity_raw",
    ],
}


def select_speech_cluster(cluster_name):
    selected_names = SPEECH_EXPERIMENT_CLUSTERS[cluster_name]

    selected_configs = [
        cfg for cfg in SPEECH_EXPERIMENT_CONFIGS
        if cfg.combo_name in selected_names
    ]

    found_names = [cfg.combo_name for cfg in selected_configs]
    missing_names = [name for name in selected_names if name not in found_names]

    print("Selected speech cluster:", cluster_name)
    print("Configs found:", len(selected_configs))

    if missing_names:
        print("Missing speech configs:")
        for name in missing_names:
            print("-", name)

    if selected_configs:
        display(pd.DataFrame([config_to_row(cfg) for cfg in selected_configs]))
    else:
        print("This speech cluster is empty.")

    return selected_configs

### Run selected speech cluster

In [ ]:
def run_speech_cluster(
    cluster_name="speech_00_sanity_check",
    competition_ids=None,
    num_runs=1,
    sleep_between_questions=0.5,
    verbose=True,
    whisper_model=None,
    warmup_llms=True,
):
    global SPEECH_CLUSTER_TO_RUN
    global SPEECH_CONFIGS_TO_RUN
    global speech_question_level_results_df
    global speech_run_summary_df
    global speech_per_competition_summary_df
    global speech_overall_summary_df

    SPEECH_CLUSTER_TO_RUN = cluster_name
    SPEECH_CONFIGS_TO_RUN = select_speech_cluster(SPEECH_CLUSTER_TO_RUN)

    if competition_ids is None:
        competition_ids = resolve_competition_ids()

    # Load Whisper once before starting any live speech game.
    # This avoids transcription model-loading delay inside the competition.
    if whisper_model is None:
        print("\nLoading speech transcription model before starting games...")
        whisper_model = get_whisper_model()
        print("Speech transcription model ready.")

    # Warm up answering LLMs before starting any live speech game.
    # This avoids downloading/loading Qwen or other local models during the timer.
    if warmup_llms:
        print("\nWarming up LLMs before starting games...")

        llm_keys_to_warmup = sorted({cfg.llm_key for cfg in SPEECH_CONFIGS_TO_RUN})

        for llm_key in llm_keys_to_warmup:
            print(f"Warmup LLM: {llm_key}")
            try:
                _ = invoke_llm(
                    "Return only the digit 0.",
                    llm_key=llm_key,
                    max_new_tokens=2,
                    temperature=0.0,
                )
            except Exception as e:
                print(f"LLM warmup failed for {llm_key}: {repr(e)}")
                raise

        print("LLM warmup complete.")

    print("\n" + "=" * 100)
    print("Running speech cluster:", SPEECH_CLUSTER_TO_RUN)
    print("Competitions:", competition_ids)
    print("Runs per config/competition:", num_runs)
    print("=" * 100)

    all_question_rows = []
    all_run_summaries = []

    for cfg in SPEECH_CONFIGS_TO_RUN:
        for competition_id in competition_ids:
            for run_id in range(1, num_runs + 1):

                question_df, run_summary = play_live_game_with_config(
                    client=client,
                    competition_id=competition_id,
                    cfg=cfg,
                    run_id=run_id,
                    sleep_between_questions=sleep_between_questions,
                    verbose=verbose,
                    whisper_model=whisper_model,
                )

                if not question_df.empty:
                    all_question_rows.append(question_df)

                all_run_summaries.append(run_summary)

    if all_question_rows:
        speech_question_level_results_df = pd.concat(
            all_question_rows,
            ignore_index=True,
        )
    else:
        speech_question_level_results_df = pd.DataFrame()

    speech_run_summary_df = pd.DataFrame(all_run_summaries)

    if not speech_question_level_results_df.empty:
        speech_per_competition_summary_df = (
            speech_question_level_results_df
            .groupby(["combo_name", "competition_id", "competition_name"], dropna=False)
            .agg(
                questions_answered=("correct", "count"),
                correct_answers=("correct", "sum"),
                timeout_count=("timed_out", "sum"),
                max_level_reached=("level_before_answer", "max"),
                final_earned_amount=("earned_amount_after_question", "max"),
                avg_decision_time_sec=("decision_time_sec", "mean"),
            )
            .reset_index()
        )

        speech_per_competition_summary_df["accuracy"] = (
            speech_per_competition_summary_df["correct_answers"]
            / speech_per_competition_summary_df["questions_answered"]
        )

        speech_overall_summary_df = (
            speech_question_level_results_df
            .groupby(["combo_name"], dropna=False)
            .agg(
                questions_answered=("correct", "count"),
                correct_answers=("correct", "sum"),
                timeout_count=("timed_out", "sum"),
                max_level_reached=("level_before_answer", "max"),
                avg_decision_time_sec=("decision_time_sec", "mean"),
            )
            .reset_index()
        )

        speech_overall_summary_df["accuracy"] = (
            speech_overall_summary_df["correct_answers"]
            / speech_overall_summary_df["questions_answered"]
        )

    else:
        speech_per_competition_summary_df = pd.DataFrame()
        speech_overall_summary_df = pd.DataFrame()

    print("\nSpeech run completed.")

    print("\nSpeech run summary:")
    display(speech_run_summary_df)

    if not speech_overall_summary_df.empty:
        print("\nSpeech overall summary:")
        display(speech_overall_summary_df)

    return (
        speech_question_level_results_df,
        speech_run_summary_df,
        speech_per_competition_summary_df,
        speech_overall_summary_df,
    )

### Optional speech RAG test


In [ ]:
speech_question_level_results_df, speech_run_summary_df, speech_per_competition_summary_df, speech_overall_summary_df = run_speech_cluster(
    cluster_name="speech_00_sanity_check",
    competition_ids=[0],
    num_runs=1,
    sleep_between_questions=0.5,
    verbose=True,
    whisper_model = get_whisper_model(),
)

Selected speech cluster: speech_00_sanity_check
Configs found: 1


,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,embedder_name,use_mmr,use_reranker,reranker_name,temperature,max_new_tokens
0,speech_qwen3b_llm_only_strong,qwen2_5_3b,strong,speech,none,none,False,5,none,0,bge_small,False,False,mini_cross_encoder,0.0,32



Warming up LLMs before starting games...
Warmup LLM: qwen2_5_3b
LLM warmup complete.

Running speech cluster: speech_00_sanity_check
Competitions: [0]
Runs per config/competition: 1

Combo: speech_qwen3b_llm_only_strong | Mode: speech | Competition: 0 - Entertainment | Run: 1
Session ID: 260639

--- Level 1 ---
What is Mr. Bean known for in his television and film appearances?
0. Playing a highly intelligent detective.
1. Engaging in physical comedy and silly situations.
2. Singing and dancing and musical performances.
3. using complex dialogue and language.
Time remaining: 28.1s
Predicted zero-based option: 1
Predicted letter: B
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 1
Decision time: 0.297
Raw answer: 1
Correct: True Timed out: False Earned: 100

--- Level 2 ---
Which of the following awards did Judy Dance win for her role as Queen Elizabeth I in Shakespearean Love?
0. Oscar for Best Supporting Actress!
1. Golden Globe for Best Supporting Actress.
2. OUPTI

,combo_name,llm_key,prompt_style,game_mode,retrieval_tools,query_strategies,fetch_pages,max_results_per_tool,filter_method,top_k,...,max_new_tokens,competition_id,competition_name,run_id,questions_answered,correct_answers,timeout_count,final_earned_amount,max_level_reached,avg_decision_time_sec
0,speech_qwen3b_llm_only_strong,qwen2_5_3b,strong,speech,none,none,False,5,none,0,...,32,0,Entertainment,1,2,1,0,100.0,2,0.267



Speech overall summary:


,combo_name,questions_answered,correct_answers,timeout_count,max_level_reached,avg_decision_time_sec,accuracy
0,speech_qwen3b_llm_only_strong,2,1,0,2,0.267,0.5


# Math Competetion Further Investigation

This section runs a focused follow-up investigation for the math competition using two reasoning-focused models: `DeepSeek-R1-Distill-Qwen-7B` and `Qwen2.5-7B`.

The investigation tests timing, priming, and calculator-tool choices while keeping the rest of the notebook untouched. Function names in this section were checked against earlier sections; no function-name conflicts were found. The section-local configuration class was renamed to `MathInvestigationConfig` because `ExperimentConfig` already exists earlier in the notebook.

### Imports and Runtime Dependencies

This cell gathers the libraries needed only for the math investigation. It includes standard utilities, tabular result handling, GPU/model loading, symbolic math tools, optimization helpers, and Hugging Face generation controls.


In [ ]:
import importlib.util
import itertools
import gc
import json
import math
import os
import random
import re
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from statistics import mean, median, mode, stdev, variance

import pandas as pd
import torch
from scipy.optimize import linprog, minimize, minimize_scalar
from sympy import (
    E,
    I,
    Matrix,
    Rational,
    binomial,
    cos,
    diff,
    exp,
    expand,
    factor,
    factorial,
    factorint,
    gcd,
    integrate,
    isprime,
    lcm,
    limit,
    log,
    oo,
    pi,
    simplify,
    sin,
    solve,
    sqrt,
    symbols,
    tan,
)
import sympy
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

### Question Formatting Helper

`format_question` converts the game question object into a compact prompt with the question text, numbered answer options, and an explicit answer area. The solvers reuse this formatting so each model receives a consistent multiple-choice prompt.


In [ ]:
def format_question(question):
    """
    Strong prompt forcing structured output.
    """

    options_text = "\n".join(
        [f"{opt.id}. {opt.text}" for opt in question.options]
    )

    return f"""
          Question:
          {question.text}

          Options:
          {options_text}

          Answer:
          """.strip()

### Game Logging and Automatic Play Loop

`log_question_result` stores one row per answered question, including the selected answer, correctness, timeout status, earned amount, decision time, and model reasoning text.

`play_game_auto` runs one full game with a supplied solver function. It prints each question, calls the solver, submits the chosen answer to the game client, logs the result, and adds a per-game summary to `game_logs`.

In [ ]:
def log_question_result(
    run_id,
    solver_name,
    model_name,
    level,
    question,
    selected_answer,
    result,
    time_taken,
    thinking="",
):
    run_logs.append(
        {
            "run_id": run_id,
            "solver": solver_name,
            "model": model_name,
            "level": level,
            "question": question.text,
            "options": {opt.id: opt.text for opt in question.options},
            "selected_answer": selected_answer,
            "correct": result.correct,
            "timed_out": result.timed_out,
            "earned_amount": result.earned_amount,
            "decision_time_sec": round(time_taken, 3),
            "thinking": thinking,
        }
    )


def play_game_auto(game, solver_function, solver_name, model_name, run_id):
    game_start_log_idx = len(run_logs)

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available.")
            break

        current_level = game.current_level
        print(f"\n--- Level {current_level} ---")
        print(format_question(question))

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        start_time = time.time()
        answer_id, thinking = solver_function(question)
        time_taken = time.time() - start_time

        print(f"\nSelected answer: {answer_id}")
        print(f"Decision time: {time_taken:.2f}s")

        result = game.answer(answer_id)

        log_question_result(
            run_id=run_id,
            solver_name=solver_name,
            model_name=model_name,
            level=current_level,
            question=question,
            selected_answer=answer_id,
            result=result,
            time_taken=time_taken,
            thinking=thinking,
        )

        if result.correct:
            print("CORRECT!")
        elif result.timed_out:
            print("TIMED OUT!")
        else:
            print("WRONG ANSWER!")

    game_question_logs = run_logs[game_start_log_idx:]
    answered_levels = [row["level"] for row in game_question_logs]
    decision_times = [row["decision_time_sec"] for row in game_question_logs]
    timeout_count = sum(1 for row in game_question_logs if row["timed_out"])

    summary = {
        "run_id": run_id,
        "solver": solver_name,
        "model": model_name,
        "levels_reached": game.current_level,
        "decision_time": mean(decision_times) if decision_times else None,
        "total_decision_time": sum(decision_times),
        "questions_answered": len(answered_levels),
        "timeouts": timeout_count,
        "earned_amount": game.earned_amount,
    }
    game_logs.append(summary)

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")
    return summary

### Model Output Cleaning and Answer Parsing

These helpers turn raw model text into a valid option ID. They remove thinking/tool markup, look for valid numeric option IDs, and fall back to matching option text when the model answers with words instead of an index.


In [ ]:
def clean_model_response(response):
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_call>.*?</tool_call>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_response>.*?</tool_response>", "", response, flags=re.DOTALL)
    return response.replace("<|im_end|>", "").replace("<|im_start|>", "").strip()

def parse_model_answer(response, question):
    """
    Parse model output into a valid option ID.
    Handles numeric answers and option-text answers.
    """
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_call>.*?</tool_call>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_response>.*?</tool_response>", "", response, flags=re.DOTALL)
    response = response.replace("<|im_end|>", "").replace("<|im_start|>", "").strip()

    response_clean = response.lower()
    valid_option_ids = [opt.id for opt in question.options]

    for match in reversed(list(re.finditer(r"\b(\d+)\b", response_clean))):
        num = int(match.group(1))
        if num in valid_option_ids:
            return num

    for opt in question.options:
        option_text = opt.text.strip().lower()
        if option_text and (option_text in response_clean or response_clean in option_text):
            return opt.id

    return None

def parse_final_answer(response, question):
    """
    Parse a short completion after prompts like "The answer index is".
    Searches from the beginning because the first digit should be the answer;
    later digits are often explanation steps.
    """
    response = clean_model_response(response)
    response_clean = response.lower()
    valid_option_ids = [opt.id for opt in question.options]

    for match in re.finditer(r"\b(\d+)\b", response_clean):
        num = int(match.group(1))
        if num in valid_option_ids:
            return num

    for opt in question.options:
        option_text = opt.text.strip().lower()
        if option_text and (option_text in response_clean or response_clean in option_text):
            return opt.id

    return None

### Model Loading

`load_model` loads the tokenizer and causal language model for the selected checkpoint. It uses 4-bit quantization and automatic device placement so the larger reasoning models can run within GPU memory constraints when CUDA is available.


In [ ]:
def load_model(model_name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)
    print("Loading model:", model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    quant_config = BitsAndBytesConfig(load_in_4bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
    )
    return tokenizer, model, device

### Generation Stop Criteria

These custom stopping criteria let generation stop when either the time budget expires or a specific token sequence appears. The solver phases use them to enforce soft and hard deadlines during reasoning.


In [ ]:
class StopOnTime(StoppingCriteria):
    def __init__(self, deadline):
        self.deadline = deadline

    def __call__(self, input_ids, scores, **kwargs):
        return time.time() >= self.deadline


class StopOnTokenSeq(StoppingCriteria):
    def __init__(self, token_ids):
        self.seq = token_ids

    def __call__(self, input_ids, scores, **kwargs):
        return len(self.seq) > 0 and input_ids[0, -len(self.seq) :].tolist() == self.seq

### Shared Model Runtime Wrapper

`ModelRuntime` wraps the tokenizer and model behind small convenience methods for appending text, decoding newly generated tokens, running timed generation, and building chat-template token inputs. This keeps the Qwen and DeepSeek solver code focused on strategy instead of low-level model calls.


In [ ]:
class ModelRuntime:
    def __init__(self, model_name):
        self.tokenizer, self.model, self.device = load_model(model_name)

    def append_text(self, tokens, text):
        ids = self.tokenizer.encode(text, add_special_tokens=False)
        return torch.cat([tokens, torch.tensor([ids], device=tokens.device)], dim=-1)

    def decode_new(self, tokens, prev_len, skip_special_tokens=False):
        return self.tokenizer.decode(tokens[0, prev_len:], skip_special_tokens=skip_special_tokens)

    def generate(self, tokens, max_new, deadline, stop_sequences=None):
        criteria_list = [StopOnTime(deadline)]
        for seq in stop_sequences or []:
            criteria_list.append(StopOnTokenSeq(seq))

        mask = torch.ones((1, tokens.shape[-1]), device=tokens.device)
        with torch.no_grad():
            out = self.model.generate(
                tokens,
                attention_mask=mask,
                max_new_tokens=max_new,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList(criteria_list),
            )
        return out[0].unsqueeze(0)

    def chat_tokens(self, messages):
        return self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )["input_ids"].to(self.device)

### Qwen Calculator Tool

This block defines a restricted math namespace and a small tool-call executor for Qwen. When enabled, Qwen can emit a `math_tool` request, evaluate expressions with Python/SymPy utilities, and receive the result back in the conversation.

In [ ]:
# =============================================================================
# Qwen-only math tool
# =============================================================================

x, y, z, n, t, a, b, c, k = symbols("x y z n t a b c k")

CALC_NAMESPACE = {
    "len": len,
    "list": list,
    "range": range,
    "sum": sum,
    "min": min,
    "max": max,
    "abs": abs,
    "round": round,
    "int": int,
    "float": float,
    "str": str,
    "enumerate": enumerate,
    "zip": zip,
    "map": map,
    "sorted": sorted,
    "reversed": reversed,
    "mean": mean,
    "median": median,
    "mode": mode,
    "stdev": stdev,
    "variance": variance,
    "x": x,
    "y": y,
    "z": z,
    "n": n,
    "t": t,
    "a": a,
    "b": b,
    "c": c,
    "k": k,
    "math": math,
    "comb": math.comb,
    "perm": math.perm,
    "ceil": math.ceil,
    "floor": math.floor,
    "log": math.log,
    "sqrt": math.sqrt,
    "sympy": sympy,
    "symbols": symbols,
    "solve": solve,
    "simplify": simplify,
    "expand": expand,
    "factor": factor,
    "integrate": integrate,
    "diff": diff,
    "limit": limit,
    "Matrix": Matrix,
    "Rational": Rational,
    "factorial": factorial,
    "binomial": binomial,
    "isprime": isprime,
    "factorint": factorint,
    "gcd": gcd,
    "lcm": lcm,
    "oo": oo,
    "E": E,
    "I": I,
    "pi": pi,
    "sin": sin,
    "cos": cos,
    "tan": tan,
    "exp": exp,
    "itertools": itertools,
    "linprog": linprog,
    "minimize": minimize,
    "minimize_scalar": minimize_scalar,
    "__builtins__": {},
}


def qwen_calc(expr):
    expr = expr.strip().rstrip(";")
    try:
        result = eval(expr, CALC_NAMESPACE)
        print(f"calc({expr!r}) -> {result}")
        return str(result)
    except NameError as exc:
        missing = re.findall(r"'(\w)'", str(exc))
        if missing:
            extra = {name: symbols(name) for name in missing}
            try:
                result = eval(expr, {**CALC_NAMESPACE, **extra})
                print(f"calc({expr!r}) -> {result} (auto-symbols: {missing})")
                return str(result)
            except Exception as exc2:
                return f"Error: {exc2}"
        return f"Error: {exc}"
    except Exception as exc:
        return f"Error: {exc}"

def execute_qwen_tool_call(raw_json):
    try:
        call = json.loads(raw_json.strip())
        if call.get("name") != "math_tool":
            return f"Unknown tool: {call.get('name')}"
        return qwen_calc(call.get("arguments", {}).get("expression", ""))
    except json.JSONDecodeError as exc:
        return f"JSON parse error: {exc}"
    except Exception as exc:
        return f"Tool execution error: {exc}"


QWEN_CALC_SYSTEM_PROMPT = """You are attending a quiz. Answer accurately in as few words as possible with the following format:

STEP A - In one sentence, state the idea to solve the problem.

STEP B - Solve the problem. For all calculations use the MATH TOOL described below for a variety of mathematical calculations.

STEP C - Look for the corresponding option and output the chosen option in ONE digit: 0, 1, 2, or 3. Nothing after it.

MATH TOOL (only for non-trivial arithmetic):
You have access to a math_tool that evaluates SymPy-compatible Python expressions.

To call it, use the following syntax:
<tool_call>
{"name": "math_tool", "arguments": {"expression": "YOUR_SYMPY_EXPRESSION"}}
</tool_call>

Don't forget to close the tool call tag after you opened it.

RULES:
- Use the smartest way to find the answer.
- Step 3 is the answer INDEX (0-3), not the computed value.
"""

QWEN_NO_TOOL_SYSTEM_PROMPT = """You are attending a quiz. Answer accurately in as few words as possible with the following format:

STEP A - In one sentence, state the idea to solve the problem.

STEP B - Solve the problem directly without calling tools.

STEP C - Look for the corresponding option and output the chosen option in ONE digit: 0, 1, 2, or 3. Nothing after it.

RULES:
- Use the smartest way to find the answer.
- Step 3 is the answer INDEX (0-3), not the computed value.
"""

DEEPSEEK_SYSTEM_PROMPT = (
    "You are a math expert. Think concisely and answer the multiple-choice "
    "question with a single digit: 0, 1, 2, or 3."
)


def system_prompt_for(cfg):
    if cfg.model_family == "deepseek":
        return DEEPSEEK_SYSTEM_PROMPT
    if cfg.use_calc_tool:
        return QWEN_CALC_SYSTEM_PROMPT
    return QWEN_NO_TOOL_SYSTEM_PROMPT

### Optional Priming Conversation

`build_priming_history` can warm up the model before the timed game starts. It sends the selected strategy as a short pre-game instruction and stores the model confirmation so later prompts can include the same conversational context.

In [ ]:
def build_priming_history(runtime, cfg):
    if not cfg.use_priming:
        return []

    system_prompt = system_prompt_for(cfg)
    if cfg.model_family == "qwen":
        priming_messages = [
            {
                "role": "system",
                "content": "You are a quiz expert that answers multiple-choice questions.",
            },
            {
                "role": "user",
                "content": (
                    "Before we start, here is the strategy you must follow for every question.\n\n"
                    + system_prompt
                    + "\n\nConfirm you understood by summarizing the strategy in 3 bullet points."
                ),
            },
        ]
    else:
        priming_messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    "Before we start, confirm the answering strategy in 3 short bullets. "
                    "Remember: the final output for each question must be only the option index."
                ),
            },
        ]

    tokens = runtime.chat_tokens(priming_messages)
    prev_len = tokens.shape[-1]
    tokens = runtime.generate(tokens, max_new=150, deadline=time.time() + 30.0)
    confirmation = runtime.decode_new(tokens, prev_len).replace("<|im_end|>", "").strip()
    print(f"Model confirmed strategy:\n{confirmation}\n")

    return [*priming_messages, {"role": "assistant", "content": confirmation}]

### Qwen Thinking Solver

The Qwen solver manages a timed reasoning phase, optional calculator-tool calls, speed-up prompts, wrap-up prompts, and final answer forcing. It is designed to let Qwen reason when time allows while still producing a valid option index before the deadline.


In [ ]:
# =============================================================================
# Qwen solver
# =============================================================================

QWEN_SPEEDUP_MSG = "\nLet's reason a little bit faster:"
QWEN_WRAPUP_MSG = "\nI need to wrap up. In conclusion:"


def qwen_thinking_phase(runtime, tokens, t_start, cfg, options_text):
    tool_call_end_ids = runtime.tokenizer.encode("</tool_call>", add_special_tokens=False)
    im_end_ids = runtime.tokenizer.encode("<|im_end|>", add_special_tokens=False)

    gen_stops = [im_end_ids]
    if cfg.use_calc_tool:
        gen_stops.append(tool_call_end_ids)

    log = []
    tool_calls = 0
    t_hard = t_start + cfg.t_hard_s

    def force_answer(current_tokens, deadline, max_new, label):
        nudge = f"\nOptions: {options_text}. My final answer option index is "
        out = runtime.append_text(current_tokens, nudge)
        log.append(nudge)
        print(f"{label} nudge: '{nudge.strip()}'")

        prev_len = out.shape[-1]
        out = runtime.generate(out, max_new=max_new, deadline=deadline, stop_sequences=[])
        segment = runtime.decode_new(out, prev_len)
        log.append(segment)

        n_im = len(im_end_ids)
        if n_im > 0 and out.shape[-1] >= n_im and out[0, -n_im:].tolist() == im_end_ids:
            out = out[:, :-n_im]

        early_text = segment.replace("<|im_end|>", "").strip()
        print(f"✅ Answer: '{early_text}' ({time.time() - t_start:.1f}s).")
        return out, log, early_text

    def generate_until(deadline, max_new=150, wrapup=False):
        nonlocal tokens, tool_calls
        while True:
            prev_len = tokens.shape[-1]
            tokens = runtime.generate(
                tokens,
                max_new=max_new,
                deadline=deadline,
                stop_sequences=gen_stops,
            )
            segment = runtime.decode_new(tokens, prev_len)
            log.append(segment)

            elapsed = time.time() - t_start
            if wrapup:
                print(f"💭 wrap-up ({elapsed:.1f}s):\n{segment}\n")
            else:
                print(f"💭 ({elapsed:.1f}s):\n{segment}\n")

            natural_end = False
            n_im = len(im_end_ids)
            if n_im > 0 and tokens.shape[-1] >= n_im and tokens[0, -n_im:].tolist() == im_end_ids:
                tokens = tokens[:, :-n_im]
                natural_end = True

            if cfg.use_calc_tool:
                tool_matches = re.findall(r"<tool_call>(.*?)</tool_call>", segment, re.DOTALL)
                if tool_matches and tool_calls < cfg.max_tool_calls:
                    tool_calls += 1
                    result = execute_qwen_tool_call(tool_matches[-1])
                    tool_response = f"<tool_response>\n{result}\n</tool_response>\n"
                    tokens = runtime.append_text(tokens, tool_response)
                    log.append(tool_response)
                    continue

            already_answered = bool(
                re.search(r"\b[0-3]\b\s*(?:<\|im_end\|>)?\s*$", segment.strip())
            )
            if already_answered or natural_end:
                if wrapup:
                    print(f"✅ Wrapped up naturally ({elapsed:.1f}s).")
                else:
                    print(f"✅ Done ({elapsed:.1f}s).")
                return "done"

            return "deadline"

    if not cfg.use_hard_deadline:
        generate_until(t_start + cfg.t_answer_s)
        return tokens, log, None

    warning_deadlines = [t_start + t for t in cfg.warning_times_s]

    for idx, warning_deadline in enumerate(warning_deadlines):
        state = generate_until(warning_deadline)
        if state == "done":
            return tokens, log, None

        elapsed = time.time() - t_start
        is_last_warning = idx == len(warning_deadlines) - 1
        if is_last_warning:
            print(f"⏳ Wrap-up warning ({elapsed:.1f}s), nudging to conclude...")
            tokens = runtime.append_text(tokens, QWEN_WRAPUP_MSG)
            log.append(QWEN_WRAPUP_MSG)

            state = generate_until(t_hard, max_new=150, wrapup=True)
            if state == "done":
                return tokens, log, None

            print(f"🛑 Hard deadline ({time.time() - t_start:.1f}s), forcing answer.")
            return force_answer(
                tokens,
                deadline=t_start + cfg.t_answer_s,
                max_new=3,
                label="🛑 Hard",
            )

        print(
            f"⚡ Speed-up warning {idx}"
            f" ({elapsed:.1f}s), nudging to go faster..."
        )
        tokens = runtime.append_text(tokens, QWEN_SPEEDUP_MSG)
        log.append(QWEN_SPEEDUP_MSG)

    state = generate_until(t_hard)
    if state == "done":
        return tokens, log, None

    print(f"🛑 Hard deadline ({time.time() - t_start:.1f}s), forcing answer.")
    return force_answer(tokens, deadline=t_start + cfg.t_answer_s, max_new=3, label="🛑 Hard")


def build_qwen_solver(runtime, cfg, priming_history):
    def solver(question):
        options_text = " | ".join(f"{opt.id}={opt.text}" for opt in question.options)
        display_options = "\n".join(f"{opt.id}. {opt.text}" for opt in question.options)

        base_messages = priming_history or [{"role": "system", "content": system_prompt_for(cfg)}]
        messages = [
            *base_messages,
            {
                "role": "user",
                "content": f"Question: {question.text}\n\nOptions:\n{display_options}\n\n",
            },
        ]

        tokens = runtime.chat_tokens(messages)
        t_start = time.time()
        tokens, log, early = qwen_thinking_phase(runtime, tokens, t_start, cfg, options_text)

        if early is not None:
            answer_id = parse_final_answer(early, question)
            if answer_id is not None:
                print(f"[{cfg.name}] Option {answer_id} (nudged, {time.time() - t_start:.1f}s)")
                return answer_id, "\n".join(log)

        suffix = f"\nOptions: {options_text}. The answer option index is "
        print(f"🔚 Suffix: '{suffix.strip()}'")
        tokens = runtime.append_text(tokens, suffix)
        prev_len = tokens.shape[-1]
        tokens = runtime.generate(tokens, max_new=5, deadline=t_start + cfg.t_answer_s)
        response = runtime.decode_new(tokens, prev_len, skip_special_tokens=True).strip()

        print(f"🤖 [{cfg.name}] '{response}' (total: {time.time() - t_start:.1f}s)")
        answer_id = parse_final_answer(response, question)
        if answer_id is not None:
            print(f"✔️  Option {answer_id}")
            return answer_id, "\n".join(log)

        print("⚠️  Fallback to random")
        return random_solver(question)

    return solver

### DeepSeek Thinking Solver

The DeepSeek solver follows the same timed-answer idea but stays tool-free. It lets the model think until warning deadlines, nudges it to speed up or conclude when necessary, and then extracts a final multiple-choice option.


In [ ]:
# =============================================================================
# DeepSeek solver, intentionally tool-free
# =============================================================================

DEEPSEEK_SPEEDUP_MSG = "\nLet's reason a little bit faster:"
DEEPSEEK_WRAPUP_MSG = "\nI need to wrap up. In conclusion:"


def deepseek_thinking_phase(runtime, tokens, warning_deadlines, t_hard, t_start):
    think_end = runtime.tokenizer.encode("</think>", add_special_tokens=False)
    log = []
    warnings = list(warning_deadlines)

    while warnings:
        current_deadline = warnings.pop(0)
        is_last = len(warnings) == 0

        while True:
            prev_len = tokens.shape[-1]
            tokens = runtime.generate(
                tokens,
                max_new=4096,
                deadline=current_deadline,
                stop_sequences=[think_end],
            )
            segment = runtime.decode_new(tokens, prev_len)
            log.append(segment)

            elapsed = time.time() - t_start
            print(f"💭 ({elapsed:.1f}s):\n{segment}\n")

            if "</think>" in segment:
                print(f"✅ Finished naturally in {elapsed:.1f}s.")
                return tokens, log

            break

        if is_last:
            print(f"⏳ Wrap-up warning ({elapsed:.1f}s), nudging to conclude...")
            tokens = runtime.append_text(tokens, DEEPSEEK_WRAPUP_MSG)
            log.append(DEEPSEEK_WRAPUP_MSG)

            prev_len = tokens.shape[-1]
            tokens = runtime.generate(
                tokens,
                max_new=4096,
                deadline=t_hard,
                stop_sequences=[think_end],
            )
            segment = runtime.decode_new(tokens, prev_len)
            log.append(segment)

            elapsed = time.time() - t_start
            print(f"💭 wrap-up ({elapsed:.1f}s):\n{segment}\n")

            if "</think>" in segment:
                print(f"✅ Wrapped up naturally ({elapsed:.1f}s).")
            else:
                print(f"🛑 Hard deadline ({elapsed:.1f}s), forcing </think>.")
                tokens = runtime.append_text(tokens, "</think>\n")
                log.append("</think>  [FORCED]")
        else:
            print(
                f"⚡ Speed-up warning {len(warning_deadlines) - len(warnings) - 1}"
                f" ({elapsed:.1f}s), nudging to go faster..."
            )
            tokens = runtime.append_text(tokens, DEEPSEEK_SPEEDUP_MSG)
            log.append(DEEPSEEK_SPEEDUP_MSG)

    return tokens, log


def build_deepseek_solver(runtime, cfg, priming_history):
    def solver(question):
        options_text = "\n".join(f"{opt.id}. {opt.text}" for opt in question.options)
        base_messages = priming_history or [{"role": "system", "content": system_prompt_for(cfg)}]
        messages = [
            *base_messages,
            {
                "role": "user",
                "content": f"Question: {question.text}\n\nOptions:\n{options_text}\n\nAnswer:",
            },
        ]

        tokens = runtime.chat_tokens(messages)
        tokens = runtime.append_text(tokens, "<think>\nLet's start:\n")

        t_start = time.time()
        warning_deadlines = [t_start + t for t in cfg.warning_times_s]

        if cfg.use_hard_deadline:
            if warning_deadlines:
                tokens, log = deepseek_thinking_phase(
                    runtime,
                    tokens,
                    warning_deadlines,
                    t_start + cfg.t_hard_s,
                    t_start,
                )
            else:
                think_end = runtime.tokenizer.encode("</think>", add_special_tokens=False)
                prev_len = tokens.shape[-1]
                tokens = runtime.generate(
                    tokens,
                    max_new=4096,
                    deadline=t_start + cfg.t_hard_s,
                    stop_sequences=[think_end],
                )
                log = [runtime.decode_new(tokens, prev_len)]
                if "</think>" not in log[-1]:
                    tokens = runtime.append_text(tokens, "</think>\n")
                    log.append("</think> [FORCED]")
        else:
            log = []

        tokens = runtime.append_text(tokens, "\nThe answer index is ")
        prev_len = tokens.shape[-1]
        tokens = runtime.generate(tokens, max_new=10, deadline=t_start + cfg.t_answer_s)
        response = runtime.decode_new(tokens, prev_len, skip_special_tokens=True).strip()

        print(f"🤖 '{response}' (total: {time.time() - t_start:.1f}s)")
        answer_id = parse_final_answer(response, question)
        if answer_id is not None:
            print(f"✔️  Option {answer_id}")
            return answer_id, "\n".join(log)

        print("⚠️  Fallback to random")
        return random_solver(question)

    return solver

### Solver Selection and Result Summary

`build_solver` chooses the correct solver builder from the configuration. `summarize_results` aggregates per-game logs into comparison metrics such as mean level reached, decision-time variability, total timeouts, and best level reached.


In [ ]:
def build_solver(runtime, cfg, priming_history):
    if cfg.model_family == "qwen":
        return build_qwen_solver(runtime, cfg, priming_history)
    if cfg.model_family == "deepseek":
        return build_deepseek_solver(runtime, cfg, priming_history)
    raise ValueError(f"Unsupported model family: {cfg.model_family}")

def summarize_results(game_df):
    metrics = [
        "levels_reached",
        "decision_time",
        "timeouts",
    ]

    grouped = game_df.groupby(["solver", "model"], dropna=False)
    rows = []
    for (solver, model), group in grouped:
        row = {
            "solver": solver,
            "model": model,
            "games_played": len(group),
            "timeouts_total": pd.to_numeric(group["timeouts"], errors="coerce").sum(),
        }
        for metric in metrics:
            values = pd.to_numeric(group[metric], errors="coerce")
            row[f"{metric}_mean"] = values.mean()
            row[f"{metric}_std"] = values.std(ddof=1)

        row['level_reached_max'] = pd.to_numeric(group['levels_reached'], errors="coerce").max()
        rows.append(row)


    return pd.DataFrame(rows)

### Math Investigation Configuration

`MathInvestigationConfig` describes one ablation setting: model family, priming, deadline behavior, calculator-tool usage, timing values, and number of games. The class name is intentionally section-specific to avoid colliding with the earlier `ExperimentConfig` definition in the notebook.


In [ ]:
# =============================================================================
# Model and solver configuration
# =============================================================================

QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DEEPSEEK_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"


@dataclass
class MathInvestigationConfig:
    model_name: str
    model_family: str

    use_priming: bool = True
    use_hard_deadline: bool = True
    use_soft_deadline: bool = True
    use_calc_tool: bool = False

    # Warning semantics:
    # - warnings 0 through n-2 inject a speed-up prompt and continue generation.
    # - warning n-1 is the soft deadline and injects a wrap-up prompt.
    # - t_hard_s remains a separate hard deadline after the soft deadline.
    warning_times_s: tuple = (22.0,)
    t_soft_s: float = 22.0
    t_hard_s: float = 27.0
    t_answer_s: float = 29.0
    max_tool_calls: int = 8
    num_games: int = 10

    def __post_init__(self):
        self.model_family = self.model_family.lower()
        if self.model_family not in {"qwen", "deepseek"}:
            raise ValueError("model_family must be 'qwen' or 'deepseek'")
        if self.use_calc_tool and self.model_family == "deepseek":
            raise ValueError("DeepSeek-R1 cannot use the math tool. Set use_calc_tool=False.")
        if self.use_soft_deadline and not self.use_hard_deadline:
            raise ValueError("use_soft_deadline requires use_hard_deadline=True")
        if self.use_soft_deadline:
            if not self.warning_times_s:
                self.warning_times_s = (self.t_soft_s,)
            self.warning_times_s = tuple(sorted(float(t) for t in self.warning_times_s))
            if self.warning_times_s[-1] >= self.t_hard_s:
                raise ValueError("The final warning/soft deadline must be before t_hard_s.")
        else:
            self.warning_times_s = tuple()
        if self.t_hard_s >= self.t_answer_s:
            raise ValueError("t_hard_s must be before t_answer_s.")

    @property
    def name(self):
        parts = [self.model_family]
        if self.use_priming:
            parts.append("primed")
        if self.use_hard_deadline:
            parts.append("hard")
        if self.use_soft_deadline:
            parts.append("soft")
        if self.use_calc_tool:
            parts.append("calc")
        return "+".join(parts)

### Ablation Runner

`run_ablation` executes each math investigation configuration against the selected competition. It reuses a loaded runtime when consecutive configurations use the same model, runs the requested number of games, prints comparisons, saves CSV outputs, and returns the game, question, and summary data frames.


In [ ]:
def run_ablation(client, competition_id, ablations, output_dir):
    competitions = client.competitions.list_all()
    print("\n=== Available Competitions ===")
    for comp in competitions:
        print(f"{comp.id}: {comp.name} ({comp.max_levels} questions)")

    current_model_name = None
    current_runtime = None

    for cfg in ablations:
        print(f"\n{'=' * 60}\nABLATION: {cfg.name}\nMODEL: {cfg.model_name}\n{'=' * 60}")

        if cfg.model_name != current_model_name:
            if current_runtime is not None:
                del current_runtime
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            current_runtime = ModelRuntime(cfg.model_name)
            current_model_name = cfg.model_name
        runtime = current_runtime

        priming_history = build_priming_history(runtime, cfg)
        solver = build_solver(runtime, cfg, priming_history)

        for game_idx in range(cfg.num_games):
            run_id = f"{cfg.name}__game_{game_idx + 1:03d}"
            print(f"\nGame {game_idx + 1}/{cfg.num_games}: {run_id}")
            game = client.game.start(competition_id=competition_id)
            play_game_auto(
                game,
                solver,
                solver_name=cfg.name,
                model_name=cfg.model_name,
                run_id=run_id,
            )

    game_df = pd.DataFrame(game_logs)
    question_df = pd.DataFrame(run_logs)
    summary_df = summarize_results(game_df)

    print("\n=== Per-game results ===")
    print(game_df.to_string(index=False))

    print("\n=== Mean and standard-deviation comparison ===")
    print(summary_df.to_string(index=False))

    os.makedirs(output_dir, exist_ok=True)
    game_df.to_csv(os.path.join(output_dir, "polimillionaire_game_results.csv"), index=False)
    question_df.to_csv(os.path.join(output_dir, "polimillionaire_question_logs.csv"), index=False)
    summary_df.to_csv(os.path.join(output_dir, "polimillionaire_summary_stats.csv"), index=False)

    return game_df, question_df, summary_df

### DeepSeek Ablation Setup

This configuration list compares DeepSeek with and without the soft-deadline wrap-up behavior. Both variants keep priming and calculator tools disabled so the comparison isolates the effect of the timing strategy.


In [ ]:
ablations = [
    MathInvestigationConfig(
        model_name=DEEPSEEK_MODEL,
        model_family="deepseek",
        use_priming=False,
        use_hard_deadline=True,
        use_soft_deadline=True,
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    ),

    MathInvestigationConfig(
        model_name=DEEPSEEK_MODEL,
        model_family="deepseek",
        use_priming=False,
        use_hard_deadline=True,
        use_soft_deadline=False,
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    )
]

In [ ]:
run_logs = []
game_logs = []

comp_id = 3

This cell launches the DeepSeek math experiments and writes the resulting CSV files into `deepseek_results`. It can take a while because it plays multiple full games with the local model.

In [ ]:
game_df, question_df, summary_df = run_ablation(client, comp_id, ablations, 'deepseek_results')


=== Available Competitions ===
0: Entertainment (15 questions)
1: Ancient History and Politics (15 questions)
2: Science and Nature (15 questions)
3: Maths (15 questions)
4: Philosophy and Psychology (15 questions)
5: News (15 questions)

ABLATION: deepseek+hard+soft
MODEL: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
Using device: cuda
Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/6.62G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


Game 1/15: deepseek+hard+soft__game_001

--- Level 1 ---
Question:
          (i + 1)(5 – 5i)(5 + 5i) =

          Options:
          0. 50 + 50i
1. 25 – 25i
2. 50 – 50i
3. 25 + 25i

          Answer:

Time remaining: 29.9s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


💭 (11.3s):
First, I'll expand the expression (i + 1)(5 – 5i)(5 + 5i).

I notice that (5 – 5i)(5 + 5i) is a difference of squares, so it simplifies to 25 + 25i².

Since i² = -1, this becomes 25 - 25 = 0.

Now, the expression simplifies to (i + 1) * 0 = 0.

So, the final answer is 0.
</think>

✅ Finished naturally in 11.3s.
🤖 '0.

**Step-by-Step Explanation:**' (total: 13.2s)
✔️  Option 0

Selected answer: 0
Decision time: 13.31s
CORRECT!

--- Level 2 ---
Question:
          Let $f(x)=3x+4$ and $g(x)=2x-3$. If $h(x)=f(g(x))$, then what is the inverse of $h(x)$?

          Options:
          0. \frac{x+5}{6}
1. \frac{x+5}{3}
2. \frac{x-5}{6}
3. \frac{x-5}{3}

          Answer:

Time remaining: 29.9s
💭 (22.1s):
First, I need to find the composition of functions h(x) = f(g(x)). 

So, I'll substitute g(x) into f(x). That means wherever I see an x in f(x), I'll replace it with g(x). 

Given f(x) = 3x + 4 and g(x) = 2x - 3, substituting g(x) into f(x) gives:
h(x) = 3*(2x - 3) + 4.

Next, I'll 

### Inspect DeepSeek Summary

As we can see in the following table, using the soft deadline leads to better results, both in terms of mean level reached and max level reached. Moreover, the decision time is also faster in this setup. However, the standard deviation is a little bit higher than the hard setup, indicating more variance in the performance of the model.

In [ ]:
summary_df.head()

,solver,model,games_played,timeouts_total,levels_reached_mean,levels_reached_std,decision_time_mean,decision_time_std,timeouts_mean,timeouts_std,level_reached_max
0,deepseek+hard,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,2.666667,1.799471,23.932217,3.721685,0.0,0.0,7
1,deepseek+hard+soft,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,3.266667,2.250926,23.565408,3.608467,0.0,0.0,10


### Qwen Ablation Setup

This configuration list compares Qwen with and without the calculator tool while keeping priming and deadline behavior enabled. The goal is to test whether explicit symbolic calculation improves math-game performance.


In [ ]:
ablations = [
    MathInvestigationConfig(
        model_name=DEEPSEEK_MODEL,
        model_family="deepseek",
        use_priming=False,
        use_hard_deadline=True,
        use_soft_deadline=True,
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    ),
]

comp_id = 3

game_df, question_df, summary_df = run_ablation(client, comp_id, ablations, 'math_final_results')

This cell launches the Qwen math experiments and writes the resulting CSV files into `qwen_results`. It reuses the same ablation runner so the Qwen and DeepSeek summaries are directly comparable.

In [ ]:
game_df, question_df, summary_df = run_ablation(client, comp_id, ablations, 'qwen_results')


=== Available Competitions ===
0: Entertainment (15 questions)
1: Ancient History and Politics (15 questions)
2: Science and Nature (15 questions)
3: Maths (15 questions)
4: Philosophy and Psychology (15 questions)
5: News (15 questions)

ABLATION: qwen+primed+hard+soft
MODEL: Qwen/Qwen2.5-7B-Instruct
Using device: cuda
Loading model: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model confirmed strategy:
- Answer each question concisely in one sentence.
- Provide the solution directly without using external tools.
- Choose the answer index (0-3) as the final output.


Game 1/15: qwen+primed+hard+soft__game_001

--- Level 1 ---
Question:
          A university’s mathematics department has 10 professors and will offer 20 different courses next semester. Each professor will be assigned to teach exactly 2 of the courses, and each course will have exactly one professor assigned to teach it. If any professor can be assigned to teach any course, how many different complete assignments of the 10 professors to the 20 courses are possible?

          Options:
          0. 10^(20) - 2^(10)
1. 20!/2^(10)
2. 10!/2^9
3. 10^(20) - 100

          Answer:

Time remaining: 29.9s
💭 (18.1s):
STEP A - Calculate the number of ways to assign 10 professors to 20 courses such that each professor teaches exactly 2 courses and each course has exactly one professor.
STEP B - First, choos

### Inspect Final Summary

As you can see in the following table, Qwen model with calculator tool performs slightly better than the Deepseek model with soft deadline. However, the Qwen setup has also hit the timeout 3 times, indicating that it might get stuck in certain questions. It also performs with a large variance.

As a result, we consider Deepseek with soft ddeadline as our best model.

In [ ]:
summary_df.head()

,solver,model,games_played,timeouts_total,levels_reached_mean,levels_reached_std,decision_time_mean,decision_time_std,timeouts_mean,timeouts_std,level_reached_max
0,deepseek+hard,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,2.666667,1.799471,23.932217,3.721685,0.0,0.000000,7
1,deepseek+hard+soft,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,3.266667,2.250926,23.565408,3.608467,0.0,0.000000,10
2,qwen+primed+hard+soft,Qwen/Qwen2.5-7B-Instruct,15,0,2.333333,1.676163,13.246093,4.738451,0.0,0.000000,7
3,qwen+primed+hard+soft+calc,Qwen/Qwen2.5-7B-Instruct,15,3,3.333333,2.350279,18.714999,5.995511,0.2,0.414039,10


# Analysis and Best Configuration Selection

## Results Summary from Actual Cluster Runs

The final selection is based on the logged experiment summaries already present in this notebook. Speech mode is **not** selected independently because after transcription the solver receives the same text-style `q_item`; therefore, the text experiments determine the best retrieval/LLM configuration.

## Selected Best Config per Final Competition

| # | Competition | Final Config | Evidence from logs |
|---|---|---|---|
| 0 | Entertainment | `qwen3b_wiki_bm25_k1_entity_raw` | Cluster 08: avg earned **341,466**, avg correct **6.333**, 0 timeouts |
| 1 | Ancient History and Politics | `qwen1_5b_ddg_bm25_k3_entity_raw` | Cluster 03: avg earned **86,100**, avg correct **7.333**, 0 timeouts |
| 2 | Science and Nature | `qwen3b_ddg_entity_only_k3` | Cluster 04: avg earned **16,166**, avg correct **7.667**, 0 timeouts |
| 4 | Philosophy and Psychology | `qwen3b_ddg_keyword_only_k3` | Cluster 03: avg earned **684,000**, avg correct **12.333**, 0 timeouts |
| 5 | News | `qwen3b_ddg_bm25_k5_entity_options` | Cluster 03: avg earned **666.667**, avg correct **2.000**, 0 timeouts |

## Speech Mode Decision

No separate speech experiment grid is needed. Speech is an additional input feature:

`audio question -> Whisper transcription -> same text question object -> same final config`

So the final section below runs the same chosen config in both text and speech modes. Speech testing is only to verify transcription quality.


# Final Testing Configs

## 0: Entertainment

### Speech Mode

In [ ]:
_cfg_entertainment_speech = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_wiki_bm25_k1_entity_raw"
)
import dataclasses
FINAL_SPEECH_CONFIG_ENTERTAINMENT = dataclasses.replace(
    _cfg_entertainment_speech,
    combo_name="final_entertainment_speech",
    game_mode="speech",
)

_whisper_model = get_whisper_model()
play_live_game_with_config(
    client=client,
    competition_id=0,
    cfg=FINAL_SPEECH_CONFIG_ENTERTAINMENT,
    run_id=1,
    verbose=True,
    whisper_model=_whisper_model,
)



Combo: final_entertainment_speech | Mode: speech | Competition: 0 - Entertainment | Run: 1
Session ID: 323249

--- Level 1 ---
What is the fundamental principle of Martin Scorsese's filmmaking style?
0. expressive and kinetic visuals.
1. highly stylized musicals.
2. Strictly Linear Narratives.
3. Slow, static scenes.
Time remaining: 24.7s
Predicted zero-based option: 0
Predicted letter: A
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 0
Decision time: 0.696
Raw answer: 0
Correct: True Timed out: False Earned: 100

--- Level 2 ---
Which of the following best describes Clark Gable's role in World War II?
0. He was a military general.
1. He served in the United States Army Air Forces.
2. He remained in Hollywood and did not participate.
3. He was a spy for the United States.
Time remaining: 25.3s
Wikipedia API search | query=describes Clark Gable role World War military served failed on attempt 1. Retrying in 0.48s. Error: HTTPError('429 Client Error: Too Many Request

(                   combo_name     llm_key prompt_style game_mode  \
 0  final_entertainment_speech  qwen2_5_3b       strong    speech   
 1  final_entertainment_speech  qwen2_5_3b       strong    speech   
 
   retrieval_tools             query_strategies  fetch_pages  \
 0       wikipedia  entity_focused+raw_question        False   
 1       wikipedia  entity_focused+raw_question        False   
 
    max_results_per_tool filter_method  top_k  ... retrieval_time_sec  \
 0                     3          bm25      1  ...              0.439   
 1                     3          bm25      1  ...              1.386   
 
    filtering_time_sec  raw_answer error  \
 0                 0.0           0  None   
 1                 0.0           2  None   
 
                                  question_audio_path  \
 0  speech_audio/session_323249/run_1/level_1/ques...   
 1  speech_audio/session_323249/run_1/level_2/ques...   
 
                                   option_audio_paths  \
 0  [speech_

### Text Mode

In [ ]:
# ── Competition 0: Entertainment — Final Text Config ──────────────────────────

_cfg_entertainment_text = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_wiki_bm25_k1_entity_raw"
)
# Override game_mode to text explicitly (already text by default):
import dataclasses
FINAL_TEXT_CONFIG_ENTERTAINMENT = dataclasses.replace(
    _cfg_entertainment_text,
    combo_name="final_entertainment_text",
    game_mode="text",
)

play_live_game_with_config(
    client=client,
    competition_id=0,
    cfg=FINAL_TEXT_CONFIG_ENTERTAINMENT,
    run_id=1,
    verbose=True,
)



Combo: final_entertainment_text | Mode: text | Competition: 0 - Entertainment | Run: 1
Session ID: 323228

--- Level 1 ---
Which term best describes the technique of projecting a series of images to create the illusion of motion?
0. Beta movement
1. Phi phenomenon
2. Stroboscopic disc
3. Persistence of vision
Time remaining: 29.9s
Predicted zero-based option: 3
API option id: 3
Decision time: 3.306
Raw answer: 3
Correct: True Timed out: False Earned: 100

--- Level 2 ---
Which of the following best describes the relationship between Don Vito Corleone and Michael Corleone in the film?
0. Teacher and student
1. Friend and confidant
2. Father and son
3. Boss and underling
Time remaining: 27.9s
Predicted zero-based option: 2
API option id: 2
Decision time: 2.662
Raw answer: 2
Correct: True Timed out: False Earned: 200

--- Level 3 ---
How does Nolan's use of practical effects in his films relate to his technical approach to filmmaking?
0. It limits the scope of his storytelling
1. It is p

(                 combo_name     llm_key prompt_style game_mode  \
 0  final_entertainment_text  qwen2_5_3b       strong      text   
 1  final_entertainment_text  qwen2_5_3b       strong      text   
 2  final_entertainment_text  qwen2_5_3b       strong      text   
 3  final_entertainment_text  qwen2_5_3b       strong      text   
 4  final_entertainment_text  qwen2_5_3b       strong      text   
 5  final_entertainment_text  qwen2_5_3b       strong      text   
 6  final_entertainment_text  qwen2_5_3b       strong      text   
 7  final_entertainment_text  qwen2_5_3b       strong      text   
 8  final_entertainment_text  qwen2_5_3b       strong      text   
 
   retrieval_tools             query_strategies  fetch_pages  \
 0       wikipedia  entity_focused+raw_question        False   
 1       wikipedia  entity_focused+raw_question        False   
 2       wikipedia  entity_focused+raw_question        False   
 3       wikipedia  entity_focused+raw_question        False   
 4      

In [ ]:
# ── Competition 0: Entertainment — Final Text Config ──────────────────────────

_cfg_entertainment_text = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_multiquery_bm25_k5"
)
# Override game_mode to text explicitly (already text by default):
import dataclasses
FINAL_TEXT_CONFIG_ENTERTAINMENT = dataclasses.replace(
    _cfg_entertainment_text,
    combo_name="final_entertainment_text",
    game_mode="text",
)

play_live_game_with_config(
    client=client,
    competition_id=0,
    cfg=FINAL_TEXT_CONFIG_ENTERTAINMENT,
    run_id=1,
    verbose=True,
)



Combo: final_entertainment_text | Mode: text | Competition: 0 - Entertainment | Run: 1
Session ID: 323557

--- Level 1 ---
Which of the following best describes the character Mark Grayson in the series Invincible?
0. A villain trying to overthrow the world
1. A human-alien hybrid who becomes a superhero
2. A sidekick assisting his father
3. A scientist developing superpowers
Time remaining: 29.9s
Predicted zero-based option: 1
API option id: 1
Decision time: 6.646
Raw answer: 1
Correct: True Timed out: False Earned: 100

--- Level 2 ---
Which term best describes Alfred Hitchcock's technique of making viewers feel like voyeurs in his films?
0. Suspense
1. Realism
2. Cinematic voyeurism
3. Mise-en-scène
Time remaining: 27.9s
Predicted zero-based option: 2
API option id: 2
Decision time: 5.875
Raw answer: 2
Correct: True Timed out: False Earned: 200

--- Level 3 ---
What was the fundamental principle behind Eminem's development of the Slim Shady persona?
0. To explore comedic storytellin

(                 combo_name     llm_key prompt_style game_mode  \
 0  final_entertainment_text  qwen2_5_3b       strong      text   
 1  final_entertainment_text  qwen2_5_3b       strong      text   
 2  final_entertainment_text  qwen2_5_3b       strong      text   
 3  final_entertainment_text  qwen2_5_3b       strong      text   
 4  final_entertainment_text  qwen2_5_3b       strong      text   
 5  final_entertainment_text  qwen2_5_3b       strong      text   
 6  final_entertainment_text  qwen2_5_3b       strong      text   
 
   retrieval_tools                               query_strategies  fetch_pages  \
 0      duckduckgo  entity_focused+keyword_condensed+raw_question        False   
 1      duckduckgo  entity_focused+keyword_condensed+raw_question        False   
 2      duckduckgo  entity_focused+keyword_condensed+raw_question        False   
 3      duckduckgo  entity_focused+keyword_condensed+raw_question        False   
 4      duckduckgo  entity_focused+keyword_condensed


##  1: Ancient History and Politics

### Speech Mode

In [ ]:

_cfg_history_speech = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_bm25_k1_entity_raw"
)
import dataclasses
FINAL_SPEECH_CONFIG_HISTORY = dataclasses.replace(
    _cfg_history_speech,
    combo_name="final_history_speech",
    game_mode="speech",
)

_whisper_model = get_whisper_model()
play_live_game_with_config(
    client=client,
    competition_id=1,
    cfg=FINAL_SPEECH_CONFIG_HISTORY,
    run_id=1,
    verbose=True,
    whisper_model=_whisper_model,
)



Combo: final_history_speech | Mode: speech | Competition: 1 - Ancient History and Politics | Run: 1
Session ID: 323500

--- Level 1 ---
Which of the following best describes the term Acropolis?
0. Ha ha ha! An upper part of an ancient Greek city, especially Acidado.
1. residential area in an ancient Greek city.
2. plain area in an ancient Greek city.
3. lower part of an ancient Greek city.
Time remaining: 19.0s
Predicted zero-based option: 0
Predicted letter: A
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 0
Decision time: 2.432
Raw answer: 0
Correct: True Timed out: False Earned: 100

--- Level 2 ---
What term describes the ancient tribe that lived on the alluvial plain around the rivers Haldiakmon and Lower Axios in the northeastern part of mainland Greece?
0. Oh, uh, cans.
1. Macedonians!
2. elerians!
3. Trations
Time remaining: 0.4s
Predicted zero-based option: 1
Predicted letter: B
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 1
Decision 

(             combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0  final_history_speech  qwen2_5_3b       strong    speech      duckduckgo   
 1  final_history_speech  qwen2_5_3b       strong    speech      duckduckgo   
 
               query_strategies  fetch_pages  max_results_per_tool  \
 0  entity_focused+raw_question        False                     3   
 1  entity_focused+raw_question        False                     3   
 
   filter_method  top_k  ... retrieval_time_sec  filtering_time_sec  \
 0          bm25      1  ...              2.132               0.000   
 1          bm25      1  ...              3.128               0.001   
 
    raw_answer error                                question_audio_path  \
 0           0  None  speech_audio/session_323500/run_1/level_1/ques...   
 1           1  None  speech_audio/session_323500/run_1/level_2/ques...   
 
                                   option_audio_paths  \
 0  [speech_audio/session_323500/run_1/level_1/opt

### Text Mode

In [ ]:
_cfg_history_text = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen1_5b_ddg_bm25_k3_entity_raw"
)
import dataclasses
FINAL_TEXT_CONFIG_HISTORY = dataclasses.replace(
    _cfg_history_text,
    combo_name="final_history_text",
    game_mode="text",
)

play_live_game_with_config(
    client=client,
    competition_id=1,
    cfg=FINAL_TEXT_CONFIG_HISTORY,
    run_id=1,
    verbose=True,
)



Combo: final_history_text | Mode: text | Competition: 1 - Ancient History and Politics | Run: 1
Session ID: 349125

--- Level 1 ---
What term describes the ritual act that resulted in the creation of a shrine that housed a cult image?
0. Augurium
1. Aedes
2. Consecratio
3. Atrium
Time remaining: 29.9s
Predicted zero-based option: 2
API option id: 2
Decision time: 3.47
Raw answer: 2
Correct: True Timed out: False Earned: 100

--- Level 2 ---
According to the passage, what was one of the main reasons Socrates was accused of impiety?
0. For not paying taxes
1. For failing to acknowledge the gods of Athens and introducing new deities
2. For teaching democracy
3. For supporting the Thirty Tyrants
Time remaining: 27.9s
Predicted zero-based option: 1
API option id: 1
Decision time: 5.671
Raw answer: 1
Correct: True Timed out: False Earned: 200

--- Level 3 ---
What term describes the three-way distinction between voiced, voiceless, and aspirated stops in Ancient Greek?
0. Nasal consonant
1. 

(           combo_name       llm_key prompt_style game_mode retrieval_tools  \
 0  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 1  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 2  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 3  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 4  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 5  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 6  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 7  final_history_text  qwen2_5_1_5b       strong      text      duckduckgo   
 
               query_strategies  fetch_pages  max_results_per_tool  \
 0  entity_focused+raw_question        False                     3   
 1  entity_focused+raw_question        False                     3   
 2  entity_focused+raw_question        False                     3   
 3  ent


##  2: Science and Nature

### Speech Mode

In [ ]:
# ── Competition 2: Science and Nature — Final Speech Config ───────────────────
# Same pipeline as text (qwen3b_ddg_multiquery_bm25_k5), game_mode="speech".

_cfg_science_speech = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_entity_only_k3"
)
import dataclasses
FINAL_SPEECH_CONFIG_SCIENCE = dataclasses.replace(
    _cfg_science_speech,
    combo_name="final_science_speech",
    game_mode="speech",
)

_whisper_model = get_whisper_model()
play_live_game_with_config(
    client=client,
    competition_id=2,
    cfg=FINAL_SPEECH_CONFIG_SCIENCE,
    run_id=1,
    verbose=True,
    whisper_model=_whisper_model,
)



Combo: final_science_speech | Mode: speech | Competition: 2 - Science and Nature | Run: 1
Session ID: 323294

--- Level 1 ---
A dietary supplement manufacturer developed a new calcium supplement that appears to increase the bone density in middle-aged women. When tested in a small clinical trial, the supplement was shown to cause a 15% increase in bone density in 8 out of 10 women. Before the supplement is offered to the general public, which steps should the manufacturer take next?
0. Test the supplement on mice in the laboratory.
1. Tephibopin, moin, light, in, light, a little, self-carny, Prant, septamine, one of you, past, mon-
2. evelop packaging that is attractive to women.
3. dd a variety of vitamins to the supplement.
Time remaining: 22.5s
Predicted zero-based option: 2
Predicted letter: C
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 2
Decision time: 1.53
Raw answer: 2
Correct: False Timed out: False Earned: 0

Run summary:
{
  "combo_name": "final_scienc

(             combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0  final_science_speech  qwen2_5_3b       strong    speech      duckduckgo   
 
   query_strategies  fetch_pages  max_results_per_tool filter_method  top_k  \
 0   entity_focused        False                     3          bm25      3   
 
    ... retrieval_time_sec  filtering_time_sec  raw_answer error  \
 0  ...              1.072                 0.0           2  None   
 
                                  question_audio_path  \
 0  speech_audio/session_323294/run_1/level_1/ques...   
 
                                   option_audio_paths  \
 0  [speech_audio/session_323294/run_1/level_1/opt...   
 
                   speech_option_map  \
 0  {'A': 0, 'B': 1, 'C': 2, 'D': 3}   
 
                                 transcribed_question  \
 0  A dietary supplement manufacturer developed a ...   
 
                              transcribed_options_raw  \
 0  [Option A. Test the supplement on mice in the ...  

### Text Mode

In [ ]:

_cfg_science_text = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_entity_only_k3"
)
import dataclasses
FINAL_TEXT_CONFIG_SCIENCE = dataclasses.replace(
    _cfg_science_text,
    combo_name="final_science_text",
    game_mode="text",
)

play_live_game_with_config(
    client=client,
    competition_id=2,
    cfg=FINAL_TEXT_CONFIG_SCIENCE,
    run_id=1,
    verbose=True,
)



Combo: final_science_text | Mode: text | Competition: 2 - Science and Nature | Run: 1
Session ID: 323283

--- Level 1 ---
Which characteristic is inherited rather than learned?
0. telling a story
1. having blue eyes
2. saluting the flag
3. riding a bicycle
Time remaining: 29.9s
Predicted zero-based option: 1
API option id: 1
Decision time: 2.952
Raw answer: 1
Correct: True Timed out: False Earned: 100

--- Level 2 ---
Most of the mass of the atom consists of
0. protons and neutrons.
1. protons only.
2. electrons only.
3. neutrons and electrons.
Time remaining: 27.9s
Predicted zero-based option: 0
API option id: 0
Decision time: 1.683
Raw answer: 0
Correct: True Timed out: False Earned: 200

--- Level 3 ---
Which activity illustrates the main function of the motor neuron complex?
0. transferring impulses to the appropriate muscle
1. delivering regular impulses to the autonomic system
2. processing impulses within the cerebral cortex
3. receiving sensory impulses from the environment
Ti

(           combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 1  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 2  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 3  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 4  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 5  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 6  final_science_text  qwen2_5_3b       strong      text      duckduckgo   
 
   query_strategies  fetch_pages  max_results_per_tool filter_method  top_k  \
 0   entity_focused        False                     3          bm25      3   
 1   entity_focused        False                     3          bm25      3   
 2   entity_focused        False                     3          bm25      3   
 3   entity_focused        False                     3          bm

## 3: Maths

### Text Mode

In [ ]:
ablations = [
    ExperimentConfig(
        model_name=DEEPSEEK_MODEL,
        model_family="deepseek",
        use_priming=False,
        use_hard_deadline=True,
        use_soft_deadline=True,
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    ),
]

comp_id = 3

game_df, question_df, summary_df = run_ablation(client, comp_id, ablations, 'math_final_results')


=== Available Competitions ===
0: Entertainment (15 questions)
1: Ancient History and Politics (15 questions)
2: Science and Nature (15 questions)
3: Maths (15 questions)
4: Philosophy and Psychology (15 questions)
5: News (15 questions)

ABLATION: deepseek+hard+soft
MODEL: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
Using device: cuda
Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Game 1/15: deepseek+hard+soft__game_001

--- Level 1 ---
Question:
          Consider a hypothesis test with H0 : μ = 70 and Ha : μ < 70. Which of the following choices of significance level and sample size results in the greatest power of the test when μ = 65?

          Options:
          0. α = 0.01, n = 15
1. α = 0.05, n = 30
2. α = 0.05, n = 15
3. α = 0.01, n = 30

          Answer:

Time remaining: 29.9s
💭 (22.0s):
I need to determine which combination of significance level (α) and sample size (n) provides the greatest power for a hypothesis test where H0: μ = 70 and Ha: μ < 70. The true mean is μ = 65.

Power is the probability of correctly rejecting H0 when it's false, which depends on α, sample size, and the effect size (the difference between the true mean and the hypothesized mean).

A higher α increases the power because it lowers the chance of a Type II error. A larger sample size also increases power because it reduces the standard error, making the test more sensitive t

In [ ]:
summary_df.head()

,solver,model,games_played,timeouts_total,levels_reached_mean,levels_reached_std,decision_time_mean,decision_time_std,timeouts_mean,timeouts_std,level_reached_max
0,deepseek+hard+soft,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,3.666667,2.919556,25.068433,3.701948,0.0,0.0,12



##  4: Philosophy and Psychology


### Speech Mode

In [ ]:
_cfg_philosophy_speech = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_keyword_only_k3"
)
import dataclasses
FINAL_SPEECH_CONFIG_PHILOSOPHY = dataclasses.replace(
    _cfg_philosophy_speech,
    combo_name="final_philosophy_speech",
    game_mode="speech",
)

_whisper_model = get_whisper_model()
play_live_game_with_config(
    client=client,
    competition_id=4,
    cfg=FINAL_SPEECH_CONFIG_PHILOSOPHY,
    run_id=1,
    verbose=True,
    whisper_model=_whisper_model,
)



Combo: final_philosophy_speech | Mode: speech | Competition: 4 - Philosophy and Psychology | Run: 1
Session ID: 323091

--- Level 1 ---
What term describes a company's approach to being a good corporate citizen by engaging in actions that benefit society and the environment?
0. orporate Philanthropy.
1. He, he, he, he, he, he, he, he, he, he, he, he, he, he, he, he, he. Corporate governance.
2. orporate Sustainability.
3. Topson D. Corporate Espionage.
Time remaining: 0.0s
Predicted zero-based option: 2
Predicted letter: C
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 2
Decision time: 2.052
Raw answer: 2
Correct: None Timed out: True Earned: 0

Run summary:
{
  "combo_name": "final_philosophy_speech",
  "llm_key": "qwen2_5_3b",
  "prompt_style": "strong",
  "game_mode": "speech",
  "retrieval_tools": "duckduckgo",
  "query_strategies": "keyword_condensed",
  "fetch_pages": false,
  "max_results_per_tool": 3,
  "filter_method": "bm25",
  "top_k": 3,
  "embedder_nam

(                combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0  final_philosophy_speech  qwen2_5_3b       strong    speech      duckduckgo   
 
     query_strategies  fetch_pages  max_results_per_tool filter_method  top_k  \
 0  keyword_condensed        False                     3          bm25      3   
 
    ... retrieval_time_sec  filtering_time_sec  raw_answer error  \
 0  ...              1.707                 0.0           2  None   
 
                                  question_audio_path  \
 0  speech_audio/session_323091/run_1/level_1/ques...   
 
                                   option_audio_paths  \
 0  [speech_audio/session_323091/run_1/level_1/opt...   
 
                   speech_option_map  \
 0  {'A': 0, 'B': 1, 'C': 2, 'D': 3}   
 
                                 transcribed_question  \
 0  What term describes a company's approach to be...   
 
                              transcribed_options_raw  \
 0  [Option A, Corporate Philanthropy., Optio

### Text Mode

In [ ]:
_cfg_philosophy_text = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_keyword_only_k3"
)
import dataclasses
FINAL_TEXT_CONFIG_PHILOSOPHY = dataclasses.replace(
    _cfg_philosophy_text,
    combo_name="final_philosophy_text",
    game_mode="text",
)


play_live_game_with_config(
    client=client,
    competition_id=4,
    cfg=FINAL_TEXT_CONFIG_PHILOSOPHY,
    run_id=1,
    verbose=True,
)



Combo: final_philosophy_text | Mode: text | Competition: 4 - Philosophy and Psychology | Run: 1
Session ID: 323074

--- Level 1 ---
What term describes Mozi's belief in the importance of hard work and virtuous acts in changing one's position in life?
0. Predestination
1. Anti-fatalism
2. Determinism
3. Fatalism
Time remaining: 29.9s
Predicted zero-based option: 1
API option id: 1
Decision time: 2.068
Raw answer: 1
Correct: True Timed out: False Earned: 100

--- Level 2 ---
Which of the following best describes chauvinism?
0. The promotion of international cooperation and diplomacy
1. The practice of fair and unbiased treatment of all people
2. The belief in the superiority of one's own group or people
3. The support for gender equality and diversity
Time remaining: 27.9s
Predicted zero-based option: 2
API option id: 2
Decision time: 1.568
Raw answer: 2
Correct: True Timed out: False Earned: 200

--- Level 3 ---
Which of the following best describes the concept of 'theory of mind'?
0. 

(               combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 1   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 2   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 3   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 4   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 5   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 6   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 7   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 8   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 9   final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 10  final_philosophy_text  qwen2_5_3b       strong      text      duckduckgo   
 11  final_philosophy_text  

##  5: News

### Speech Mode

In [ ]:

_cfg_news_speech = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_bm25_k5_entity_options"
)
import dataclasses
FINAL_SPEECH_CONFIG_NEWS = dataclasses.replace(
    _cfg_news_speech,
    combo_name="final_news_speech",
    game_mode="speech",
)

_whisper_model = get_whisper_model()
play_live_game_with_config(
    client=client,
    competition_id=5,
    cfg=FINAL_SPEECH_CONFIG_NEWS,
    run_id=1,
    verbose=True,
    whisper_model=_whisper_model,
)



Combo: final_news_speech | Mode: speech | Competition: 5 - News | Run: 1
Session ID: 323463

--- Level 1 ---
According to the article published on 20260517, which city saw swatch stores close for a second day due to large crowds?
0. Liverpool and Manchester Mon fi de one to L'Ecoge Day
1. London and Birmingham.
2. r-Bristolin leads.
3. Thompson D. Udenburg in... Glasgow?
Time remaining: 24.5s
Predicted zero-based option: 3
Predicted letter: D
Speech option map: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
API option id: 3
Decision time: 7.076
Raw answer: 3
Correct: False Timed out: False Earned: 0

Run summary:
{
  "combo_name": "final_news_speech",
  "llm_key": "qwen2_5_3b",
  "prompt_style": "strong",
  "game_mode": "speech",
  "retrieval_tools": "duckduckgo",
  "query_strategies": "entity_focused+question_plus_options",
  "fetch_pages": false,
  "max_results_per_tool": 3,
  "filter_method": "bm25",
  "top_k": 5,
  "embedder_name": "bge_small",
  "use_mmr": false,
  "use_reranker": false,
  "re

(          combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0  final_news_speech  qwen2_5_3b       strong    speech      duckduckgo   
 
                        query_strategies  fetch_pages  max_results_per_tool  \
 0  entity_focused+question_plus_options        False                     3   
 
   filter_method  top_k  ... retrieval_time_sec  filtering_time_sec  \
 0          bm25      5  ...              6.213               0.001   
 
    raw_answer error                                question_audio_path  \
 0           3  None  speech_audio/session_323463/run_1/level_1/ques...   
 
                                   option_audio_paths  \
 0  [speech_audio/session_323463/run_1/level_1/opt...   
 
                   speech_option_map  \
 0  {'A': 0, 'B': 1, 'C': 2, 'D': 3}   
 
                                 transcribed_question  \
 0  According to the article published on 20260517...   
 
                              transcribed_options_raw  \
 0  [Option A Liver

### Text Mode

In [ ]:

_cfg_news_text = next(
    cfg for cfg in MASTER_EXPERIMENT_CONFIGS
    if cfg.combo_name == "qwen3b_ddg_bm25_k5_entity_options"

)
import dataclasses
FINAL_TEXT_CONFIG_NEWS = dataclasses.replace(
    _cfg_news_text,
    combo_name="final_news_text",
    game_mode="text",
)

play_live_game_with_config(
    client=client,
    competition_id=5,
    cfg=FINAL_TEXT_CONFIG_NEWS,
    run_id=1,
    verbose=True,
)



Combo: final_news_text | Mode: text | Competition: 5 - News | Run: 1
Session ID: 323448

--- Level 1 ---
According to the news article published on 2026-05-18, which court declared the Kamal Maula mosque in Dhar a temple?
0. Bombay High Court
1. Delhi High Court
2. Madhya Pradesh High Court
3. Supreme Court of India
Time remaining: 29.9s
Predicted zero-based option: 2
API option id: 2
Decision time: 3.577
Raw answer: 2
Correct: True Timed out: False Earned: 100

--- Level 2 ---
According to the news article published on 2026-05-15, how many people had died from the Ebola outbreak in the Democratic Republic of the Congo?
0. 246
1. 65
2. 11,000
3. 35
Time remaining: 27.9s
Predicted zero-based option: 1
API option id: 1
Decision time: 4.657
Raw answer: 1
Correct: True Timed out: False Earned: 200

--- Level 3 ---
What is the expected delivery date of the first new military trucks, as mentioned in the article published on 2026-05-17?
0. 2035
1. 2025
2. 2030
3. 2028
Time remaining: 27.9s
P

(        combo_name     llm_key prompt_style game_mode retrieval_tools  \
 0  final_news_text  qwen2_5_3b       strong      text      duckduckgo   
 1  final_news_text  qwen2_5_3b       strong      text      duckduckgo   
 2  final_news_text  qwen2_5_3b       strong      text      duckduckgo   
 
                        query_strategies  fetch_pages  max_results_per_tool  \
 0  entity_focused+question_plus_options        False                     3   
 1  entity_focused+question_plus_options        False                     3   
 2  entity_focused+question_plus_options        False                     3   
 
   filter_method  top_k  ... retrieval_time_sec  filtering_time_sec  \
 0          bm25      5  ...              3.098               0.000   
 1          bm25      5  ...              4.058               0.001   
 2          bm25      5  ...              3.680               0.000   
 
    raw_answer error  question_audio_path  option_audio_paths  \
 0           2  None             